# سامانهٔ امنیتی — ماژول تشخیص پوشش صورت

**نسخهٔ تک‌فایلی و بدون نیاز به هیچ آموزشی.**

کافی است مسیر ویدیو را در سلول «اجرا» بگذاری و از بالا تا پایین Run کنی.

## چه کاری می‌کند

| وضعیت | رنگ | معنی |
|---|---|---|
| Analyzing | آبی روشن | هنوز شواهد کافی جمع نشده |
| Clear | سبز | صورت واضح و قابل شناسایی |
| Covered | خاکستری | ماسک پزشکی یا پشت به دوربین — بی‌خطر |
| Watch | زرد | نشانهٔ پوشش کامل، ولی زیر آستانهٔ آلارم |
| **ALERT** | **قرمز** | **ماسک دزدی / پوشش کامل صورت** |

خروجی‌ها: ویدیوی حاشیه‌نویسی‌شده، برش تمیز چهرهٔ هر سوژه، و یک فایل
رویداد JSONL.

## ایدهٔ معماری

> هیچ ماژولی خودش تصمیم نمی‌گیرد. هر ماژول فقط **شاهد** تولید می‌کند و
> آن را روی یک **هویت سراسری** ثبت می‌کند؛ تصمیم نهایی را لایهٔ fusion
> با جمعِ وزن‌دارِ شواهد در طول زمان می‌گیرد.

به همین دلیل ماژول‌های بعدی (اسلحه، آتش، رفتار) بدون بازنویسی روی
همین هسته سوار می‌شوند.

> **قبل از شروع:** Runtime ← Change runtime type ← GPU

## ۰) نصب و بررسی سخت‌افزار

In [ ]:
!pip install -q ultralytics transformers

import torch, cv2, numpy as np, sys
print("Python      :", sys.version.split()[0])
print("PyTorch     :", torch.__version__)
print("OpenCV      :", cv2.__version__)
if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))
else:
    print("⚠️ GPU روشن نیست — Runtime ← Change runtime type ← GPU")

---
# بخش اول: کد هسته

سلول‌های زیر فقط **تعریف** می‌کنند و چیزی اجرا نمی‌کنند. همه را یک بار
Run کن و برو سراغ بخش دوم. ترتیبشان مهم است.

### قراردادهای داده (کلاس‌ها و وضعیت‌ها)
<sub>`security_core/types.py`</sub>

In [ ]:
# ==========================================================================
#  قراردادهای داده (کلاس‌ها و وضعیت‌ها)
#  (منبع: security_core/types.py)
# ==========================================================================
from __future__ import annotations

"""
types.py — قراردادهای دادهٔ مشترک بین همهٔ ماژول‌ها.

اگر می‌خواهی بفهمی اجزای سیستم چطور با هم حرف می‌زنند، از همین فایل شروع کن.
همهٔ ماژول‌ها فقط با این ساختارها تبادل داده می‌کنند، نه با دیکشنری خام.
دلیلش: وقتی ماژول اسلحه/آتش اضافه شود، امضای توابع تغییر نمی‌کند.
"""

from dataclasses import dataclass, field
from enum import Enum
from typing import Dict, List, Optional, Tuple

import numpy as np

# ---------------------------------------------------------------------------
#  کلاس‌های وضعیت پوشش صورت — مجموعهٔ کلاسِ «متعارف» (canonical)
#  هر طبقه‌بندی (SigLIP، MobileNet، ONNX، ROI-Head) موظف است خروجی‌اش را
#  به همین مجموعه نگاشت کند. این تنها نقطه‌ای است که کلاس‌ها تعریف می‌شوند.
# ---------------------------------------------------------------------------
class FaceClass(str, Enum):
    CLEAR         = "clear"          # صورت باز و قابل شناسایی
    MEDICAL_MASK  = "medical_mask"   # ماسک پزشکی/بهداشتی (پیشانی و چشم پیداست)
    FULL_COVER    = "full_cover"     # بالاکلاوا / اسکی‌ماسک / کلاه‌کاسکت / شال دور صورت  ← تهدید
    BACK_HEAD     = "back_head"      # پشت به دوربین یا نیم‌رخ شدید — بی‌اطلاع، نه تهدید
    UNKNOWN       = "unknown"        # کیفیت ناکافی برای قضاوت

    @staticmethod
    def all() -> List["FaceClass"]:
        return [FaceClass.CLEAR, FaceClass.MEDICAL_MASK, FaceClass.FULL_COVER,
                FaceClass.BACK_HEAD, FaceClass.UNKNOWN]


# ---------------------------------------------------------------------------
#  وضعیت نمایشی/عملیاتی هر فرد (خروجی لایهٔ fusion → policy)
#  این دقیقاً همان جدول رنگی است که مشتری می‌بیند.
# ---------------------------------------------------------------------------
class ThreatState(str, Enum):
    ANALYZING = "analyzing"   # آبی کم‌رنگ — هنوز شواهد کافی نیست
    CLEAR     = "clear"       # سبز        — صورت واضح، بی‌خطر
    COVERED   = "covered"     # خاکستری    — پوشیده ولی بی‌خطر (ماسک پزشکی / نامشخص)
    WATCH     = "watch"       # زرد        — مشکوک ولی زیر آستانهٔ آلارم
    SUSPECT   = "suspect"     # قرمز       — ماسک دزدی / پوشش کامل صورت → آلارم
    # ★ «پشت به دوربین» عمداً یک وضعیت مستقل است، نه زیرمجموعهٔ COVERED.
    #   قضاوت دربارهٔ پوششِ صورتِ کسی که پشتش به ماست بی‌معنی است؛
    #   نه بی‌خطر است و نه مشکوک — فقط «قابل قضاوت نیست».
    #   قاطی‌کردن این دو باعث می‌شد پشتِ سرِ افراد خاکستری (پوشیده)
    #   نشان داده شود و کل خروجی بی‌معنا به نظر برسد.
    AWAY      = "away"        # آبی تیره   — پشت به دوربین، قابل ارزیابی نیست


@dataclass
class Keypoints:
    """۱۷ کی‌پوینت COCO به‌همراه اطمینان هرکدام (مختصات در فضای فریم کامل)."""
    xy: np.ndarray     # (17, 2) float32
    conf: np.ndarray   # (17,)   float32

    # اندیس‌های COCO-17 — به‌صورت ثابت کلاسی تا هیچ‌جا عدد جادویی ننویسیم
    NOSE, LEYE, REYE, LEAR, REAR = 0, 1, 2, 3, 4
    LSHO, RSHO, LELB, RELB, LWRI, RWRI = 5, 6, 7, 8, 9, 10
    LHIP, RHIP, LKNE, RKNE, LANK, RANK = 11, 12, 13, 14, 15, 16

    def pt(self, idx: int) -> Tuple[float, float]:
        return float(self.xy[idx][0]), float(self.xy[idx][1])

    def c(self, idx: int) -> float:
        return float(self.conf[idx])


@dataclass
class HeadROI:
    """
    جعبهٔ سر. نکتهٔ کلیدی معماری:
    «کجاست سر» از «آیا صورت پیداست» جدا شده است.
    این جعبه حتی وقتی چشم و بینی اصلاً دیده نمی‌شوند هم محاسبه می‌شود
    (از روی هندسهٔ شانه‌ها)، چون دقیقاً همان حالت است که باید بررسی شود.
    """
    xyxy: Tuple[int, int, int, int]
    source: str          # "eyes" | "shoulders" | "bbox_fallback"
    reliability: float   # 0..1 — چقدر به این جعبه اعتماد داریم


@dataclass
class FaceObservation:
    """یک مشاهدهٔ خام از صورت یک فرد در یک فریم، قبل از طبقه‌بندی."""
    crop: np.ndarray            # کروپ تراز‌شدهٔ صورت (BGR) — همیشه از فریم *تمیز*
    head_roi: HeadROI
    quality: float              # 0..1 — وزن این مشاهده در رأی‌گیری
    quality_parts: Dict[str, float] = field(default_factory=dict)  # برای دیباگ
    aligned: bool = False       # آیا چرخش تراز چشم اعمال شد؟


@dataclass
class PersonFrame:
    """وضعیت یک فرد در یک فریم مشخص."""
    track_id: int
    global_id: int
    bbox: Tuple[int, int, int, int]
    det_conf: float
    kpts: Optional[Keypoints]
    head_roi: Optional[HeadROI] = None
    face_obs: Optional[FaceObservation] = None
    probs: Optional[Dict[FaceClass, float]] = None   # خروجی طبقه‌بند این فریم
    state: ThreatState = ThreatState.ANALYZING
    state_conf: float = 0.0


@dataclass
class SecurityEvent:
    """
    رویدادی که به بیرون گزارش می‌شود (JSONL / MQTT / وب‌هوک).
    ماژول‌های آینده (اسلحه/آتش) هم دقیقاً همین ساختار را تولید می‌کنند.
    """
    global_id: int
    kind: str                 # "face.suspect" | "weapon.detected" | "fire.detected" | ...
    state: str
    confidence: float
    frame_idx: int
    timestamp: float
    camera_id: str = "cam0"
    bbox: Optional[Tuple[int, int, int, int]] = None
    extra: Dict = field(default_factory=dict)

    def to_dict(self) -> Dict:
        d = dict(self.__dict__)
        d["bbox"] = list(self.bbox) if self.bbox else None
        return d

### تنظیمات — همهٔ آستانه‌ها اینجاست
<sub>`security_core/config.py`</sub>

In [ ]:
# ==========================================================================
#  تنظیمات — همهٔ آستانه‌ها اینجاست
#  (منبع: security_core/config.py)
# ==========================================================================
from __future__ import annotations

"""
config.py — بارگذاری تنظیمات از YAML.

قانون پروژه: هیچ عدد جادویی داخل کد منطق نوشته نمی‌شود.
هر آستانه‌ای که ممکن است روزی تیون شود، باید اینجا باشد.
دلیل: هنگام نصب روی سایت مشتری، فقط yaml عوض می‌شود نه کد.
"""

import copy
from dataclasses import dataclass, field, fields, is_dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional



# --------------------------------------------------------------------------- #
#                              بخش‌های تنظیمات                                  #
# --------------------------------------------------------------------------- #
@dataclass
class RuntimeCfg:
    device: str = "auto"              # auto | cuda | cpu
    half: bool = True                 # FP16 روی GPU — تقریباً ۲ برابر سرعت، افت دقت ناچیز
    channels_last: bool = True        # چیدمان حافظه بهینه برای Tensor Core
    cudnn_benchmark: bool = True
    torch_compile: bool = False       # کامپایل اولیه کند است؛ برای اجرای طولانی روشن کن
    max_faces_per_frame: int = 8      # سقف بودجهٔ محاسباتی هر فریم (تضمین latency)
    log_every_n_frames: int = 60


@dataclass
class PoseCfg:
    weights: str = "yolo11n-pose.pt"  # n = سریع | s = دقیق‌تر (مچ و دست در فاصلهٔ دور)
    imgsz: int = 640
    conf: float = 0.35
    iou: float = 0.6
    min_person_area_ratio: float = 0.0008   # حذف افراد خیلی دور (منبع نویز)
    kpt_conf_th: float = 0.35               # آستانهٔ «این کی‌پوینت را باور کن»


@dataclass
class TrackerCfg:
    cfg_path: str = "config/trackers/botsort_security.yaml"
    fallback_cfg: str = "bytetrack.yaml"   # اگر نسخهٔ ultralytics botsort را نپذیرفت


@dataclass
class ReIDCfg:
    enabled: bool = True
    backend: str = "auto"            # auto | torchreid | torchvision | colorhist
    embed_every_n: int = 8           # هر N فریم برای هر فرد (نه هر فریم — گران است)
    input_size: List[int] = field(default_factory=lambda: [128, 256])   # w, h
    match_threshold: float = 0.62    # شباهت کسینوسی؛ بالاتر = سخت‌گیرتر
    gallery_per_identity: int = 5    # چند embedding به‌ازای هر هویت نگه داریم
    lost_ttl_seconds: float = 45.0   # تا چند ثانیه بعد از گم‌شدن، هویت زنده بماند


@dataclass
class HeadROICfg:
    """پارامترهای استخراج جعبهٔ سر از هندسهٔ بدن."""
    # ★ ضرایب زیر روی هندسهٔ انسانی کالیبره شده‌اند تا هر سه مسیرِ
    #   تخمین (چشم / شانه / تنه) برای یک فردِ روبه‌رو تقریباً کادر
    #   یکسانی بدهند ≈ ۳.۲ برابر فاصلهٔ دو چشم.
    #   این همان کادری است که نسخهٔ اولِ پروژه می‌ساخت و خوب کار می‌کرد.
    #   با ضرایب قبلی (0.72 / 3.4 / 1.12) کادر ~۲.۱ برابر بزرگ‌تر می‌شد
    #   و صورت فقط ۳۱٪ تصویرِ ورودیِ طبقه‌بند را پر می‌کرد.
    head_w_over_shoulder: float = 0.50   # عرض سر ≈ ۰.۵ × عرض شانه
    head_w_over_torso: float = 0.32      # مسیر دوم (نیم‌رخ): ۰.۳۲ × طول تنه
    head_h_over_w: float = 1.25          # نسبت ارتفاع به عرض سر
    head_up_offset: float = 0.42         # فاصلهٔ مرکز سر از خط شانه (ضریبی از عرض شانه)
    eye_box_scale: float = 3.0           # وقتی چشم‌ها پیدایند: عرض ≈ ۳ × فاصلهٔ دو چشم
    bbox_top_ratio: float = 0.20         # آخرین راه‌حل: ۲۰٪ بالای جعبهٔ بدن
    expand: float = 1.06                 # حاشیهٔ کم — کادر تنگ برای طبقه‌بند بهتر است


@dataclass
class OrientationCfg:
    """
    ★ آستانه‌های تشخیص جهتِ سر (perception/orientation.py).

    facing از ‎−۱‎ (پشت) تا ‎+۱‎ (روبه‌رو) تغییر می‌کند.
    اگر دیدی افرادِ پشت‌به‌دوربین هنوز تحلیل می‌شوند، back_threshold را
    به صفر نزدیک‌تر کن (مثلاً ‎−۰.۱۵‎). اگر برعکس، افرادِ روبه‌رو
    اشتباهاً «پشت» تشخیص داده می‌شوند، از صفر دورترش کن.
    """
    enabled: bool = True
    back_threshold: float = -0.28     # زیر این مقدار: پشت به دوربین
    frontal_threshold: float = 0.22   # بالای این مقدار: رو به دوربین
    min_confidence: float = 0.22      # زیر این اطمینان: «نامشخص»


@dataclass
class FaceQualityCfg:
    min_face_px: int = 44            # کوچک‌تر از این = مشاهده اصلاً ثبت نمی‌شود
    good_face_px: int = 120          # از این به بالا امتیاز اندازه اشباع می‌شود
    min_sharpness: float = 12.0      # واریانس لاپلاسین؛ کمتر = تار
    good_sharpness: float = 120.0
    min_quality_to_vote: float = 0.18  # زیر این وزن، رأی ثبت نمی‌شود
    align_eye_conf: float = 0.40     # حداقل اطمینان چشم‌ها برای اعمال چرخش تراز


@dataclass
class ClassifierCfg:
    # hybrid = مدل فاین‌تیون‌شده + صفر-شات. دقیق‌ترین گزینهٔ بدون آموزش.
    backend: str = "hybrid"          # hybrid | siglip_binary | siglip_zeroshot | timm | onnx | roi_head
    siglip_binary_model: str = "prithivMLmods/Face-Mask-Detection"
    siglip_zeroshot_model: str = "google/siglip-base-patch16-224"
    timm_model: str = "mobilenetv4_conv_small.e2400_r224_in1k"
    timm_weights: str = "weights/face_state_student.pt"
    onnx_path: str = "weights/face_state_student.onnx"
    roi_head_weights: str = "weights/roi_face_head.pt"
    hybrid_route_threshold: float = 0.25   # از این احتمالِ «پوشیده» به بالا، مدل دوم هم اجرا شود
    input_size: int = 224
    batch_max: int = 16
    use_weak_occlusion_cue: bool = True   # سرنخ ضعیفِ بافت پیشانی (وزن کم، فقط کمکی)
    weak_cue_weight: float = 0.35


@dataclass
class FusionCfg:
    decay_per_second: float = 0.55    # شواهد قدیمی چقدر سریع محو شوند (۰..۱)
    temperature: float = 1.0
    min_observations: int = 2         # حداقل مشاهده قبل از خروج از ANALYZING

    # آستانه‌های هیسترزیس — ورود سخت، خروج آسان‌تر (ضدِ چشمک‌زدن)
    suspect_enter: float = 0.72
    suspect_exit: float = 0.42
    watch_enter: float = 0.45
    clear_enter: float = 0.70
    clear_exit: float = 0.45

    # قفلِ «نرم» — سبز هیچ‌وقت دائمی نیست (دزد می‌تواند وسط راه ماسک بزند)
    clear_revalidate_seconds: float = 2.5
    focus_revalidate_seconds: float = 0.25   # نرخ بازبینی افراد مشکوک/پوشیده

    # ★ آلارمِ بیرونی فقط وقتی صادر می‌شود که وضعیت قرمز حداقل این
    # مدت پایدار مانده باشد. یک جهشِ لحظه‌ای (مثلاً عبور یک سایه)
    # نباید آژیر بزند. رابطهٔ مستقیم با اعتماد مشتری به سیستم دارد.
    alert_min_seconds: float = 0.35

    # ★ کلاس‌های UNKNOWN و BACK_HEAD یعنی «دربارهٔ پوششِ صورت چیزی
    #   نمی‌دانم»، نه «صورت پوشیده نیست». اگر بگذاریم در همان softmax
    #   با سه کلاسِ معنادار رقابت کنند، جرمِ احتمال را می‌بلعند و
    #   p_full هرگز به آستانهٔ ۰.۷۲ نمی‌رسد → هیچ‌وقت آلارم نمی‌دهد.
    #   پس تصمیم روی توزیعِ *بازنرمال‌شده* بین سه کلاس معنادار گرفته
    #   می‌شود و این دو کلاس فقط «قابل اتکا بودن» مشاهده را می‌سنجند.
    min_informative_mass: float = 0.30

    # ★ خاکستری («ماسک پزشکی») فقط با شاهدِ واقعی اعلام می‌شود، نه
    #   به‌عنوان حالتِ پیش‌فرضِ ابهام.
    covered_enter: float = 0.50


@dataclass
class SnapshotCfg:
    enabled: bool = True
    out_dir: str = "runs/events"
    save_states: List[str] = field(default_factory=lambda: ["suspect", "watch", "covered"])
    save_body: bool = True
    save_context: bool = True
    min_quality: float = 0.30
    jpeg_quality: int = 95
    # حالت «برداشت دیتاست»: از همه، حتی سبزها، بهترین کروپ را ذخیره کن
    # تا سوخت چرخهٔ auto-label و آموزش مدل دانش‌آموز فراهم شود (فاز ۳)
    harvest_all_for_dataset: bool = False
    harvest_dir: str = "runs/harvest"
    # فریم کامل تمیز را هم کنار کروپ ذخیره کن. لازمهٔ آموزش سرِ RoI
    # در فاز ۴ (که به تصویر کامل نیاز دارد، نه فقط کروپ).
    # هزینه: حدود ۲۰۰KB به‌ازای هر نمونه — فقط موقع ساخت دیتاست روشن کن.
    harvest_frames: bool = True


@dataclass
class ZonesCfg:
    """
    ناحیهٔ حساس. مختصات نسبی (۰..۱) تا مستقل از رزولوشن دوربین باشد.
    مثال: فقط داخل مغازه پایش شود، ویترین و پیاده‌رو نه.
        include: [[[0.05,0.35],[0.95,0.35],[0.98,0.98],[0.02,0.98]]]
        exclude: [[[0.0,0.0],[1.0,0.0],[1.0,0.30],[0.0,0.30]]]
    """
    include: List = field(default_factory=list)   # فقط داخل این چندضلعی‌ها
    exclude: List = field(default_factory=list)   # هرگز داخل این چندضلعی‌ها
    normalized: bool = True
    draw: bool = True


@dataclass
class VizCfg:
    enabled: bool = True
    draw_skeleton: bool = True
    draw_head_box: bool = False
    draw_gallery: bool = True
    gallery_items: int = 4
    gallery_thumb: int = 150
    slowmo_enabled: bool = False       # فقط در پروفایل دمو
    slowmo_repeat: int = 5
    slowmo_on_new_person: bool = False
    slowmo_on_suspect: bool = True
    font_scale: float = 0.55
    thickness: int = 2
    hud: bool = True                   # نوار اطلاعات بالای تصویر (FPS/تعداد/هشدار)


@dataclass
class IOCfg:
    source: str = ""                   # مسیر فایل | "0" برای وبکم | rtsp://...
    camera_id: str = "cam0"
    output_video: str = "runs/output.mp4"
    events_jsonl: str = "runs/events.jsonl"
    write_video: bool = True
    frame_stride: int = 1              # ۱=همهٔ فریم‌ها، ۲=یکی‌درمیان (سرعت بیشتر)
    max_frames: int = 0                # ۰ = بی‌نهایت (برای تست سریع مفید است)


@dataclass
class AppConfig:
    runtime: RuntimeCfg = field(default_factory=RuntimeCfg)
    pose: PoseCfg = field(default_factory=PoseCfg)
    tracker: TrackerCfg = field(default_factory=TrackerCfg)
    reid: ReIDCfg = field(default_factory=ReIDCfg)
    head_roi: HeadROICfg = field(default_factory=HeadROICfg)
    orientation: OrientationCfg = field(default_factory=OrientationCfg)
    face_quality: FaceQualityCfg = field(default_factory=FaceQualityCfg)
    classifier: ClassifierCfg = field(default_factory=ClassifierCfg)
    fusion: FusionCfg = field(default_factory=FusionCfg)
    snapshot: SnapshotCfg = field(default_factory=SnapshotCfg)
    zones: ZonesCfg = field(default_factory=ZonesCfg)
    viz: VizCfg = field(default_factory=VizCfg)
    io: IOCfg = field(default_factory=IOCfg)

# در نسخهٔ نوت‌بوکی، تنظیمات مستقیم ساخته می‌شود و به فایل YAML
# نیازی نیست. برای دیدن همهٔ کلیدها: AppConfig() را چاپ کن.


### هندسه: جعبه، کروپ امن، شباهت کسینوسی
<sub>`security_core/utils/geometry.py`</sub>

In [ ]:
# ==========================================================================
#  هندسه: جعبه، کروپ امن، شباهت کسینوسی
#  (منبع: security_core/utils/geometry.py)
# ==========================================================================
from __future__ import annotations

"""
geometry.py — عملیات هندسی روی جعبه‌ها و نقاط.

همهٔ توابع «امن» هستند: هیچ‌وقت جعبهٔ خارج از تصویر یا با عرض/ارتفاع منفی
برنمی‌گردانند. یکی از باگ‌های رایج نسخهٔ قبلی همین بود که کروپ خالی
(size == 0) تولید می‌شد و بعداً در جای دیگری خطا می‌داد.
"""

from typing import Sequence, Tuple

import numpy as np

BBox = Tuple[int, int, int, int]   # x1, y1, x2, y2


def clamp_box(box: Sequence[float], w: int, h: int, min_size: int = 1) -> BBox:
    """جعبه را داخل مرزهای تصویر می‌برد و صحت اندازه را تضمین می‌کند."""
    x1, y1, x2, y2 = [float(v) for v in box]
    if x2 < x1:
        x1, x2 = x2, x1
    if y2 < y1:
        y1, y2 = y2, y1
    x1 = int(max(0, min(w - min_size, x1)))
    y1 = int(max(0, min(h - min_size, y1)))
    x2 = int(max(x1 + min_size, min(w, x2)))
    y2 = int(max(y1 + min_size, min(h, y2)))
    return x1, y1, x2, y2


def expand_box(box: Sequence[float], scale: float, w: int, h: int) -> BBox:
    """جعبه را حول مرکز خودش بزرگ (یا کوچک) می‌کند."""
    x1, y1, x2, y2 = box
    cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
    bw, bh = (x2 - x1) * scale, (y2 - y1) * scale
    return clamp_box((cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2), w, h)


def box_area(box: Sequence[float]) -> float:
    return max(0.0, box[2] - box[0]) * max(0.0, box[3] - box[1])


def iou(a: Sequence[float], b: Sequence[float]) -> float:
    """اشتراک بر اجتماع — برای تشخیص جابه‌جایی ID و تطبیق جعبه‌ها."""
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    union = box_area(a) + box_area(b) - inter
    return float(inter / union) if union > 0 else 0.0


def safe_crop(img: np.ndarray, box: Sequence[float], min_size: int = 2) -> np.ndarray:
    """کروپ تضمین‌شده — اگر ناحیه معتبر نباشد، آرایهٔ خالی برمی‌گرداند نه خطا."""
    if img is None or img.size == 0:
        return np.empty((0, 0, 3), dtype=np.uint8)
    h, w = img.shape[:2]
    x1, y1, x2, y2 = clamp_box(box, w, h, min_size=min_size)
    crop = img[y1:y2, x1:x2]
    return crop if crop.size else np.empty((0, 0, 3), dtype=np.uint8)


def l2(p: Sequence[float], q: Sequence[float]) -> float:
    return float(np.hypot(p[0] - q[0], p[1] - q[1]))


def unit(vx: float, vy: float) -> Tuple[float, float]:
    n = float(np.hypot(vx, vy))
    return (0.0, 0.0) if n < 1e-6 else (vx / n, vy / n)


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """شباهت کسینوسی بین دو بردار ویژگی (برای ReID)."""
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na < 1e-8 or nb < 1e-8:
        return 0.0
    return float(np.dot(a, b) / (na * nb))

### عملیات تصویری و سنجه‌های کیفیت
<sub>`security_core/utils/imageops.py`</sub>

In [ ]:
# ==========================================================================
#  عملیات تصویری و سنجه‌های کیفیت
#  (منبع: security_core/utils/imageops.py)
# ==========================================================================
from __future__ import annotations

"""
imageops.py — عملیات تصویری سبک و سنجه‌های کیفیت.

نکتهٔ سرعت: تمام پیش‌پردازش با OpenCV انجام می‌شود، نه PIL.
در نسخهٔ قبلی، تبدیل به PIL و عبور از `AutoImageProcessor` هاگینگ‌فیس
یک گلوگاه CPU حدود ۸ میلی‌ثانیه‌ای در هر فریم بود — یعنی بیشتر از
خودِ استنتاج YOLO. اینجا آن مسیر کاملاً حذف شده است.
"""

from typing import List, Tuple

import cv2
import numpy as np


# --------------------------------------------------------------------------- #
#                              سنجه‌های کیفیت                                   #
# --------------------------------------------------------------------------- #
def sharpness(img_bgr: np.ndarray) -> float:
    """
    واریانس لاپلاسین = میزان «لبه‌دار بودن» تصویر.
    تصویر تار (حرکت سریع یا خارج از فوکوس) عدد پایینی می‌دهد.
    از این برای وزن‌دهی به رأی‌ها و انتخاب «بهترین شات» استفاده می‌کنیم.
    """
    if img_bgr is None or img_bgr.size == 0:
        return 0.0
    g = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY) if img_bgr.ndim == 3 else img_bgr
    # روی تصویر کوچک‌شده حساب می‌کنیم تا هزینه ثابت و ناچیز بماند
    if max(g.shape) > 160:
        s = 160.0 / max(g.shape)
        g = cv2.resize(g, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    return float(cv2.Laplacian(g, cv2.CV_32F).var())


def brightness_ok(img_bgr: np.ndarray, lo: float = 25.0, hi: float = 235.0) -> float:
    """
    امتیاز ۰..۱ برای «نوردهی قابل قبول».
    تصویر کاملاً سیاه یا سوخته را از رأی‌گیری کنار می‌گذاریم.
    """
    if img_bgr is None or img_bgr.size == 0:
        return 0.0
    m = float(img_bgr.mean())
    if m < lo or m > hi:
        return 0.0
    # نزدیک وسط بازه = بهترین
    center = (lo + hi) / 2.0
    return float(1.0 - abs(m - center) / (center - lo))


def normalize_range(x: float, lo: float, hi: float) -> float:
    """نگاشت خطی x از بازهٔ [lo, hi] به [0, 1] با کلیپ."""
    if hi <= lo:
        return 0.0
    return float(np.clip((x - lo) / (hi - lo), 0.0, 1.0))


# --------------------------------------------------------------------------- #
#                     پیش‌پردازش دسته‌ای برای طبقه‌بند                            #
# --------------------------------------------------------------------------- #
# مقادیر نرمال‌سازی استاندارد
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)
SIGLIP_MEAN = np.array([0.5, 0.5, 0.5], dtype=np.float32)
SIGLIP_STD = np.array([0.5, 0.5, 0.5], dtype=np.float32)


def stack_crops(crops: List[np.ndarray], size: int) -> np.ndarray:
    """
    لیستی از کروپ BGR را به یک آرایهٔ NHWC از نوع float32 در بازهٔ [0,1] و RGB تبدیل می‌کند.
    (تبدیل به تنسور و نرمال‌سازی روی GPU انجام می‌شود — سریع‌تر از CPU.)
    """
    out = np.empty((len(crops), size, size, 3), dtype=np.float32)
    for i, c in enumerate(crops):
        if c is None or c.size == 0:
            out[i] = 0.0
            continue
        # INTER_AREA برای کوچک‌کردن بهترین کیفیت را با هزینهٔ کم می‌دهد
        interp = cv2.INTER_AREA if max(c.shape[:2]) > size else cv2.INTER_LINEAR
        r = cv2.resize(c, (size, size), interpolation=interp)
        out[i] = cv2.cvtColor(r, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    return out


def letterbox_square(img: np.ndarray, size: int, pad_value: int = 114) -> np.ndarray:
    """
    مربعی‌کردن بدون تغییر نسبت ابعاد (برای وقتی که کشیدگی تصویر مضر است).
    برای کروپ صورت معمولاً `stack_crops` کافی است؛ این برای ReID بدن مفید است.
    """
    h, w = img.shape[:2]
    s = size / max(h, w)
    nh, nw = max(1, int(round(h * s))), max(1, int(round(w * s)))
    r = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_LINEAR)
    canvas = np.full((size, size, 3), pad_value, dtype=img.dtype)
    y0, x0 = (size - nh) // 2, (size - nw) // 2
    canvas[y0:y0 + nh, x0:x0 + nw] = r
    return canvas


def rotate_bound_point(pt: Tuple[float, float], M: np.ndarray) -> Tuple[float, float]:
    """اعمال ماتریس آفین ۲×۳ روی یک نقطه."""
    x, y = pt
    return (float(M[0, 0] * x + M[0, 1] * y + M[0, 2]),
            float(M[1, 0] * x + M[1, 1] * y + M[1, 2]))

### پروفایلر زمان و سنجش FPS
<sub>`security_core/utils/timing.py`</sub>

In [ ]:
# ==========================================================================
#  پروفایلر زمان و سنجش FPS
#  (منبع: security_core/utils/timing.py)
# ==========================================================================
from __future__ import annotations

"""
timing.py — پروفایلر سبک برای اندازه‌گیری هزینهٔ هر مرحله.

چرا مهم است: هدف محصول «چند دوربین روی یک GPU» است.
بدون اندازه‌گیریِ دقیقِ هر مرحله، بهینه‌سازی حدس‌وگمان می‌شود.
این پروفایلر سربار تقریباً صفر دارد و همیشه می‌تواند روشن بماند.
"""

import time
from collections import defaultdict
from contextlib import contextmanager
from typing import Dict, List


class Profiler:
    def __init__(self, enabled: bool = True, sync_cuda: bool = False):
        self.enabled = enabled
        # روی GPU، اندازه‌گیری بدون همگام‌سازی گمراه‌کننده است (اجرا ناهمگام است).
        # ولی sync خودش هزینه دارد؛ فقط موقع بنچمارک روشنش کن.
        self.sync_cuda = sync_cuda
        self._t: Dict[str, List[float]] = defaultdict(list)

    def _sync(self):
        if self.sync_cuda:
            try:
                import torch
                if torch.cuda.is_available():
                    torch.cuda.synchronize()
            except Exception:
                pass

    @contextmanager
    def section(self, name: str):
        if not self.enabled:
            yield
            return
        self._sync()
        t0 = time.perf_counter()
        try:
            yield
        finally:
            self._sync()
            self._t[name].append((time.perf_counter() - t0) * 1000.0)

    def add(self, name: str, ms: float):
        self._t[name].append(ms)

    def summary(self, last_n: int = 0) -> Dict[str, float]:
        """میانگین میلی‌ثانیهٔ هر بخش. last_n>0 یعنی فقط N نمونهٔ آخر."""
        out = {}
        for k, v in self._t.items():
            vals = v[-last_n:] if last_n else v
            if vals:
                out[k] = sum(vals) / len(vals)
        return out

    def report(self, title: str = "پروفایل زمان (میانگین میلی‌ثانیه بر فریم)") -> str:
        s = self.summary()
        if not s:
            return ""
        width = max(len(k) for k in s) + 2
        total = s.get("frame_total", sum(v for k, v in s.items() if k != "frame_total"))
        lines = [f"\n{title}", "-" * (width + 22)]
        for k, v in sorted(s.items(), key=lambda kv: -kv[1]):
            pct = (v / total * 100.0) if total > 0 else 0.0
            lines.append(f"  {k:<{width}} {v:7.2f} ms   {pct:5.1f}%")
        if total > 0:
            lines.append("-" * (width + 22))
            lines.append(f"  {'≈ FPS':<{width}} {1000.0 / total:7.1f}")
        return "\n".join(lines)

    def reset(self):
        self._t.clear()


class FPSMeter:
    """میانگین متحرک نرخ فریم — برای نمایش روی HUD."""

    def __init__(self, window: int = 30):
        self.window = window
        self._times: List[float] = []
        self._last = None

    def tick(self) -> float:
        now = time.perf_counter()
        if self._last is not None:
            self._times.append(now - self._last)
            if len(self._times) > self.window:
                self._times.pop(0)
        self._last = now
        if not self._times:
            return 0.0
        avg = sum(self._times) / len(self._times)
        return 1.0 / avg if avg > 0 else 0.0

### ★ جعبهٔ سر از هندسهٔ بدن
<sub>`security_core/perception/head_roi.py`</sub>

In [ ]:
# ==========================================================================
#  ★ جعبهٔ سر از هندسهٔ بدن
#  (منبع: security_core/perception/head_roi.py)
# ==========================================================================
from __future__ import annotations

"""
head_roi.py — ★ استخراج جعبهٔ سر از هندسهٔ بدن.

این ماژول مهم‌ترین اصلاح معماری نسبت به نسخهٔ قبلی است.

مشکل نسخهٔ قبلی
---------------
کروپ صورت فقط وقتی ساخته می‌شد که میانگین اطمینانِ کی‌پوینت‌های
چشم و بینی از یک آستانه بیشتر باشد. اما کسی که بالاکلاوا یا
کلاه‌کاسکت تمام‌صورت زده، دقیقاً همان کسی است که YOLO-pose برای
چشم و بینی‌اش اطمینان پایین می‌دهد. یعنی:

    دزدِ نقاب‌دار → گیت ردش می‌کند → هیچ رأیی ثبت نمی‌شود
                  → تا آخر ویدیو خاکستری می‌ماند.

اصل درست: «نبودِ شواهد، خودش یک شاهد است».

راه‌حل
------
دو سؤالِ متفاوت را از هم جدا می‌کنیم:

  ۱) «سر کجاست؟»      → هندسهٔ شانه/تنه. تقریباً همیشه در دسترس است،
                          حتی وقتی هیچ جزئیاتی از صورت دیده نمی‌شود.
  ۲) «صورت پیداست؟»   → وظیفهٔ طبقه‌بند، نه وظیفهٔ گیت.

بنابراین جعبهٔ سر همیشه محاسبه می‌شود و همیشه به طبقه‌بند می‌رود.
طبقه‌بند خودش کلاس `back_head` دارد تا حالت پشت‌به‌دوربین را
مدیریت کند — پس دیگر به گیت کی‌پوینت نیازی نیست.
"""

from typing import List, Optional, Tuple

import numpy as np


K = Keypoints  # کوتاه‌نویسی برای اندیس‌ها


# --------------------------------------------------------------------------- #
#                   کاندیدهای مختلف برای تخمین جعبهٔ سر                          #
# --------------------------------------------------------------------------- #
def _cand_from_eyes(kp: Keypoints, cfg: HeadROICfg, kconf_th: float):
    """
    دقیق‌ترین حالت: هر دو چشم دیده می‌شوند.
    فاصلهٔ دو چشم یک مقیاس بسیار پایدار برای اندازهٔ سر است.
    """
    if kp.c(K.LEYE) < kconf_th or kp.c(K.REYE) < kconf_th:
        return None
    le, re = kp.pt(K.LEYE), kp.pt(K.REYE)
    d = l2(le, re)
    if d < 2.0:
        return None
    hw = cfg.eye_box_scale * d
    hh = hw * cfg.head_h_over_w
    cx = (le[0] + re[0]) / 2.0
    # چشم‌ها حدوداً در ۴۲٪ ارتفاع سر از بالا قرار دارند،
    # پس مرکز سر کمی پایین‌تر از خط چشم است.
    cy = (le[1] + re[1]) / 2.0 + 0.10 * hh
    rel = 0.95 * min(1.0, (kp.c(K.LEYE) + kp.c(K.REYE)) / 1.6)
    return (cx, cy, hw, hh, rel)


def _cand_from_shoulders(kp: Keypoints, cfg: HeadROICfg, kconf_th: float):
    """
    حالت کلیدی: صورت اصلاً دیده نمی‌شود ولی شانه‌ها هستند.
    این همان حالتی است که نسخهٔ قبلی کاملاً از دست می‌داد.
    """
    if kp.c(K.LSHO) < kconf_th or kp.c(K.RSHO) < kconf_th:
        return None
    ls, rs = kp.pt(K.LSHO), kp.pt(K.RSHO)
    sw = l2(ls, rs)
    if sw < 4.0:
        return None

    mid = ((ls[0] + rs[0]) / 2.0, (ls[1] + rs[1]) / 2.0)

    # بردار عمود بر خط شانه‌ها. جهتش باید «به‌سمت سر» باشد.
    vx, vy = rs[0] - ls[0], rs[1] - ls[1]
    nx, ny = unit(vy, -vx)          # چرخش ۹۰ درجه

    nose_ok = kp.c(K.NOSE) >= kconf_th
    if nose_ok:
        # اگر بینی را داریم، جهت را با آن تعیین کن (مقاوم به چرخش دوربین)
        no = kp.pt(K.NOSE)
        if (no[0] - mid[0]) * nx + (no[1] - mid[1]) * ny < 0:
            nx, ny = -nx, -ny
    elif ny > 0:
        # وگرنه فرض می‌کنیم سر بالاتر از شانه است (y کمتر در تصویر)
        nx, ny = -nx, -ny

    # مقیاس سر: عرض شانه معیار اصلی است، اما در نیم‌رخ عرض شانه
    # کوچک دیده می‌شود و سر را کوچک تخمین می‌زند. پس طول تنه
    # (شانه تا لگن) را هم به‌عنوان معیار دوم می‌گیریم و بزرگ‌تر را برمی‌داریم.
    hw = cfg.head_w_over_shoulder * sw
    torso = _torso_length(kp, kconf_th)
    if torso is not None:
        # ★ ضریب کالیبره‌شده: برای فردِ روبه‌رو این دو معیار باید عدد
        # *یکسان* بدهند، وگرنه max() کادر را برای همه بزرگ می‌کند.
        # با 0.50×شانه و 0.32×تنه هر دو ≈ ۳.۲ برابر فاصلهٔ چشم‌اند —
        # یعنی دقیقاً همان کادری که نسخهٔ اولِ پروژه می‌ساخت.
        # (ضریب قبلی 0.36 بود و کادر را ~۲.۲ برابر باد می‌کرد.)
        hw = max(hw, cfg.head_w_over_torso * torso)

    hh = hw * cfg.head_h_over_w
    cx = mid[0] + nx * cfg.head_up_offset * sw
    cy = mid[1] + ny * cfg.head_up_offset * sw

    rel = 0.78 * min(1.0, (kp.c(K.LSHO) + kp.c(K.RSHO)) / 1.6)
    if nose_ok:
        rel = min(0.90, rel + 0.08)
    return (cx, cy, hw, hh, rel)


def _cand_from_single_shoulder(kp: Keypoints, cfg: HeadROICfg, kconf_th: float):
    """حالت نیم‌رخ شدید: فقط یک شانه + طول تنه."""
    torso = _torso_length(kp, kconf_th)
    if torso is None:
        return None
    sho = None
    for idx in (K.LSHO, K.RSHO):
        if kp.c(idx) >= kconf_th:
            sho = kp.pt(idx)
            break
    if sho is None:
        return None
    hw = 0.40 * torso
    hh = hw * cfg.head_h_over_w
    return (sho[0], sho[1] - 0.75 * hh, hw, hh, 0.50)


def _cand_from_bbox(person_bbox, cfg: HeadROICfg):
    """آخرین راه‌حل: هیچ کی‌پوینت قابل اتکایی نداریم — ۲۲٪ بالای جعبهٔ بدن."""
    x1, y1, x2, y2 = person_bbox
    ph = max(1.0, y2 - y1)
    pw = max(1.0, x2 - x1)
    hh = cfg.bbox_top_ratio * ph
    hw = min(pw * 0.85, hh / cfg.head_h_over_w)
    return ((x1 + x2) / 2.0, y1 + hh / 2.0, hw, hh, 0.32)


def _torso_length(kp: Keypoints, kconf_th: float) -> Optional[float]:
    """فاصلهٔ میانهٔ شانه‌ها تا میانهٔ لگن — مقیاسی پایدار حتی در نیم‌رخ."""
    sh, hp = [], []
    for i in (K.LSHO, K.RSHO):
        if kp.c(i) >= kconf_th:
            sh.append(kp.pt(i))
    for i in (K.LHIP, K.RHIP):
        if kp.c(i) >= kconf_th:
            hp.append(kp.pt(i))
    if not sh or not hp:
        return None
    smid = np.mean(sh, axis=0)
    hmid = np.mean(hp, axis=0)
    d = float(np.hypot(*(smid - hmid)))
    return d if d > 5.0 else None


# --------------------------------------------------------------------------- #
#                                تابع اصلی                                      #
# --------------------------------------------------------------------------- #
def estimate_head_roi(kp: Optional[Keypoints],
                      person_bbox: Tuple[int, int, int, int],
                      frame_w: int, frame_h: int,
                      cfg: HeadROICfg,
                      kconf_th: float = 0.35) -> HeadROI:
    """
    جعبهٔ سر را برمی‌گرداند. **هرگز None برنمی‌گرداند** — همیشه بهترین
    تخمین ممکن را می‌دهد و کیفیتِ تخمین را در فیلد `reliability` گزارش می‌کند.

    لایهٔ fusion از همین `reliability` برای وزن‌دهی به رأی استفاده می‌کند،
    پس تخمین ضعیف خودبه‌خود اثر کمی می‌گذارد و نیازی به دور انداختنش نیست.
    """
    named: List[Tuple[str, Tuple[float, float, float, float, float]]] = []

    if kp is not None:
        for fn, name in ((_cand_from_eyes, "eyes"),
                         (_cand_from_shoulders, "shoulders"),
                         (_cand_from_single_shoulder, "single_shoulder")):
            c = fn(kp, cfg, kconf_th)
            if c is not None:
                named.append((name, c))

    if not named:
        named.append(("bbox_fallback", _cand_from_bbox(person_bbox, cfg)))

    # ★ ترکیب فقط بین کاندیدهای «هم‌مقیاس»
    #
    # باگی که اینجا بود و بیشترین افت دقت را ساخت:
    # قبلاً میانگینِ وزن‌دارِ *همهٔ* کاندیدها گرفته می‌شد. اما تخمینِ
    # چشم‌محور یک کادر تنگِ صورت می‌دهد و تخمینِ شانه‌محور یک کادر
    # بزرگِ سر-و-شانه. میانگین این دو، کادری می‌سازد که هیچ‌کدام
    # نیست: صورت فقط ~۳۱٪ کادر را پر می‌کرد (به‌جای ~۶۴٪).
    # طبقه‌بند تصویری می‌دید که صورت در آن ریز و گم بود.
    #
    # حالا: بهترین کاندید مبنا است، و فقط کاندیدهایی با آن میانگین
    # گرفته می‌شوند که مقیاسشان تا ±۳۵٪ با آن هم‌خوان باشد. اثر
    # پایدارسازی (کم‌لرزشی) حفظ می‌شود، بدون باد کردن کادر.
    named.sort(key=lambda nc: -nc[1][4])
    source, best = named[0]
    cands = [c for _, c in named if 0.65 <= c[2] / max(1e-6, best[2]) <= 1.35]
    if not cands:
        cands = [best]

    w = np.array([c[4] for c in cands], dtype=np.float64)
    w = w / w.sum()
    cx = float(np.dot(w, [c[0] for c in cands]))
    cy = float(np.dot(w, [c[1] for c in cands]))
    hw = float(np.dot(w, [c[2] for c in cands]))
    hh = float(np.dot(w, [c[3] for c in cands]))
    rel = float(max(c[4] for c in cands))

    box = (cx - hw / 2, cy - hh / 2, cx + hw / 2, cy + hh / 2)
    box = expand_box(box, cfg.expand, frame_w, frame_h)
    return HeadROI(xyxy=clamp_box(box, frame_w, frame_h, min_size=2),
                   source=source, reliability=rel)


# --------------------------------------------------------------------------- #
#                       سرنخ‌های کمکی برای لایهٔ fusion                          #
# --------------------------------------------------------------------------- #
def facing_score(kp: Optional[Keypoints], kconf_th: float = 0.35) -> float:
    """
    چقدر احتمال دارد فرد رو به دوربین باشد؟ (۰..۱)

    کاربرد: اگر فرد رو به دوربین است ولی هیچ جزئیاتی از صورتش دیده
    نمی‌شود، این خودش شاهدِ پوشیده‌بودن صورت است. اگر پشت به دوربین
    باشد، نبودِ صورت کاملاً طبیعی است و نباید سوءظن ایجاد کند.
    """
    if kp is None:
        return 0.0
    if kp.c(K.LSHO) < kconf_th or kp.c(K.RSHO) < kconf_th:
        return 0.0
    sw = l2(kp.pt(K.LSHO), kp.pt(K.RSHO))
    torso = _torso_length(kp, kconf_th)
    if torso is None or torso < 1e-3:
        return 0.5
    # نسبت عرض شانه به طول تنه: روبه‌رو ≈ ۰.۹ تا ۱.۲ ، نیم‌رخ ≈ ۰.۳
    ratio = sw / torso
    return float(np.clip((ratio - 0.35) / 0.45, 0.0, 1.0))


def face_kpt_visibility(kp: Optional[Keypoints]) -> dict:
    """اطمینانِ کی‌پوینت‌های ناحیهٔ سر — ورودیِ سرنخ ضعیفِ پوشیدگی."""
    if kp is None:
        return {"nose": 0.0, "eyes": 0.0, "ears": 0.0}
    return {
        "nose": kp.c(K.NOSE),
        "eyes": (kp.c(K.LEYE) + kp.c(K.REYE)) / 2.0,
        "ears": (kp.c(K.LEAR) + kp.c(K.REAR)) / 2.0,
    }

### ★ جهتِ سر — رو / نیم‌رخ / پشت
<sub>`security_core/perception/orientation.py`</sub>

In [ ]:
# ==========================================================================
#  ★ جهتِ سر — رو / نیم‌رخ / پشت
#  (منبع: security_core/perception/orientation.py)
# ==========================================================================
from __future__ import annotations

"""
orientation.py — ★ تشخیص جهتِ سر: رو به دوربین، نیم‌رخ، یا پشت به دوربین.

اشتباهی که این ماژول جبران می‌کند
---------------------------------
نسخهٔ اولِ پروژه یک گیت داشت: «فقط وقتی چشم و بینی با اطمینان دیده
می‌شوند، صورت را تحلیل کن». من آن گیت را برداشتم، با این استدلال که
دزدِ نقاب‌دار دقیقاً همان کسی است که چشم و بینی‌اش دیده نمی‌شود.

استدلال درست بود، ولی **جایگزینش غلط بود**. من قضاوتِ «اصلاً این
صورت است یا پشتِ سر؟» را به طبقه‌بند سپردم. طبقه‌بند در این کار ضعیف
است — مخصوصاً مدل دوکلاسهٔ ماسک که هرگز پشتِ سر ندیده. نتیجه: پشتِ
سرِ افراد هم مثل صورت قضاوت می‌شد.

راه درست: از خودِ مدل ژست بپرس
------------------------------
مدل ژست چیزی می‌داند که هیچ‌کدام از طبقه‌بندها نمی‌دانند: کی‌پوینت‌ها
برچسبِ **آناتومیک** دارند — «شانهٔ چپِ شخص»، نه «شانهٔ سمت چپِ تصویر».

از این یک نکتهٔ ساده، یک سیگنال بسیار قوی بیرون می‌آید:

    فرد رو به دوربین  →  شانهٔ چپِ او در سمت راستِ تصویر است
    فرد پشت به دوربین →  شانهٔ چپِ او در سمت چپِ تصویر است

یعنی علامتِ  (x شانهٔ چپ) − (x شانهٔ راست)  جهت را می‌گوید. به این
خاصیت «چیرالیته» (دست‌وارگی) می‌گویند.

چرا این دقیقاً همان چیزی است که لازم داشتیم:

  • **مستقل از دیده‌شدن صورت است.** فقط به هندسهٔ بدن نیاز دارد، پس
    برای فردی که بالاکلاوا زده هم کار می‌کند — همان حالتی که گیتِ
    نسخهٔ اول از دستش می‌داد.
  • در نیم‌رخِ کامل، دو شانه روی هم می‌افتند و اختلاف x به صفر میل
    می‌کند → خودِ فرمول می‌گوید «مطمئن نیستم». یعنی عدمِ قطعیت را
    خودش گزارش می‌کند، لازم نیست جداگانه مدیریتش کنیم.
  • رایگان است: چند عمل حسابی روی کی‌پوینت‌هایی که همین حالا داریم.

سه منبعِ چیرالیته داریم (شانه‌ها، گوش‌ها، چشم‌ها) و رأی‌شان را
وزن‌دار جمع می‌کنیم. دیده‌شدنِ اجزای صورت هم به‌عنوان یک سرنخِ
**ضعیف** اضافه می‌شود — عمداً ضعیف، چون فردِ نقاب‌دار هم چشم و بینی
ندارد و نباید با «پشت به دوربین» اشتباه گرفته شود. این تفکیک، قلبِ
درستیِ این ماژول است.
"""

import math
from dataclasses import dataclass
from typing import List, Optional, Tuple

import numpy as np


K = Keypoints


@dataclass
class Orientation:
    """
    facing:      −۱ کاملاً پشت به دوربین … ۰ نیم‌رخ … +۱ کاملاً رو به دوربین
    confidence:  چقدر به همین عدد اطمینان داریم (۰..۱)
    mode:        frontal | profile | back | unknown
    face_vis:    میزان دیده‌شدن اجزای صورت (۰..۱) — برای سرنخ پوشیدگی
    """
    facing: float
    confidence: float
    mode: str
    face_vis: float

    @property
    def is_back(self) -> bool:
        return self.mode == "back"

    @property
    def weight_scale(self) -> float:
        """
        ضریبی که وزنِ مشاهده در آن ضرب می‌شود.

        منطق: هرچه کمتر مطمئنیم فرد رو به دوربین است، رأیِ طبقه‌بند
        هم باید کم‌اثرتر باشد. این «نرم‌ترین» شکلِ گیت است — به‌جای
        دور انداختنِ مشاهده، فقط از وزنش کم می‌کنیم.
        """
        if self.mode == "frontal":
            return 1.0
        if self.mode == "profile":
            return 0.60
        if self.mode == "unknown":
            return 0.45
        return 0.0        # back — اصلاً رأی نمی‌دهد


def _torso_length(kp: Keypoints, conf_th: float) -> Optional[float]:
    """فاصلهٔ میانهٔ شانه‌ها تا میانهٔ لگن — مقیاسی که با چرخش عوض نمی‌شود."""
    sh = [kp.pt(i) for i in (K.LSHO, K.RSHO) if kp.c(i) >= conf_th]
    hp = [kp.pt(i) for i in (K.LHIP, K.RHIP) if kp.c(i) >= conf_th]
    if not sh or not hp:
        return None
    d = float(np.hypot(*(np.mean(sh, axis=0) - np.mean(hp, axis=0))))
    return d if d > 5.0 else None


def _chirality(kp: Keypoints, i_left: int, i_right: int, conf_th: float,
               expected_sep: Optional[float] = None
               ) -> Optional[Tuple[float, float]]:
    """
    یک رأیِ چیرالیته از یک جفت کی‌پوینتِ چپ/راست.

    خروجی: (مقدار در ‎[−۱,+۱]‎ ، وزن) یا None اگر جفت قابل اتکا نیست.
    مثبت = رو به دوربین، منفی = پشت به دوربین.

    نکتهٔ ظریفِ «کوتاه‌شدگی» (foreshortening)
    ----------------------------------------
    علامتِ اختلاف x جهت را می‌گوید، ولی *بزرگیِ* آن هم اطلاعات دارد:
    وقتی فرد به نیم‌رخ می‌چرخد، دو شانه در تصویر روی هم می‌افتند و
    فاصله‌شان کوچک می‌شود. در آن حالت علامت عملاً نویز است.

    اگر فقط dx را بر فاصلهٔ *فعلی* تقسیم کنیم (کاری که اول کردم)،
    این اطلاعات از بین می‌رود: دو شانه با فاصلهٔ ۵ پیکسل هم مقدار ۱.۰
    می‌دهند، انگار کاملاً روبه‌رو باشد. پس فاصلهٔ فعلی را با فاصلهٔ
    *مورد انتظار* (از روی مقیاس بدن) می‌سنجیم و وزن رأی را به همان
    نسبت کم می‌کنیم. حالا نیم‌رخ خودش را «نامطمئن» اعلام می‌کند.
    """
    cl, cr = kp.c(i_left), kp.c(i_right)
    if cl < conf_th or cr < conf_th:
        return None
    pl, pr = kp.pt(i_left), kp.pt(i_right)
    dx = float(pl[0] - pr[0])
    sep = math.hypot(pl[0] - pr[0], pl[1] - pr[1])
    if sep < 2.0:
        return None

    v = float(np.clip(dx / (0.70 * sep), -1.0, 1.0))
    w = float(min(cl, cr))
    if expected_sep and expected_sep > 1e-3:
        # ۱.۰ در حالت روبه‌روی کامل → نزدیک صفر در نیم‌رخِ کامل
        w *= float(np.clip(sep / expected_sep, 0.0, 1.0))
    return v, w


def estimate_orientation(kp: Optional[Keypoints], conf_th: float = 0.35,
                         cfg=None) -> Orientation:
    """جهتِ سر را از هندسهٔ کی‌پوینت‌ها تخمین می‌زند."""
    back_th = getattr(cfg, "back_threshold", -0.28) if cfg else -0.28
    front_th = getattr(cfg, "frontal_threshold", 0.22) if cfg else 0.22
    min_conf = getattr(cfg, "min_confidence", 0.22) if cfg else 0.22

    if kp is None:
        return Orientation(0.0, 0.0, "unknown", 0.0)

    # ---- میزان دیده‌شدن اجزای صورت (فقط برای گزارش و سرنخ پوشیدگی) ----
    eyes_v = (kp.c(K.LEYE) + kp.c(K.REYE)) / 2.0
    nose_v = kp.c(K.NOSE)
    ears_v = (kp.c(K.LEAR) + kp.c(K.REAR)) / 2.0
    face_vis = float(np.clip(0.45 * eyes_v + 0.35 * nose_v + 0.20 * ears_v, 0.0, 1.0))

    votes: List[Tuple[float, float]] = []

    # مقیاس‌های مرجع برای سنجشِ کوتاه‌شدگی. نسبت‌ها از هندسهٔ انسانی
    # می‌آیند: عرض شانه ≈ ۰.۶۵ طول تنه، فاصلهٔ گوش‌ها ≈ ۰.۴۰ عرض شانه،
    # فاصلهٔ چشم‌ها ≈ ۰.۱۶ عرض شانه.
    # مرجعِ مقیاس باید نسبت به چرخشِ بدن **ثابت** باشد، وگرنه سنجشِ
    # کوتاه‌شدگی بی‌معنی می‌شود. طول تنه (شانه تا لگن) با چرخش حول
    # محور عمودی تقریباً تغییر نمی‌کند، پس مرجعِ درستی است.
    #
    # اشتباهی که اینجا کردم و اصلاح شد: اول عرضِ *فعلیِ* شانه را مرجعِ
    # فاصلهٔ گوش‌ها گرفته بودم. ولی در نیم‌رخ، عرض شانه هم کوچک می‌شود،
    # پس نسبت همیشه ۱.۰ درمی‌آمد و کوتاه‌شدگی اصلاً دیده نمی‌شد.
    torso = _torso_length(kp, conf_th)
    body = 0.65 * torso if torso else None      # عرضِ شانه در حالت روبه‌رو

    # ---- ۱) چیرالیتهٔ شانه‌ها — پایه‌ای‌ترین و مقاوم‌ترین سیگنال -------
    # روی فردِ نقاب‌دار هم کار می‌کند، چون به صورت کاری ندارد.
    c = _chirality(kp, K.LSHO, K.RSHO, conf_th, body)
    if c is not None:
        votes.append((c[0], 1.00 * c[1]))

    # ---- ۲) چیرالیتهٔ گوش‌ها — نزدیک‌تر به سر، پس دقیق‌تر --------------
    c = _chirality(kp, K.LEAR, K.REAR, conf_th, 0.40 * body if body else None)
    if c is not None:
        votes.append((c[0], 0.85 * c[1]))

    # ---- ۳) چیرالیتهٔ چشم‌ها — قوی ولی فقط وقتی صورت باز است ----------
    c = _chirality(kp, K.LEYE, K.REYE, conf_th, 0.16 * body if body else None)
    if c is not None:
        votes.append((c[0], 0.70 * c[1]))

    # ---- ۴) سرنخِ دیده‌شدن — عمداً کم‌وزن ------------------------------
    # اگر این را پروزن کنیم، فردِ بالاکلاوا‌پوش (که چشم و بینی ندارد)
    # به‌اشتباه «پشت به دوربین» تشخیص داده می‌شود و کاملاً نادیده گرفته
    # می‌شود — یعنی دقیقاً همان خطایی که می‌خواهیم از آن فرار کنیم.
    if eyes_v > 0.05 or nose_v > 0.05 or ears_v > 0.05:
        v_vis = float(np.clip((0.6 * eyes_v + 0.6 * nose_v - 0.5 * ears_v) * 1.6,
                              -1.0, 1.0))
        votes.append((v_vis, 0.30))

    if not votes:
        return Orientation(0.0, 0.0, "unknown", face_vis)

    w_sum = sum(w for _, w in votes)
    facing = sum(v * w for v, w in votes) / max(1e-6, w_sum)

    # اطمینان = هم به کیفیتِ کی‌پوینت‌ها بستگی دارد، هم به اینکه رأی‌ها
    # چقدر قاطع بوده‌اند. نیم‌رخ ذاتاً رأی‌های نزدیک صفر می‌دهد، پس
    # اطمینانِ پایین می‌گیرد — که همان رفتار درست است.
    decisiveness = sum(abs(v) * w for v, w in votes) / max(1e-6, w_sum)
    confidence = float(np.clip(min(1.0, w_sum / 1.6) * (0.35 + 0.65 * decisiveness),
                               0.0, 1.0))

    if confidence < min_conf:
        mode = "unknown"
    elif facing <= back_th:
        mode = "back"
    elif facing >= front_th:
        mode = "frontal"
    else:
        mode = "profile"

    return Orientation(float(facing), confidence, mode, face_vis)

### ★ تراز صحیح صورت + کیفیت مشاهده
<sub>`security_core/perception/face_align.py`</sub>

In [ ]:
# ==========================================================================
#  ★ تراز صحیح صورت + کیفیت مشاهده
#  (منبع: security_core/perception/face_align.py)
# ==========================================================================
from __future__ import annotations

"""
face_align.py — تراز و کروپ صورت + سنجش کیفیت مشاهده.

★ رفعِ باگ بحرانی نسخهٔ قبلی (چرخش ۱۸۰ درجه)
--------------------------------------------
کد قبلی این بود:

    dy, dx = reye[1] - leye[1], reye[0] - leye[0]
    angle  = degrees(atan2(dy, dx))

در استاندارد COCO-17، اندیس ۱ (LEYE) چشمِ چپِ *خودِ شخص* است که وقتی
روبه‌روی دوربین می‌ایستد، در تصویر سمت *راست* دیده می‌شود. بنابراین
برای یک صورتِ روبه‌روی کاملاً عادی:

    reye.x = 90 , leye.x = 110  →  dx = -20 , dy = 0
    angle  = atan2(0, -20) = 180°

یعنی تصویر ۱۸۰ درجه می‌چرخید و کروپ به‌جای صورت، «پیشانی و موی
وارونه» را برمی‌داشت. دو پیامد زنجیره‌ای داشت:
  • طبقه‌بند ماسک یک تصویر بی‌معنی می‌دید → خروجی تصادفی
  • تابع skin_ratio روی موی سر اجرا می‌شد → همیشه «مشکوک»

اینجا جهت بر اساس مختصات x در *تصویر* تعیین می‌شود، نه بر اساس
برچسب چپ/راستِ آناتومیک.
"""

import math
from typing import Optional

import cv2
import numpy as np


K = Keypoints


def _eye_roll_angle(kp: Keypoints, conf_th: float) -> Optional[float]:
    """
    زاویهٔ کج‌شدگی سر بر حسب درجه، یا None اگر چشم‌ها قابل اتکا نیستند.

    نکتهٔ اصلاح‌شده: مرتب‌سازی بر اساس x در تصویر — هرکدام که چپ‌تر
    است مبدأ بردار می‌شود. با این کار برای صورت روبه‌روی صاف، زاویه
    صفر می‌شود (نه ۱۸۰).
    """
    if kp.c(K.LEYE) < conf_th or kp.c(K.REYE) < conf_th:
        return None
    a, b = kp.pt(K.LEYE), kp.pt(K.REYE)
    left, right = (a, b) if a[0] <= b[0] else (b, a)   # ← اصلاح باگ
    dx, dy = right[0] - left[0], right[1] - left[1]
    if math.hypot(dx, dy) < 2.0:
        return None
    ang = math.degrees(math.atan2(dy, dx))
    # کج‌شدگی بیش از ۴۵ درجه یعنی کی‌پوینت‌ها اشتباه‌اند (یا فرد افتاده)؛
    # در آن حالت چرخش نمی‌دهیم تا تصویر خراب‌تر نشود.
    return ang if abs(ang) <= 45.0 else None


def extract_face(frame_clean: np.ndarray,
                 head: HeadROI,
                 kp: Optional[Keypoints],
                 cfg: FaceQualityCfg) -> Optional[FaceObservation]:
    """
    از فریمِ *تمیز* (بدون هیچ نقاشی روی آن) کروپ صورت را می‌سازد.

    چرا «تمیز» تأکید شده؟ در نسخهٔ قبلی، کروپِ سوژهٔ مشکوک از فریمی
    گرفته می‌شد که مستطیل رنگی، خطوط اسکلت و متن روی آن کشیده شده بود.
    آن تصاویر هم برای ارائه به کارفرما زشت بودند و هم برای استفادهٔ
    بعدی به‌عنوان دیتاست آموزشی بی‌ارزش.

    خروجی None یعنی «این مشاهده آن‌قدر بی‌کیفیت است که حتی نباید
    رأی ضعیف هم بدهد» (مثلاً صورت ۱۵ پیکسلی).
    """
    H, W = frame_clean.shape[:2]
    x1, y1, x2, y2 = head.xyxy
    hw, hh = x2 - x1, y2 - y1

    # --- گیت اندازه: کروپ خیلی کوچک = نویز خالص برای طبقه‌بند -------------
    if min(hw, hh) < cfg.min_face_px:
        return None

    angle = _eye_roll_angle(kp, cfg.align_eye_conf) if kp is not None else None

    if angle is None or abs(angle) < 2.0:
        # چرخش لازم نیست (یا ممکن نیست) — مستقیم کروپ می‌کنیم
        crop = safe_crop(frame_clean, head.xyxy)
        aligned = False
    else:
        # ناحیه‌ای بزرگ‌تر برمی‌داریم تا بعد از چرخش، گوشه‌ها سیاه نشوند
        px1, py1, px2, py2 = expand_box(head.xyxy, 1.45, W, H)
        pad = frame_clean[py1:py2, px1:px2]
        if pad.size == 0:
            return None
        # حول مرکزِ جعبهٔ سر می‌چرخانیم (نه مرکز چشم‌ها) تا کادربندی ثابت بماند
        cx = (x1 + x2) / 2.0 - px1
        cy = (y1 + y2) / 2.0 - py1
        M = cv2.getRotationMatrix2D((cx, cy), angle, 1.0)
        rot = cv2.warpAffine(pad, M, (pad.shape[1], pad.shape[0]),
                             flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
        crop = safe_crop(rot, (cx - hw / 2, cy - hh / 2, cx + hw / 2, cy + hh / 2))
        aligned = True

    if crop.size == 0 or min(crop.shape[:2]) < 8:
        return None

    # --- سنجه‌های کیفیت -------------------------------------------------
    px = float(min(crop.shape[0], crop.shape[1]))
    s_size = normalize_range(px, cfg.min_face_px, cfg.good_face_px)
    s_sharp = normalize_range(sharpness(crop), cfg.min_sharpness, cfg.good_sharpness)
    s_light = brightness_ok(crop)
    s_roi = head.reliability

    # وزن‌ها تجربی‌اند و در yaml قابل تغییر نیستند چون بخشی از تعریف
    # «کیفیت» هستند، نه یک آستانهٔ عملیاتی. اندازه و وضوح مهم‌ترین‌اند.
    quality = 0.34 * s_size + 0.30 * s_sharp + 0.14 * s_light + 0.22 * s_roi

    if quality < cfg.min_quality_to_vote:
        return None

    return FaceObservation(
        crop=crop,
        head_roi=head,
        quality=float(np.clip(quality, 0.0, 1.0)),
        quality_parts={"size": s_size, "sharp": s_sharp,
                       "light": s_light, "roi": s_roi},
        aligned=aligned,
    )

### YOLO11-pose + ردیابی
<sub>`security_core/perception/pose_detector.py`</sub>

In [ ]:
# ==========================================================================
#  YOLO11-pose + ردیابی
#  (منبع: security_core/perception/pose_detector.py)
# ==========================================================================
from __future__ import annotations

"""
pose_detector.py — لایهٔ ادراک پایه: تشخیص فرد + ۱۷ کی‌پوینت + ردیابی.

چرا یک مدل به‌جای دو مدل؟
--------------------------
اگر detector و pose جدا باشند، تصویر دو بار encode می‌شود که گران‌ترین
بخش کار است. YOLO11-pose هر دو را در یک forward می‌دهد. این تصمیمِ
درستِ نسخهٔ قبلی بود و حفظ شده است.

تفاوت مهم با نسخهٔ قبلی: آنجا `model.track(source=path, stream=True)`
صدا زده می‌شد، یعنی خود ultralytics فایل را باز می‌کرد. نتیجه‌اش این
بود که خط لوله فقط با «فایل» کار می‌کرد. اینجا فریم‌به‌فریم کار می‌کنیم
(`model.track(frame, persist=True)`) تا همین کد بدون تغییر با فایل،
وبکم، RTSP و چند دوربین همزمان کار کند.
"""

from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np



class RawDetection:
    """خروجی خام یک فرد در یک فریم (قبل از هر تحلیلی)."""
    __slots__ = ("track_id", "bbox", "conf", "kpts")

    def __init__(self, track_id: int, bbox: Tuple[int, int, int, int],
                 conf: float, kpts: Optional[Keypoints]):
        self.track_id = track_id
        self.bbox = bbox
        self.conf = conf
        self.kpts = kpts


class PoseDetector:
    def __init__(self, cfg: AppConfig, device: str, project_root: Optional[Path] = None):
        from ultralytics import YOLO

        self.cfg = cfg
        self.device = device
        self.root = project_root or Path.cwd()

        self.model = YOLO(cfg.pose.weights)
        self.model.to(device)

        # مسیر فایل ردیاب: اگر پیدا نشد یا نسخهٔ ultralytics نپذیرفت،
        # به‌صورت خودکار به bytetrack برمی‌گردیم (بدون کرش).
        p = (self.root / cfg.tracker.cfg_path)
        self._tracker_cfg = str(p) if p.exists() else cfg.tracker.fallback_cfg
        self._tracker_ok = True

        self._half = bool(cfg.runtime.half and device.startswith("cuda"))

    # ------------------------------------------------------------------ #
    def warmup(self, w: int = 640, h: int = 384, n: int = 2):
        """چند forward خالی تا کرنل‌های CUDA کامپایل و کش شوند."""
        dummy = np.zeros((h, w, 3), dtype=np.uint8)
        for _ in range(n):
            try:
                self.model.predict(dummy, imgsz=self.cfg.pose.imgsz, verbose=False,
                                   device=self.device, half=self._half)
            except Exception:
                break

    # ------------------------------------------------------------------ #
    def track(self, frame: np.ndarray) -> List[RawDetection]:
        """
        یک فریم را پردازش می‌کند و لیست افراد ردیابی‌شده را برمی‌گرداند.
        `persist=True` یعنی وضعیت ردیاب بین فراخوانی‌ها حفظ می‌شود.
        """
        H, W = frame.shape[:2]
        kwargs = dict(
            source=frame,
            classes=[0],                      # فقط «person»
            conf=self.cfg.pose.conf,
            iou=self.cfg.pose.iou,
            imgsz=self.cfg.pose.imgsz,
            persist=True,
            verbose=False,
            device=self.device,
            half=self._half,
        )

        if self._tracker_ok:
            try:
                res = self.model.track(tracker=self._tracker_cfg, **kwargs)[0]
            except Exception as e:      # نسخهٔ ultralytics قدیمی یا کلید ناشناخته در yaml
                print(f"[pose] ⚠️ ردیاب '{self._tracker_cfg}' کار نکرد ({e}). "
                      f"بازگشت به '{self.cfg.tracker.fallback_cfg}'.")
                self._tracker_cfg = self.cfg.tracker.fallback_cfg
                self._tracker_ok = False
                res = self.model.track(tracker=self._tracker_cfg, **kwargs)[0]
        else:
            res = self.model.track(tracker=self._tracker_cfg, **kwargs)[0]

        out: List[RawDetection] = []
        if res.boxes is None or len(res.boxes) == 0:
            return out

        boxes = res.boxes.xyxy.cpu().numpy()
        confs = res.boxes.conf.cpu().numpy()
        # اگر ردیاب هنوز ID نداده (فریم‌های اول)، ID موقت منفی می‌دهیم تا
        # فرد از قلم نیفتد؛ لایهٔ هویت این‌ها را نادیده می‌گیرد.
        if res.boxes.id is not None:
            ids = res.boxes.id.int().cpu().numpy()
        else:
            ids = -np.arange(1, len(boxes) + 1)

        kxy = kconf = None
        if res.keypoints is not None and res.keypoints.xy is not None:
            kxy = res.keypoints.xy.cpu().numpy()
            kconf = (res.keypoints.conf.cpu().numpy()
                     if res.keypoints.conf is not None
                     else np.ones(kxy.shape[:2], dtype=np.float32))

        min_area = self.cfg.pose.min_person_area_ratio * (W * H)

        for i in range(len(boxes)):
            bb = clamp_box(boxes[i], W, H, min_size=2)
            if (bb[2] - bb[0]) * (bb[3] - bb[1]) < min_area:
                continue        # افراد خیلی دور فقط نویز تولید می‌کنند
            kp = None
            if kxy is not None and i < len(kxy):
                kp = Keypoints(xy=kxy[i].astype(np.float32),
                               conf=kconf[i].astype(np.float32))
            out.append(RawDetection(int(ids[i]), bb, float(confs[i]), kp))
        return out

    # ------------------------------------------------------------------ #
    def reset_tracker(self):
        """شروع دوربین/ویدیوی جدید — وضعیت ردیاب باید پاک شود."""
        try:
            if hasattr(self.model, "predictor") and self.model.predictor is not None:
                trackers = getattr(self.model.predictor, "trackers", None)
                if trackers:
                    for t in trackers:
                        t.reset()
        except Exception:
            pass

### قرارداد طبقه‌بند + پیش‌پردازش GPU
<sub>`security_core/perception/classifiers/base.py`</sub>

In [ ]:
# ==========================================================================
#  قرارداد طبقه‌بند + پیش‌پردازش GPU
#  (منبع: security_core/perception/classifiers/base.py)
# ==========================================================================
from __future__ import annotations

"""
base.py — قرارداد مشترک همهٔ طبقه‌بندهای وضعیت صورت.

هر backend (SigLIP، MobileNet دانش‌آموز، ONNX، ROI-Head فاز ۴) فقط
باید یک متد `classify` پیاده کند که برای هر کروپ، یک توزیع احتمال
روی کلاس‌های متعارف `FaceClass` برگرداند.

نتیجه: عوض کردن مدل = عوض کردن یک خط در yaml. هیچ کد منطقی
دست نمی‌خورد. این همان چیزی است که اجازه می‌دهد فاز ۳ و ۴ را
بدون بازنویسی پیاده کنیم.
"""

from abc import ABC, abstractmethod
from typing import Dict, List, Optional

import numpy as np



class FaceStateClassifier(ABC):
    """رابط پایه. name فقط برای لاگ و گزارش است."""

    name: str = "base"
    # اگر True باشد، این backend به‌جای کروپ تصویر، جعبهٔ سر می‌گیرد
    # و ویژگی‌ها را از نقشهٔ ویژگی مشترک YOLO برمی‌دارد (فاز ۴).
    needs_rois: bool = False

    @abstractmethod
    def classify(self, crops: List[np.ndarray]) -> List[Dict[FaceClass, float]]:
        """
        ورودی: لیست کروپ BGR (اندازه‌های مختلف مجاز است).
        خروجی: لیست هم‌طول از دیکشنری {FaceClass: احتمال}، مجموع = ۱.
        """
        raise NotImplementedError

    def warmup(self, n: int = 2, size: int = 224):
        dummy = [np.zeros((size, size, 3), dtype=np.uint8)] * 2
        for _ in range(n):
            try:
                self.classify(dummy)
            except Exception:
                break


# --------------------------------------------------------------------------- #
#                   کمکی: تبدیل دسته‌ای کروپ‌ها به تنسور GPU                     #
# --------------------------------------------------------------------------- #
def to_tensor_batch(crops: List[np.ndarray], size: int,
                    mean: np.ndarray, std: np.ndarray,
                    device: str, half: bool):
    """
    پیش‌پردازش سریع: resize با OpenCV، بقیهٔ کار روی GPU.

    نکتهٔ سرعت: نسخهٔ قبلی از `AutoImageProcessor` هاگینگ‌فیس روی
    لیست تصاویر PIL استفاده می‌کرد. آن مسیر پایتونِ خالص است و برای
    چند کروپ حدود ۸ میلی‌ثانیه CPU می‌خورد — بیشتر از خودِ YOLO.
    این تابع همان کار را در حدود ۰.۵ میلی‌ثانیه انجام می‌دهد.
    """
    import torch

    arr = stack_crops(crops, size)                       # (N, S, S, 3) float32 RGB [0,1]
    t = torch.from_numpy(arr).to(device, non_blocking=True)
    t = t.permute(0, 3, 1, 2).contiguous()               # → NCHW
    m = torch.as_tensor(mean, device=device).view(1, 3, 1, 1)
    s = torch.as_tensor(std, device=device).view(1, 3, 1, 1)
    t = (t - m) / s
    return t.half() if half else t


def uniform_probs() -> Dict[FaceClass, float]:
    """توزیع «هیچ اطلاعی ندارم» — همهٔ وزن روی UNKNOWN."""
    p = {c: 0.0 for c in FaceClass.all()}
    p[FaceClass.UNKNOWN] = 1.0
    return p


def normalize(d: Dict[FaceClass, float]) -> Dict[FaceClass, float]:
    s = sum(max(0.0, v) for v in d.values())
    if s <= 1e-9:
        return uniform_probs()
    return {k: max(0.0, v) / s for k, v in d.items()}


def blend(a: Dict[FaceClass, float], b: Dict[FaceClass, float],
          w_b: float) -> Dict[FaceClass, float]:
    """
    ترکیب خطی دو توزیع. برای اضافه‌کردن «سرنخ ضعیف» به خروجی مدل
    استفاده می‌شود: w_b کوچک نگه داشته می‌شود تا سرنخ ضعیف هرگز
    نتواند نظر مدل را وارونه کند، فقط آن را کمی جابه‌جا می‌کند.
    """
    w_b = float(np.clip(w_b, 0.0, 1.0))
    return normalize({k: (1 - w_b) * a.get(k, 0.0) + w_b * b.get(k, 0.0)
                      for k in FaceClass.all()})


# --------------------------------------------------------------------------- #
#                                  کارخانه                                      #


def build_classifier(cfg, device: str) -> "FaceStateClassifier":
    """
    در این نوت‌بوک فقط backend صفر-شات وجود دارد، چون هدف «کارکردن
    بدون هیچ آموزشی» است. مدل‌های دانش‌آموز (فاز ۳) و سرِ RoI
    (فاز ۴) در پکیج کامل هستند و بعداً فقط با عوض‌کردن یک کلید
    فعال می‌شوند.
    """
    return SiglipZeroShotClassifier(cfg, device)


### سرنخ‌های کمکی (رنگی و کی‌پوینتی)
<sub>`security_core/perception/classifiers/weak_cues.py`</sub>

In [ ]:
# ==========================================================================
#  سرنخ‌های کمکی (رنگی و کی‌پوینتی)
#  (منبع: security_core/perception/classifiers/weak_cues.py)
# ==========================================================================
from __future__ import annotations

"""
weak_cues.py — سرنخ‌های تصویریِ ارزان، به‌عنوان شاهدِ کمکی (نه تصمیم‌گیرنده).

جایگزینِ `is_suspicious` نسخهٔ قبلی
-----------------------------------
کد قبلی این بود:

    upper = face[0:0.45*h, :]
    suspicious = skin_ratio(upper) < 0.12     # فقط HSV

سه ایراد داشت:
  ۱) ناحیهٔ بالای کروپ (به‌خصوص با باگ چرخش) عملاً «موی سر» بود،
     و موی سر هیچ‌وقت رنگ پوست ندارد → همه مشکوک می‌شدند.
  ۲) بازهٔ HSV با H∈[0,25] و V≥40 روی پوست تیره و نور کم شکست می‌خورد.
  ۳) روی دوربین مادون‌قرمزِ شبانه (تصویر تقریباً خاکستری) کاملاً بی‌معنی
     بود — و سرقت عمدتاً شب اتفاق می‌افتد.

اصلاحات
-------
  • مقایسهٔ «نوار پیشانی» با «نوار گونه/دهان» به‌جای یک ناحیهٔ تنها.
    این همان چیزی است که چشم انسان می‌بیند:
        پیشانیِ پوست + پایینِ پارچه  →  ماسک پزشکی
        پیشانیِ پارچه + پایینِ پارچه →  پوشش کامل (بالاکلاوا)
        هر دو پوست                  →  صورت باز
  • تشخیص پوست با YCrCb (مقاوم‌تر به روشنایی) + رأی دوم HSV.
  • ★ اگر تصویر عملاً خاکستری باشد (دوربین IR شبانه)، این ماژول
    صریحاً می‌گوید «نظری ندارم» و وزن صفر برمی‌گرداند — به‌جای اینکه
    خروجی بی‌معنی بدهد. این تفاوت یک ابزار اسباب‌بازی با یک محصول است.
  • خروجی همیشه با وزنِ کم وارد fusion می‌شود، پس حتی اگر اشتباه کند
    نمی‌تواند نظر مدل اصلی را وارونه کند.
"""

from typing import Dict, Optional, Tuple

import cv2
import numpy as np



def is_effectively_grayscale(bgr: np.ndarray, tol: float = 6.0) -> bool:
    """
    آیا تصویر رنگ واقعی دارد؟ در حالت IR شبانه، سه کانال تقریباً یکسان‌اند.
    روی نسخهٔ کوچک‌شده حساب می‌شود تا هزینه ناچیز بماند.
    """
    if bgr is None or bgr.size == 0 or bgr.ndim != 3:
        return True
    small = cv2.resize(bgr, (32, 32), interpolation=cv2.INTER_AREA).astype(np.float32)
    b, g, r = small[..., 0], small[..., 1], small[..., 2]
    return float(np.mean(np.abs(r - g)) + np.mean(np.abs(g - b))) < tol


def skin_likeness(region_bgr: np.ndarray) -> float:
    """
    نسبت پیکسل‌های «پوست‌مانند» (۰..۱) با ترکیب دو فضای رنگی.
    YCrCb رأی اصلی است چون کمتر به روشنایی حساس است.
    """
    if region_bgr is None or region_bgr.size == 0:
        return 0.0

    ycrcb = cv2.cvtColor(region_bgr, cv2.COLOR_BGR2YCrCb)
    m1 = cv2.inRange(ycrcb, np.array([40, 133, 77], np.uint8),
                            np.array([250, 178, 130], np.uint8))

    hsv = cv2.cvtColor(region_bgr, cv2.COLOR_BGR2HSV)
    # دو بازهٔ هیو (قرمز در دو انتهای چرخه قرار می‌گیرد)
    m2a = cv2.inRange(hsv, np.array([0, 25, 40], np.uint8),
                           np.array([25, 200, 255], np.uint8))
    m2b = cv2.inRange(hsv, np.array([160, 25, 40], np.uint8),
                           np.array([180, 200, 255], np.uint8))
    m2 = cv2.bitwise_or(m2a, m2b)

    both = cv2.bitwise_and(m1, m2)
    either = cv2.bitwise_or(m1, m2)
    # رأی محافظه‌کارانه: اشتراک وزن کامل، اجتماع وزن نصف
    score = (np.count_nonzero(both) + 0.5 * np.count_nonzero(either)) / (1.5 * m1.size)
    return float(np.clip(score, 0.0, 1.0))


def _bands(face_bgr: np.ndarray) -> Optional[Tuple[np.ndarray, np.ndarray]]:
    """
    دو نوار افقی از کروپِ سر برمی‌دارد:
      upper = ناحیهٔ پیشانی/ابرو  (۱۸٪ تا ۴۰٪ ارتفاع — موی بالای سر حذف می‌شود)
      lower = ناحیهٔ بینی/دهان/چانه (۵۵٪ تا ۸۵٪)
    حاشیهٔ چپ و راست هم بریده می‌شود تا پس‌زمینه وارد محاسبه نشود.
    """
    if face_bgr is None or face_bgr.size == 0:
        return None
    h, w = face_bgr.shape[:2]
    if h < 24 or w < 24:
        return None
    x0, x1 = int(0.20 * w), int(0.80 * w)
    up = face_bgr[int(0.18 * h):int(0.40 * h), x0:x1]
    lo = face_bgr[int(0.55 * h):int(0.85 * h), x0:x1]
    if up.size == 0 or lo.size == 0:
        return None
    return up, lo


def occlusion_cue(face_bgr: np.ndarray) -> Tuple[Optional[Dict[FaceClass, float]], float]:
    """
    خروجی: (توزیع احتمال روی کلاس‌ها، وزنِ اعتماد ۰..۱)

    وزن صفر یعنی «این سرنخ در این شرایط بی‌اعتبار است، نادیده بگیر»
    (تصویر خاکستری/IR، کروپ خیلی کوچک، یا نور نامناسب).
    """
    bands = _bands(face_bgr)
    if bands is None:
        return None, 0.0
    if is_effectively_grayscale(face_bgr):
        return None, 0.0        # ★ دوربین IR — صریحاً سکوت می‌کنیم

    up, lo = bands
    s_up = skin_likeness(up)     # پیشانی چقدر پوست است؟
    s_lo = skin_likeness(lo)     # پایین صورت چقدر پوست است؟

    # جدول تصمیم به‌صورت نرم (بدون آستانهٔ سخت):
    #   بالا پوست  + پایین پوست   → صورت باز
    #   بالا پوست  + پایین پارچه  → ماسک پزشکی
    #   بالا پارچه + پایین پارچه  → پوشش کامل
    #   بالا پارچه + پایین پوست   → نامتعارف (مو روی پیشانی) → نامشخص
    p_clear = s_up * s_lo
    p_medical = s_up * (1.0 - s_lo)
    p_full = (1.0 - s_up) * (1.0 - s_lo)
    p_unknown = (1.0 - s_up) * s_lo

    probs = normalize({
        FaceClass.CLEAR: p_clear,
        FaceClass.MEDICAL_MASK: p_medical,
        FaceClass.FULL_COVER: p_full,
        FaceClass.BACK_HEAD: 0.0,       # این سرنخ دربارهٔ جهت سر چیزی نمی‌داند
        FaceClass.UNKNOWN: p_unknown,
    })

    # هرچه تفکیک دو نوار واضح‌تر باشد، به سرنخ بیشتر اعتماد می‌کنیم.
    confidence = float(np.clip(abs(s_up - s_lo) * 1.6 + 0.15, 0.0, 1.0))
    return probs, confidence


def keypoint_occlusion_cue(vis: Dict[str, float], facing: float
                           ) -> Tuple[Optional[Dict[FaceClass, float]], float]:
    """
    سرنخ دومِ ارزان، و مهم‌تر از آن: **مستقل از رنگ**، پس در IR هم کار می‌کند.

    منطق: اگر فرد رو به دوربین است (facing بالا) ولی مدل pose هیچ
    کی‌پوینتی از صورتش پیدا نمی‌کند، این خودش شاهدِ پوشیده‌بودن است.
    برعکس، اگر پشت به دوربین است، نبودِ صورت کاملاً طبیعی است.

    این دقیقاً همان اصلی است که در نسخهٔ قبلی وارونه اعمال می‌شد:
    آنجا نبودِ کی‌پوینت باعث «نادیده گرفتن فرد» می‌شد.
    """
    if not vis:
        return None, 0.0

    eyes, nose, ears = vis.get("eyes", 0.0), vis.get("nose", 0.0), vis.get("ears", 0.0)
    face_vis = 0.5 * eyes + 0.3 * nose + 0.2 * ears

    if facing < 0.25:
        # واضحاً پشت/نیم‌رخ شدید
        return normalize({
            FaceClass.CLEAR: 0.05, FaceClass.MEDICAL_MASK: 0.05,
            FaceClass.FULL_COVER: 0.10, FaceClass.BACK_HEAD: 0.70,
            FaceClass.UNKNOWN: 0.10,
        }), float(np.clip(0.6 * (1.0 - facing), 0.0, 0.7))

    # رو به دوربین است
    if face_vis < 0.25:
        # رو به دوربین ولی هیچ صورتی دیده نمی‌شود → پوشش کامل محتمل
        return normalize({
            FaceClass.CLEAR: 0.05, FaceClass.MEDICAL_MASK: 0.20,
            FaceClass.FULL_COVER: 0.55, FaceClass.BACK_HEAD: 0.10,
            FaceClass.UNKNOWN: 0.10,
        }), float(np.clip(facing * 0.65, 0.0, 0.65))

    if eyes > 0.55 and nose < 0.30:
        # چشم‌ها پیدا، بینی پوشیده → ماسک (پزشکی یا بالاکلاوا با شکاف چشم)
        return normalize({
            FaceClass.CLEAR: 0.10, FaceClass.MEDICAL_MASK: 0.45,
            FaceClass.FULL_COVER: 0.35, FaceClass.BACK_HEAD: 0.02,
            FaceClass.UNKNOWN: 0.08,
        }), 0.35

    if face_vis > 0.65:
        return normalize({
            FaceClass.CLEAR: 0.70, FaceClass.MEDICAL_MASK: 0.15,
            FaceClass.FULL_COVER: 0.05, FaceClass.BACK_HEAD: 0.02,
            FaceClass.UNKNOWN: 0.08,
        }), 0.40

    return None, 0.0

### طبقه‌بند ۵ کلاسهٔ صفر-شات
<sub>`security_core/perception/classifiers/siglip_zeroshot_backend.py`</sub>

In [ ]:
# ==========================================================================
#  طبقه‌بند ۵ کلاسهٔ صفر-شات
#  (منبع: security_core/perception/classifiers/siglip_zeroshot_backend.py)
# ==========================================================================
from __future__ import annotations

"""
siglip_zeroshot_backend.py — طبقه‌بند چندکلاسهٔ صفر-شات با SigLIP.

چرا این backend نقش کلیدی دارد
------------------------------
مشکل بنیادی نسخهٔ قبلی: مدل `Face-Mask-Detection` فقط **دوکلاسه** است
(mask / no_mask). با یک مدل دوکلاسه هرگز نمی‌توان «ماسک پزشکی» را از
«ماسک دزدی» تفکیک کرد — و همین تفکیک، تمام ارزش محصول است.
به همین دلیل نسخهٔ قبلی مجبور شده بود از یک قانون رنگیِ دست‌ساز
(`skin_ratio`) استفاده کند که شکننده است.

راه‌حل: SigLIP یک مدل تصویر-متن است، پس می‌تواند **هر مجموعه کلاسی**
را بدون آموزش تشخیص دهد. کافی است برای هر کلاس چند جملهٔ توصیفی
بنویسیم. کلاس «بالاکلاوا / اسکی‌ماسک / کلاه‌کاسکت» دقیقاً همان
کلاس نادری است که دیتاست آماده برایش وجود ندارد.

نکتهٔ سرعت مهم
--------------
بردارهای متن **یک بار** در زمان راه‌اندازی محاسبه و کش می‌شوند.
پس هزینهٔ زمان اجرا دقیقاً برابر یک forward از برج تصویر است —
یعنی همان هزینهٔ حالت دوکلاسه، ولی با ۵ کلاس به‌جای ۲.
عملاً چندکلاسه‌شدن اینجا **رایگان** است.

نقش این مدل در چرخهٔ عمر محصول: «معلم».
در فاز ۳ با همین مدل هزاران کروپ را برچسب می‌زنیم و دانش آن را در
یک مدل کوچک (MobileNetV4) تقطیر می‌کنیم — که ~۵۰ برابر سریع‌تر است.
"""

from typing import Dict, List

import numpy as np


# --------------------------------------------------------------------------- #
#  مجموعهٔ پرامپت‌ها (prompt ensemble).
#  چند جمله به‌ازای هر کلاس، چون میانگین‌گیری روی چند توصیف، واریانس
#  خروجی صفر-شات را به‌شدت کم می‌کند (تکنیک استاندارد CLIP/SigLIP).
#  اینجا جای تیون‌کردن است: اگر روی ویدیوهای خودت خطای خاصی دیدی،
#  اول پرامپت‌ها را اصلاح کن، نه کد را.
# --------------------------------------------------------------------------- #
PROMPTS: Dict[FaceClass, List[str]] = {
    FaceClass.CLEAR: [
        "a photo of a person with a bare uncovered face",
        "a clear human face, nothing covering the mouth or nose",
        "a close-up of an unmasked face with visible mouth and chin",
        "a security camera photo of a person whose face is fully visible",
    ],
    FaceClass.MEDICAL_MASK: [
        "a person wearing a surgical medical face mask",
        "a face with a disposable mask covering the mouth and nose, forehead visible",
        "a person wearing a light blue medical mask, eyes and forehead uncovered",
        "a close-up of a face wearing an N95 respirator mask",
    ],
    FaceClass.FULL_COVER: [
        "a person wearing a black balaclava covering the entire face",
        "a robber wearing a ski mask with only the eyes showing",
        "a person whose whole head and face are covered with dark fabric",
        "a person wearing a full-face motorcycle helmet with the visor down",
        "a criminal with a scarf wrapped around the entire face and a hood",
    ],
    FaceClass.BACK_HEAD: [
        "the back of a person's head, only hair visible",
        "the rear view of a human head, no facial features at all",
        "a person seen from behind, back of the scalp facing the camera",
    ],
    # ★ اینجا یک اشتباهِ ظریف ولی ویرانگر وجود داشت.
    #   پرامپت‌های قبلی این‌ها بودند:
    #       "a blurry unrecognizable low resolution image"
    #       "a dark noisy photo where nothing can be identified"
    #   این جمله‌ها «شرایطِ تصویربرداریِ هر کروپِ دوربین مداربسته» را
    #   توصیف می‌کنند، نه یک کلاسِ معنایی. یعنی روی *همهٔ* کروپ‌های
    #   واقعی امتیاز بالایی می‌گرفتند و کلاس UNKNOWN برندهٔ دائمی
    #   می‌شد. نتیجه: هیچ‌وقت هیچ هشداری صادر نمی‌شد.
    #
    #   قاعدهٔ کلی برای پرامپت صفر-شات: کلاس‌ها باید بر اساس «چه چیزی
    #   در تصویر است» تفکیک شوند، نه «تصویر چقدر باکیفیت است».
    #   کیفیت، کارِ لایهٔ face_quality است و آنجا سنجیده می‌شود.
    FaceClass.UNKNOWN: [
        "a photo of an object that is not a person's head",
        "a close-up of a wall, floor, shelf or piece of furniture",
        "a random background texture with no person in it",
    ],
}


def _as_embedding(out):
    """
    خروجی get_text_features / get_image_features را به تنسور تبدیل می‌کند.

    چرا لازم است: در transformers نسخهٔ ۴ این متدها مستقیماً یک تنسور
    برمی‌گرداندند، ولی در نسخهٔ ۵ یک شیء `BaseModelOutputWithPooling`
    برمی‌گردانند. کدی که فرض کند تنسور است، با خطای
        AttributeError: 'BaseModelOutputWithPooling' object has no attribute 'float'
    می‌افتد. این تابع هر دو حالت را می‌پذیرد تا نوت‌بوک روی هر نسخه‌ای
    از کتابخانه اجرا شود — روی Colab نسخهٔ transformers بدون اطلاع قبلی
    عوض می‌شود و نباید کل خط لوله از کار بیفتد.
    """
    import torch
    if isinstance(out, torch.Tensor):
        return out.float()
    for attr in ("pooler_output", "image_embeds", "text_embeds", "last_hidden_state"):
        v = getattr(out, attr, None)
        if v is not None:
            # last_hidden_state سه‌بعدی است؛ میانگین روی توکن‌ها می‌گیریم
            return (v.mean(dim=1) if v.dim() == 3 else v).float()
    if isinstance(out, (tuple, list)) and len(out):
        return _as_embedding(out[0])
    raise TypeError(f"خروجی ناشناختهٔ مدل: {type(out)}")


class SiglipZeroShotClassifier(FaceStateClassifier):
    name = "siglip_zeroshot"

    def __init__(self, cfg, device: str):
        import torch
        from transformers import AutoModel, AutoTokenizer

        self.cfg = cfg
        self.device = device
        self.half = bool(cfg.runtime.half and device.startswith("cuda"))
        self.size = cfg.classifier.input_size

        model_id = cfg.classifier.siglip_zeroshot_model
        self.model = AutoModel.from_pretrained(model_id).to(device).eval()
        if self.half:
            self.model = self.model.half()
        tok = AutoTokenizer.from_pretrained(model_id)

        # SigLIP روی مقادیر نرمال‌سازی ۰.۵/۰.۵ آموزش دیده است
        self.mean = np.array([0.5, 0.5, 0.5], dtype=np.float32)
        self.std = np.array([0.5, 0.5, 0.5], dtype=np.float32)

        # ---- کش‌کردن بردارهای متن (فقط یک بار) --------------------------
        self.classes: List[FaceClass] = FaceClass.all()
        texts, self.slices = [], []
        for c in self.classes:
            start = len(texts)
            texts.extend(PROMPTS[c])
            self.slices.append((start, len(texts)))

        with torch.no_grad():
            # SigLIP الزاماً padding به max_length=64 می‌خواهد
            enc = tok(texts, padding="max_length", max_length=64,
                      truncation=True, return_tensors="pt").to(device)
            temb = _as_embedding(self.model.get_text_features(**enc))
            temb = temb / temb.norm(dim=-1, keepdim=True)
        self.text_emb = temb                                  # (T, D)
        # این دو پارامتر در بعضی نسخه‌ها روی خود مدل و در بعضی روی
        # زیرماژول‌ها قرار دارند؛ با مقدار پیش‌فرضِ امن fallback می‌گیریم.
        def _scalar(name: str, default: float) -> float:
            v = getattr(self.model, name, None)
            if v is None:
                v = getattr(getattr(self.model, "model", None), name, None)
            if v is None:
                return default
            return float(v.detach().float().reshape(-1)[0].item())

        self.logit_scale = float(np.exp(_scalar("logit_scale", 0.0))) or 1.0
        self.logit_bias = _scalar("logit_bias", 0.0)
        del tok
        print(f"[classifier] SigLIP zero-shot آماده شد "
              f"({len(texts)} پرامپت / {len(self.classes)} کلاس)")

    # ------------------------------------------------------------------ #
    def classify(self, crops: List[np.ndarray]) -> List[Dict[FaceClass, float]]:
        import torch
        if not crops:
            return []

        out: List[Dict[FaceClass, float]] = []
        bs = self.cfg.classifier.batch_max
        for i in range(0, len(crops), bs):
            chunk = crops[i:i + bs]
            with torch.inference_mode():
                px = to_tensor_batch(chunk, self.size, self.mean, self.std,
                                     self.device, self.half)
                iemb = _as_embedding(self.model.get_image_features(pixel_values=px))
                iemb = iemb / iemb.norm(dim=-1, keepdim=True)
                logits = iemb @ self.text_emb.T * self.logit_scale + self.logit_bias

                # میانگین لاجیت پرامپت‌های هر کلاس
                per_class = torch.stack(
                    [logits[:, a:b].mean(dim=1) for (a, b) in self.slices], dim=1)
                probs = torch.softmax(per_class, dim=1).cpu().numpy()

            for row in probs:
                out.append(normalize({c: float(row[j])
                                      for j, c in enumerate(self.classes)}))
        return out

### مدل دوکلاسهٔ فاین‌تیون‌شده (نسخهٔ اول)
<sub>`security_core/perception/classifiers/siglip_binary_backend.py`</sub>

In [ ]:
# ==========================================================================
#  مدل دوکلاسهٔ فاین‌تیون‌شده (نسخهٔ اول)
#  (منبع: security_core/perception/classifiers/siglip_binary_backend.py)
# ==========================================================================
from __future__ import annotations

"""
siglip_binary_backend.py — مدل دوکلاسهٔ فاین‌تیون‌شده (همان مدل نسخهٔ قبلی).

نگه داشته شده چون روی محورِ «ماسک دارد / ندارد» احتمالاً از صفر-شات
دقیق‌تر است. اما محدودیت ذاتی دارد: نمی‌تواند ماسک پزشکی را از
ماسک دزدی جدا کند. پس در این معماری فقط محور اول را می‌دهد و تفکیک
نوع پوشش به سرنخ‌های کمکی و لایهٔ fusion سپرده می‌شود.

★ رفع باگ نسخهٔ قبلی: نگاشت لیبل‌ها
------------------------------------
کد قبلی این خط را داشت:

    MASK_ID2LABEL = {0: "mask", 1: "no_mask"}     # ← hard-code

اگر ترتیب کلاس‌ها در چک‌پوینت برعکس باشد، **کل سیستم وارونه** کار
می‌کند و هیچ خطایی هم نمی‌دهد؛ فقط دقت پایین می‌آید و علتش پیدا
نمی‌شود. اینجا لیبل‌ها همیشه از `model.config.id2label` خوانده
می‌شوند و متن آن‌ها نرمال‌سازی می‌شود.
"""

import re
from typing import Dict, List

import numpy as np


_NO_MASK_PAT = re.compile(r"(no[_\-\s]?mask|without|unmask|bare|clear)", re.I)
_MASK_PAT = re.compile(r"(mask|cover|face[_\-\s]?mask)", re.I)


class SiglipBinaryClassifier(FaceStateClassifier):
    name = "siglip_binary"

    def __init__(self, cfg, device: str):
        import torch
        from transformers import AutoImageProcessor, SiglipForImageClassification

        self.cfg = cfg
        self.device = device
        self.half = bool(cfg.runtime.half and device.startswith("cuda"))

        model_id = cfg.classifier.siglip_binary_model
        self.model = SiglipForImageClassification.from_pretrained(model_id).to(device).eval()
        if self.half:
            self.model = self.model.half()

        # از پردازشگر فقط *پارامترها* را می‌خوانیم، نه اینکه در حلقه استفاده‌اش کنیم
        try:
            proc = AutoImageProcessor.from_pretrained(model_id)
            self.mean = np.array(getattr(proc, "image_mean", [0.5, 0.5, 0.5]), dtype=np.float32)
            self.std = np.array(getattr(proc, "image_std", [0.5, 0.5, 0.5]), dtype=np.float32)
            sz = getattr(proc, "size", None)
            self.size = int(sz.get("height", cfg.classifier.input_size)) if isinstance(sz, dict) \
                else cfg.classifier.input_size
        except Exception:
            self.mean = np.array([0.5, 0.5, 0.5], dtype=np.float32)
            self.std = np.array([0.5, 0.5, 0.5], dtype=np.float32)
            self.size = cfg.classifier.input_size

        # ---- ★ نگاشت امنِ لیبل‌ها -----------------------------------
        id2label = getattr(self.model.config, "id2label", None) or {0: "0", 1: "1"}
        self.idx_no_mask, self.idx_mask = None, None
        for i, lab in id2label.items():
            s = str(lab)
            if _NO_MASK_PAT.search(s):
                self.idx_no_mask = int(i)
            elif _MASK_PAT.search(s):
                self.idx_mask = int(i)
        if self.idx_no_mask is None or self.idx_mask is None:
            # اگر متن لیبل‌ها گویا نبود، به ترتیب پیش‌فرض برمی‌گردیم ولی
            # صریحاً هشدار می‌دهیم تا کاربر دستی تأیید کند.
            print(f"[classifier] ⚠️ لیبل‌های مدل قابل تشخیص نبودند: {id2label}. "
                  f"فرض: 0=mask, 1=no_mask — لطفاً دستی راستی‌آزمایی کن.")
            self.idx_mask, self.idx_no_mask = 0, 1
        else:
            print(f"[classifier] نگاشت لیبل از روی چک‌پوینت خوانده شد: {id2label}")

    # ------------------------------------------------------------------ #
    def classify_binary(self, crops: List[np.ndarray]):
        """
        خروجی خام: [(p_صورت‌باز, p_پوشیده), ...]

        حالت ترکیبی (hybrid_backend) از این متد استفاده می‌کند، چون
        فقط همین یک محور را از این مدل می‌خواهد و تقسیمِ حدسیِ
        «پزشکی/دزدی» را که در classify انجام می‌شود لازم ندارد.
        """
        import torch
        if not crops:
            return []
        out = []
        bs = self.cfg.classifier.batch_max
        for i in range(0, len(crops), bs):
            with torch.inference_mode():
                px = to_tensor_batch(crops[i:i + bs], self.size, self.mean,
                                     self.std, self.device, self.half)
                logits = self.model(pixel_values=px).logits.float()
                probs = torch.softmax(logits, dim=1).cpu().numpy()
            for row in probs:
                out.append((float(row[self.idx_no_mask]), float(row[self.idx_mask])))
        return out

    # ------------------------------------------------------------------ #
    def classify(self, crops: List[np.ndarray]) -> List[Dict[FaceClass, float]]:
        import torch
        if not crops:
            return []

        out: List[Dict[FaceClass, float]] = []
        bs = self.cfg.classifier.batch_max
        for i in range(0, len(crops), bs):
            chunk = crops[i:i + bs]
            with torch.inference_mode():
                px = to_tensor_batch(chunk, self.size, self.mean, self.std,
                                     self.device, self.half)
                logits = self.model(pixel_values=px).logits.float()
                probs = torch.softmax(logits, dim=1).cpu().numpy()

            for row in probs:
                p_no = float(row[self.idx_no_mask])
                p_mask = float(row[self.idx_mask])
                # تقسیم اولیهٔ «پوشیده» بین دو زیرکلاس بر اساس نرخ پایه:
                # ماسک پزشکی به‌مراتب رایج‌تر از ماسک دزدی است.
                # لایهٔ fusion این نسبت را با سرنخ‌های کمکی اصلاح می‌کند.
                out.append(normalize({
                    FaceClass.CLEAR: p_no * 0.94,
                    FaceClass.MEDICAL_MASK: p_mask * 0.62,
                    FaceClass.FULL_COVER: p_mask * 0.38,
                    FaceClass.BACK_HEAD: p_no * 0.06,
                    FaceClass.UNKNOWN: 0.02,
                }))
        return out

### ★ طبقه‌بند ترکیبی — پیش‌فرض
<sub>`security_core/perception/classifiers/hybrid_backend.py`</sub>

In [ ]:
# ==========================================================================
#  ★ طبقه‌بند ترکیبی — پیش‌فرض
#  (منبع: security_core/perception/classifiers/hybrid_backend.py)
# ==========================================================================
from __future__ import annotations

"""
hybrid_backend.py — ★ ترکیب مدلِ فاین‌تیون‌شده با مدلِ صفر-شات.

چرا این backend ساخته شد
------------------------
درسِ گران‌قیمتی که از اجرای واقعی گرفتیم: مدل صفر-شات به‌تنهایی روی
کروپ‌های کوچک و نه‌چندان واضحِ دوربین مداربسته، از مدلِ دوکلاسهٔ
فاین‌تیون‌شدهٔ نسخهٔ اول **ضعیف‌تر** بود. این نتیجه منطقی است:

    مدل فاین‌تیون‌شده روی همان تسک و همان توزیع داده آموزش دیده.
    مدل صفر-شات باید معنا را از روی متن حدس بزند.

اما مدل دوکلاسه هم محدودیت ذاتی دارد: فقط «ماسک دارد / ندارد» را
می‌گوید و هرگز نمی‌تواند ماسک پزشکی را از ماسک دزدی جدا کند —
و همان تفکیک، تمام ارزش تجاری محصول است.

راهِ درست، انتخاب بین این دو نیست؛ **تجزیهٔ سؤال** است
--------------------------------------------------------
سؤال اصلی را به دو سؤال کوچک‌تر می‌شکنیم و هرکدام را به مدلی
می‌دهیم که در همان یکی قوی است:

    سؤال ۱: «صورت پوشیده است یا نه؟»
             → مدل فاین‌تیون‌شده. دقیق، پایدار، ارزان.

    سؤال ۲: «حالا که پوشیده است، *چه نوع* پوششی است؟»
             → مدل صفر-شات، ولی به‌صورت یک مقایسهٔ **دوتایی**
               (پزشکی در برابر پوشش کامل) نه یک رقابت ۵-کلاسه.

نکتهٔ کلیدی: مدل‌های صفر-شات در «تشخیصِ تفاوت بین دو گزینهٔ مشخص»
به‌مراتب قابل اتکاتر از «انتخاب یکی از پنج کلاس» هستند. با محدود
کردن نقش آن به سؤال دوم، از نقطه‌قوتش استفاده می‌کنیم و از
نقطه‌ضعفش دوری.

هزینه
-----
مدل صفر-شات فقط روی کروپ‌هایی اجرا می‌شود که مدل اول آن‌ها را
«پوشیده» تشخیص داده. در یک صحنهٔ عادی که بیشتر افراد صورت باز
دارند، این یعنی هزینهٔ اضافه نزدیک به صفر است.
"""

from typing import Dict, List, Optional

import numpy as np



class HybridMaskClassifier(FaceStateClassifier):
    name = "hybrid"

    def __init__(self, cfg, device: str):

        self.cfg = cfg
        # آستانه‌ای که تعیین می‌کند کدام کروپ‌ها به مدل دوم فرستاده شوند.
        # عمداً پایین است: هزینهٔ اجرای اضافه ناچیز است، ولی از دست دادن
        # یک نقاب‌دار گران تمام می‌شود.
        self.route_th = float(getattr(cfg.classifier, "hybrid_route_threshold", 0.25))

        print("[classifier] حالت ترکیبی — بارگذاری مدل دوکلاسهٔ فاین‌تیون‌شده ...")
        # اگر این مدل به هر دلیلی بارگذاری نشد (نبود شبکه، تغییر API
        # کتابخانه، حذف چک‌پوینت از هاب) کل سیستم نباید از کار بیفتد.
        # به حالت فقط-صفر-شات برمی‌گردیم و صریحاً اطلاع می‌دهیم.
        try:
            self.binary = SiglipBinaryClassifier(cfg, device)
        except Exception as e:
            print(f"[classifier] ⚠️ مدل دوکلاسه بارگذاری نشد ({type(e).__name__}: {e})")
            print("[classifier] ⚠️ بازگشت به حالت فقط صفر-شات — دقت پایین‌تر خواهد بود.")
            self.binary = None

        print("[classifier] حالت ترکیبی — بارگذاری مدل صفر-شات ...")
        self.zeroshot = SiglipZeroShotClassifier(cfg, device)

        if self.binary is None:
            self.name = "hybrid(zeroshot-only)"
        print(f"[classifier] ✅ حالت ترکیبی آماده — {self.name} "
              f"(مسیردهی به مدل دوم وقتی p(پوشیده) ≥ {self.route_th})")

    # ------------------------------------------------------------------ #
    def classify(self, crops: List[np.ndarray]) -> List[Dict[FaceClass, float]]:
        if not crops:
            return []

        if self.binary is None:
            return self.zeroshot.classify(crops)         # حالت اضطراری

        # ── گام ۱: محورِ «پوشیده / باز» از مدل فاین‌تیون‌شده ────────────
        raw = self.binary.classify_binary(crops)         # [(p_no_mask, p_mask), ...]

        # ── گام ۲: فقط کروپ‌های «احتمالاً پوشیده» به مدل دوم می‌روند ───
        idx = [i for i, (_, pm) in enumerate(raw) if pm >= self.route_th]
        zs: Dict[int, Dict[FaceClass, float]] = {}
        if idx:
            res = self.zeroshot.classify([crops[i] for i in idx])
            zs = {i: r for i, r in zip(idx, res)}

        # ── گام ۳: ترکیب ────────────────────────────────────────────────
        out: List[Dict[FaceClass, float]] = []
        for i, (p_no, p_mask) in enumerate(raw):
            z = zs.get(i)

            if z is None:
                # مدل اول با اطمینان می‌گوید صورت باز است.
                # نرخ پایهٔ کوچکی برای «پشت به دوربین» باقی می‌گذاریم،
                # چون مدلِ ماسک هرگز این حالت را نیاموخته است.
                out.append(normalize({
                    FaceClass.CLEAR: p_no * 0.92,
                    FaceClass.MEDICAL_MASK: p_mask * 0.60,
                    FaceClass.FULL_COVER: p_mask * 0.40,
                    FaceClass.BACK_HEAD: p_no * 0.08,
                    FaceClass.UNKNOWN: 0.02,
                }))
                continue

            # «این اصلاً سر نیست» یا «پشت سر است» — نظر مدل دوم که تنها
            # مدلی است که این دو مفهوم را می‌شناسد.
            p_back = z.get(FaceClass.BACK_HEAD, 0.0)
            p_unk = z.get(FaceClass.UNKNOWN, 0.0)
            off_topic = float(np.clip(p_back + p_unk, 0.0, 0.85))

            # نسبتِ «پزشکی به پوشش کامل» — یک مقایسهٔ دوتایی، نه ۵-کلاسه
            m = z.get(FaceClass.MEDICAL_MASK, 0.0)
            f = z.get(FaceClass.FULL_COVER, 0.0)
            r_med = m / (m + f) if (m + f) > 1e-6 else 0.60

            keep = 1.0 - off_topic
            out.append(normalize({
                FaceClass.CLEAR: p_no * keep,
                FaceClass.MEDICAL_MASK: p_mask * keep * r_med,
                FaceClass.FULL_COVER: p_mask * keep * (1.0 - r_med),
                FaceClass.BACK_HEAD: p_back,
                FaceClass.UNKNOWN: p_unk,
            }))
        return out

    # ------------------------------------------------------------------ #
    def warmup(self, n: int = 2, size: int = 224):
        if self.binary is not None:
            self.binary.warmup(n, size)
        self.zeroshot.warmup(n, size)

### امضای ظاهری برای شناسایی مجدد
<sub>`security_core/tracking/reid.py`</sub>

In [ ]:
# ==========================================================================
#  امضای ظاهری برای شناسایی مجدد
#  (منبع: security_core/tracking/reid.py)
# ==========================================================================
from __future__ import annotations

"""
reid.py — استخراج «امضای ظاهری» هر فرد برای شناسایی مجدد.

چرا لازم است
------------
ByteTrack (ردیاب نسخهٔ قبلی) هیچ مدل ظاهری ندارد؛ فقط IoU و حرکت.
نتیجه‌اش در دوربین واقعی:

  • فرد پشت قفسه می‌رود و برمی‌گردد → ID جدید → کل تاریخچه ریست
    → سوژهٔ قرمز دوباره خاکستری می‌شود.
  • دو نفر از هم رد می‌شوند → جابه‌جایی ID → **وضعیت «سبزِ» نفر اول
    به نفر ماسک‌دار منتقل می‌شود.** این یک حفرهٔ امنیتی است، نه یک
    اشکال کیفیت تصویر.

با یک بردار ظاهری می‌توان ID جدید را به هویت قبلی وصل کرد و
تاریخچه را حفظ کرد. (منطق اتصال در identity_bank.py است.)

سه backend با انتخاب خودکار
---------------------------
  torchreid   : OSNet — بهترین دقت، ولی وابستگی اضافه می‌خواهد
  torchvision : ویژگی MobileNetV3-Small — بدون وابستگی جدید، دقت خوب
  colorhist   : هیستوگرام رنگ بدن — تقریباً رایگان، برای اتصال کوتاه‌مدت کافی

انتخاب «auto» از بالا به پایین اولین موردِ در دسترس را برمی‌دارد.
این طراحی عمدی است: محصول نباید روی سیستم مشتری به‌خاطر نبودِ یک
کتابخانه اصلاً بالا نیاید.
"""

from typing import List, Optional

import cv2
import numpy as np



class BaseReID:
    name = "base"
    dim = 0

    def embed(self, crops: List[np.ndarray]) -> np.ndarray:
        raise NotImplementedError


# --------------------------------------------------------------------------- #
class ColorHistReID(BaseReID):
    """
    هیستوگرام رنگ در سه نوار افقی (سر / تنه / پا).
    نوارها مهم‌اند: بدون آن‌ها، «پیراهن آبی + شلوار مشکی» و
    «پیراهن مشکی + شلوار آبی» یکسان دیده می‌شوند.
    """
    name = "colorhist"

    def __init__(self, bins=(8, 8, 4), stripes: int = 3):
        self.bins = bins
        self.stripes = stripes
        self.dim = stripes * bins[0] * bins[1] * bins[2]

    def embed(self, crops: List[np.ndarray]) -> np.ndarray:
        out = np.zeros((len(crops), self.dim), dtype=np.float32)
        for i, c in enumerate(crops):
            if c is None or c.size == 0:
                continue
            hsv = cv2.cvtColor(cv2.resize(c, (64, 128), interpolation=cv2.INTER_AREA),
                               cv2.COLOR_BGR2HSV)
            h = hsv.shape[0] // self.stripes
            feats = []
            for s in range(self.stripes):
                part = hsv[s * h:(s + 1) * h]
                hist = cv2.calcHist([part], [0, 1, 2], None, self.bins,
                                    [0, 180, 0, 256, 0, 256])
                feats.append(cv2.normalize(hist, hist).flatten())
            v = np.concatenate(feats).astype(np.float32)
            n = np.linalg.norm(v)
            out[i] = v / n if n > 1e-8 else v
        return out


# --------------------------------------------------------------------------- #
class TorchvisionReID(BaseReID):
    """ویژگی سراسری MobileNetV3-Small از پیش‌آموزش ImageNet."""
    name = "torchvision"

    def __init__(self, device: str, half: bool, input_size=(128, 256)):
        import torch
        from torchvision.models import MobileNet_V3_Small_Weights, mobilenet_v3_small

        self.device, self.half = device, half
        self.w, self.h = input_size
        m = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT)
        self.backbone = torch.nn.Sequential(m.features,
                                            torch.nn.AdaptiveAvgPool2d(1),
                                            torch.nn.Flatten()).to(device).eval()
        if half:
            self.backbone = self.backbone.half()
        self.dim = 576
        self.mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        self.std = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    def embed(self, crops: List[np.ndarray]) -> np.ndarray:
        import torch
        if not crops:
            return np.zeros((0, self.dim), dtype=np.float32)
        batch = np.stack([
            cv2.cvtColor(letterbox_square(c, max(self.w, self.h)) if c.size else
                         np.zeros((self.h, self.w, 3), np.uint8), cv2.COLOR_BGR2RGB)
            for c in crops]).astype(np.float32) / 255.0
        with torch.inference_mode():
            t = torch.from_numpy(batch).to(self.device).permute(0, 3, 1, 2)
            m = torch.as_tensor(self.mean, device=self.device).view(1, 3, 1, 1)
            s = torch.as_tensor(self.std, device=self.device).view(1, 3, 1, 1)
            t = (t - m) / s
            if self.half:
                t = t.half()
            f = self.backbone(t).float()
            f = f / f.norm(dim=1, keepdim=True).clamp_min(1e-8)
        return f.cpu().numpy().astype(np.float32)


# --------------------------------------------------------------------------- #
class TorchreidOSNet(BaseReID):
    """OSNet-x0.25 — سبک‌ترین مدل واقعی ReID، دقت به‌مراتب بالاتر."""
    name = "torchreid"

    def __init__(self, device: str, half: bool, input_size=(128, 256)):
        import torch
        import torchreid

        self.device, self.half = device, half
        self.w, self.h = input_size
        self.model = torchreid.models.build_model("osnet_x0_25", num_classes=1000,
                                                  pretrained=True)
        self.model = self.model.to(device).eval()
        if half:
            self.model = self.model.half()
        self.dim = 512
        self.mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        self.std = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    def embed(self, crops: List[np.ndarray]) -> np.ndarray:
        import torch
        if not crops:
            return np.zeros((0, self.dim), dtype=np.float32)
        batch = np.stack([
            cv2.cvtColor(cv2.resize(c if c.size else np.zeros((self.h, self.w, 3), np.uint8),
                                    (self.w, self.h), interpolation=cv2.INTER_LINEAR),
                         cv2.COLOR_BGR2RGB) for c in crops]).astype(np.float32) / 255.0
        with torch.inference_mode():
            t = torch.from_numpy(batch).to(self.device).permute(0, 3, 1, 2)
            m = torch.as_tensor(self.mean, device=self.device).view(1, 3, 1, 1)
            s = torch.as_tensor(self.std, device=self.device).view(1, 3, 1, 1)
            t = (t - m) / s
            if self.half:
                t = t.half()
            f = self.model(t).float()
            f = f / f.norm(dim=1, keepdim=True).clamp_min(1e-8)
        return f.cpu().numpy().astype(np.float32)


# --------------------------------------------------------------------------- #
def build_reid(cfg, device: str) -> Optional[BaseReID]:
    """انتخاب backend با سلسله‌مراتب امن."""
    if not cfg.reid.enabled:
        return None

    want = cfg.reid.backend.lower()
    size = tuple(cfg.reid.input_size)
    half = bool(cfg.runtime.half and device.startswith("cuda"))
    order = [want] if want != "auto" else ["torchreid", "torchvision", "colorhist"]

    for b in order:
        try:
            if b == "torchreid":
                r = TorchreidOSNet(device, half, size)
            elif b == "torchvision":
                r = TorchvisionReID(device, half, size)
            elif b == "colorhist":
                r = ColorHistReID()
            else:
                continue
            print(f"[reid] backend فعال: {r.name} (dim={r.dim})")
            return r
        except Exception as e:
            print(f"[reid] {b} در دسترس نبود ({type(e).__name__}) — گزینهٔ بعدی")

    print("[reid] ⚠️ هیچ backend ای فعال نشد — شناسایی مجدد غیرفعال است")
    return None

### ★ هویت سراسری (global_id)
<sub>`security_core/tracking/identity_bank.py`</sub>

In [ ]:
# ==========================================================================
#  ★ هویت سراسری (global_id)
#  (منبع: security_core/tracking/identity_bank.py)
# ==========================================================================
from __future__ import annotations

"""
identity_bank.py — ★ لایهٔ هویتِ سراسری (global_id).

مهم‌ترین تصمیم معماری برای آیندهٔ محصول
---------------------------------------
`track_id` که ردیاب می‌دهد **کوتاه‌عمر** است: با هر انسداد، خروج از
کادر، یا تقاطع دو نفر عوض می‌شود. اگر وضعیت افراد را روی track_id
نگه داری (کاری که نسخهٔ قبلی می‌کرد)، هر بار که ردیاب سکسکه کند
تمام تاریخچه از دست می‌رود.

اینجا یک لایهٔ بالاتر اضافه می‌کنیم:

    track_id (کوتاه‌مدت، از ردیاب)  ──►  global_id (بلندمدت، پایدار)

همهٔ ماژول‌ها — چهره، اسلحه، آتش، رفتار — شواهدشان را روی
`global_id` می‌نویسند. این تنها راهی است که بتوان بعداً گفت
«این همان کسی است که ده ثانیه پیش ماسک داشت و حالا دستش را
زیر کاپشن برد». بدون این لایه، ادغام ماژول‌ها ممکن نیست.

سه کار انجام می‌دهد:
  ۱) اتصال مجدد پس از گم‌شدن (re-entry) با شباهت ظاهری
  ۲) تشخیص جابه‌جایی ID (وقتی ظاهرِ یک track ناگهان عوض می‌شود)
  ۳) جمع‌آوری زبالهٔ هویت‌های مرده — رفعِ نشتی حافظهٔ نسخهٔ قبلی
     که دیکشنری وضعیت را هیچ‌وقت پاک نمی‌کرد (برای دوربین ۲۴/۷ کشنده بود)
"""

from dataclasses import dataclass, field
from typing import Dict, List, Optional, Set, Tuple

import numpy as np



@dataclass
class Identity:
    gid: int
    created_t: float
    last_seen_t: float
    last_bbox: Tuple[int, int, int, int]
    gallery: List[np.ndarray] = field(default_factory=list)
    track_ids: Set[int] = field(default_factory=set)
    frames_seen: int = 0
    reconnects: int = 0          # چند بار پس از گم‌شدن دوباره وصل شده

    def similarity(self, emb: np.ndarray) -> float:
        """بیشترین شباهت به نمونه‌های گالری (نه میانگین — به تغییر ژست مقاوم‌تر است)."""
        if not self.gallery or emb is None:
            return 0.0
        return max(cosine_similarity(emb, g) for g in self.gallery)

    def add_embedding(self, emb: np.ndarray, cap: int):
        if emb is None:
            return
        self.gallery.append(emb)
        if len(self.gallery) > cap:
            # نمونهٔ دوم را دور می‌ریزیم، نه اولی: اولین نمونه معمولاً
            # لحظهٔ ورود و تمیزترین دید است و ارزش نگه‌داشتن دارد.
            self.gallery.pop(1)


class IdentityBank:
    # اگر شباهت بردار فعلی به گالریِ هویتِ منتسب زیر این عدد بیفتد،
    # احتمالاً ردیاب دو نفر را جابه‌جا کرده است.
    SWITCH_SUSPECT_SIM = 0.30
    SWITCH_CONFIRM_HITS = 3      # چند فریم پشت سر هم تا حکم قطعی شود

    def __init__(self, cfg):
        self.cfg = cfg
        self._next_gid = 1
        self.identities: Dict[int, Identity] = {}
        self._tid2gid: Dict[int, int] = {}
        self._switch_hits: Dict[int, int] = {}
        self.dead_gids: List[int] = []      # خط لوله بعد از هر فریم می‌خواندش

    # ------------------------------------------------------------------ #
    def _new_identity(self, tid: int, bbox, t: float, emb: Optional[np.ndarray]) -> int:
        gid = self._next_gid
        self._next_gid += 1
        ident = Identity(gid=gid, created_t=t, last_seen_t=t, last_bbox=bbox)
        ident.track_ids.add(tid)
        ident.add_embedding(emb, self.cfg.reid.gallery_per_identity)
        self.identities[gid] = ident
        self._tid2gid[tid] = gid
        return gid

    # ------------------------------------------------------------------ #
    def update(self, tid: int, bbox: Tuple[int, int, int, int], t: float,
               emb: Optional[np.ndarray] = None,
               active_gids: Optional[Set[int]] = None) -> int:
        """
        یک ردیابیِ فعال را به یک هویت سراسری نگاشت می‌کند و gid را برمی‌گرداند.
        `active_gids` = هویت‌هایی که همین فریم به شخص دیگری تخصیص یافته‌اند
        (تا دو نفرِ همزمان هرگز یک هویت نگیرند).
        """
        active_gids = active_gids or set()

        # ---------- حالت ۱: این track_id را از قبل می‌شناسیم -------------
        gid = self._tid2gid.get(tid)
        if gid is not None and gid in self.identities:
            ident = self.identities[gid]

            # پایشِ جابه‌جایی ID: آیا ظاهر ناگهان عوض شده؟
            if emb is not None and ident.gallery:
                sim = ident.similarity(emb)
                if sim < self.SWITCH_SUSPECT_SIM:
                    self._switch_hits[tid] = self._switch_hits.get(tid, 0) + 1
                    if self._switch_hits[tid] >= self.SWITCH_CONFIRM_HITS:
                        # ردیاب اشتباه کرده — این track را از هویت جدا می‌کنیم
                        ident.track_ids.discard(tid)
                        self._tid2gid.pop(tid, None)
                        self._switch_hits.pop(tid, None)
                        return self._match_or_create(tid, bbox, t, emb, active_gids)
                else:
                    self._switch_hits.pop(tid, None)
                    ident.add_embedding(emb, self.cfg.reid.gallery_per_identity)

            ident.last_seen_t = t
            ident.last_bbox = bbox
            ident.frames_seen += 1
            return gid

        # ---------- حالت ۲: track_id جدید ------------------------------
        return self._match_or_create(tid, bbox, t, emb, active_gids)

    # ------------------------------------------------------------------ #
    def _match_or_create(self, tid: int, bbox, t: float,
                         emb: Optional[np.ndarray], active_gids: Set[int]) -> int:
        """تلاش برای وصل‌کردن به هویتی که اخیراً گم شده است."""
        if emb is None:
            return self._new_identity(tid, bbox, t, emb)

        ttl = self.cfg.reid.lost_ttl_seconds
        best_gid, best_sim = None, 0.0
        for gid, ident in self.identities.items():
            if gid in active_gids:
                continue                                   # همین حالا جای دیگری فعال است
            if (t - ident.last_seen_t) > ttl:
                continue                                   # خیلی وقت است رفته
            sim = ident.similarity(emb)
            if sim > best_sim:
                best_gid, best_sim = gid, sim

        if best_gid is not None and best_sim >= self.cfg.reid.match_threshold:
            ident = self.identities[best_gid]
            ident.track_ids.add(tid)
            ident.last_seen_t = t
            ident.last_bbox = bbox
            ident.frames_seen += 1
            ident.reconnects += 1
            ident.add_embedding(emb, self.cfg.reid.gallery_per_identity)
            self._tid2gid[tid] = best_gid
            return best_gid

        return self._new_identity(tid, bbox, t, emb)

    # ------------------------------------------------------------------ #
    def collect_garbage(self, t: float) -> List[int]:
        """
        هویت‌هایی که مدت زیادی دیده نشده‌اند حذف می‌شوند و شناسه‌شان
        در `dead_gids` گزارش می‌شود تا لایه‌های دیگر (fusion/snapshot)
        هم حافظه‌شان را آزاد کنند. بدون این، اجرای ۲۴ ساعته حافظه را می‌خورد.
        """
        ttl = self.cfg.reid.lost_ttl_seconds * 2.0
        dead = [gid for gid, i in self.identities.items() if (t - i.last_seen_t) > ttl]
        for gid in dead:
            ident = self.identities.pop(gid)
            for tid in ident.track_ids:
                self._tid2gid.pop(tid, None)
        self.dead_gids = dead
        return dead

    # ------------------------------------------------------------------ #
    def knows(self, tid: int) -> bool:
        """آیا این track_id قبلاً به یک هویت نگاشت شده است؟"""
        gid = self._tid2gid.get(tid)
        return gid is not None and gid in self.identities

    # ------------------------------------------------------------------ #
    def stats(self) -> Dict[str, int]:
        return {
            "identities_alive": len(self.identities),
            "identities_total": self._next_gid - 1,
            "reconnects": sum(i.reconnects for i in self.identities.values()),
        }

### ★ انباشت شواهد (log-odds)
<sub>`security_core/fusion/evidence.py`</sub>

In [ ]:
# ==========================================================================
#  ★ انباشت شواهد (log-odds)
#  (منبع: security_core/fusion/evidence.py)
# ==========================================================================
from __future__ import annotations

"""
evidence.py — ★ انباشتگر شواهد به‌روش لگاریتم-بخت (log-odds).

جایگزینِ TrackStateManager نسخهٔ قبلی
------------------------------------
منطق قبلی: «۳ رأی جمع کن، اکثریت را بردار، اگر سبز شد برای همیشه قفل کن».
چهار ایراد داشت:

  ۱) همهٔ رأی‌ها هم‌وزن بودند. رأیِ یک کروپِ ۲۰ پیکسلیِ تار دقیقاً
     همان وزنِ یک کروپِ ۲۰۰ پیکسلیِ واضح را داشت.
  ۲) قفلِ سبز دائمی بود. دزدی که با صورت باز وارد شود و بعد بالاکلاوا
     بکشد، تا آخر ویدیو «سبز» می‌ماند و دیگر بررسی نمی‌شود.
  ۳) شواهد قدیمی هرگز محو نمی‌شدند؛ یک تصمیم غلط تا ابد می‌ماند.
  ۴) هیچ راهی برای اضافه‌کردن شواهدِ ماژول‌های دیگر (اسلحه/آتش)
     وجود نداشت، چون منطق روی «رأی طبقه‌بند ماسک» hard-code شده بود.

روش جدید
--------
برای هر هویت یک بردار امتیاز روی کلاس‌ها نگه می‌داریم:

    S ← S · decay^Δt  +  w · log(p)          سپس   posterior = softmax(S)

    w = کیفیتِ مشاهده (اندازه × وضوح × نور × اعتبار جعبهٔ سر)

خواص این فرمول:
  • شاهد بی‌کیفیت خودبه‌خود اثر کمی می‌گذارد — نیازی به دور انداختنش نیست.
  • با گذشت زمان همه‌چیز به «نمی‌دانم» میل می‌کند، پس قفل دائمی معنا ندارد.
  • هرچه شواهد هم‌جهت بیشتر شود، اطمینان بالاتر می‌رود (رفتار بیزی).
  • ماژول‌های بعدی فقط یک خط اضافه می‌کنند:
        fusion.add_module_evidence(gid, "threat", 0.8, t, source="weapon")
    بدون هیچ تغییری در منطق تصمیم.
"""

import math
from dataclasses import dataclass, field
from typing import Dict, Iterable, List, Optional

import numpy as np


_CLASSES: List[FaceClass] = FaceClass.all()
_IDX = {c: i for i, c in enumerate(_CLASSES)}

# کفِ لگاریتم — جلوی «یک مشاهده، حکم قطعی» را می‌گیرد.
# بدون این، log(p≈0) عدد بسیار منفی می‌دهد و یک خطای مدل غیرقابل جبران می‌شود.
_LOG_FLOOR = -5.0


@dataclass
class Belief:
    gid: int
    scores: np.ndarray = field(default_factory=lambda: np.zeros(len(_CLASSES), np.float64))
    last_t: float = 0.0
    n_obs: int = 0
    total_weight: float = 0.0
    # شواهدِ ماژول‌های دیگر؛ کلید = نام محور تهدید، مقدار = امتیاز انباشته
    module_evidence: Dict[str, float] = field(default_factory=dict)
    module_sources: Dict[str, str] = field(default_factory=dict)


class EvidenceFusion:
    def __init__(self, cfg):
        self.cfg = cfg
        self.beliefs: Dict[int, Belief] = {}

    # ------------------------------------------------------------------ #
    def _decay(self, b: Belief, t: float):
        dt = max(0.0, t - b.last_t)
        if dt > 0 and b.last_t > 0:
            f = self.cfg.fusion.decay_per_second ** dt
            b.scores *= f
            for k in list(b.module_evidence):
                b.module_evidence[k] *= f
                if abs(b.module_evidence[k]) < 1e-3:
                    b.module_evidence.pop(k, None)
                    b.module_sources.pop(k, None)
        b.last_t = t

    def _get(self, gid: int, t: float) -> Belief:
        b = self.beliefs.get(gid)
        if b is None:
            b = Belief(gid=gid, last_t=t)
            self.beliefs[gid] = b
        else:
            self._decay(b, t)
        return b

    # ------------------------------------------------------------------ #
    def observe(self, gid: int, probs: Dict[FaceClass, float],
                weight: float, t: float):
        """ثبت یک مشاهدهٔ طبقه‌بند با وزنِ کیفیت."""
        if weight <= 0.0:
            return
        b = self._get(gid, t)
        for c, i in _IDX.items():
            p = float(probs.get(c, 0.0))
            b.scores[i] += weight * max(_LOG_FLOOR, math.log(max(p, 1e-9)))
        # نرمال‌سازی عددی (softmax نسبت به شیفت ثابت بی‌تفاوت است)
        b.scores -= b.scores.max()
        b.n_obs += 1
        b.total_weight += weight

    # ------------------------------------------------------------------ #
    def add_module_evidence(self, gid: int, axis: str, delta: float,
                            t: float, source: str = ""):
        """
        ★ نقطهٔ اتصال ماژول‌های آینده.

        مثال‌های واقعی:
            add_module_evidence(gid, "threat", +0.9, t, "weapon.pistol")
            add_module_evidence(gid, "threat", +0.4, t, "behavior.hand_in_jacket")
            add_module_evidence(gid, "threat", -0.3, t, "staff.whitelist")

        لایهٔ policy این محور را با احتمال پوشش صورت ترکیب می‌کند.
        """
        b = self._get(gid, t)
        b.module_evidence[axis] = b.module_evidence.get(axis, 0.0) + float(delta)
        if source:
            b.module_sources[axis] = source

    # ------------------------------------------------------------------ #
    def posterior(self, gid: int, t: Optional[float] = None) -> Dict[FaceClass, float]:
        b = self.beliefs.get(gid)
        if b is None:
            return {c: (1.0 if c is FaceClass.UNKNOWN else 0.0) for c in _CLASSES}
        if t is not None:
            self._decay(b, t)
        s = b.scores / max(1e-6, self.cfg.fusion.temperature)
        e = np.exp(s - s.max())
        p = e / e.sum()
        return {c: float(p[_IDX[c]]) for c in _CLASSES}

    def n_obs(self, gid: int) -> int:
        b = self.beliefs.get(gid)
        return b.n_obs if b else 0

    def module_axis(self, gid: int, axis: str) -> float:
        b = self.beliefs.get(gid)
        return b.module_evidence.get(axis, 0.0) if b else 0.0

    def evidence_sources(self, gid: int) -> Dict[str, str]:
        b = self.beliefs.get(gid)
        return dict(b.module_sources) if b else {}

    # ------------------------------------------------------------------ #
    def drop(self, gids: Iterable[int]):
        """آزادسازی حافظهٔ هویت‌های مرده (به فراخوان IdentityBank.collect_garbage)."""
        for g in gids:
            self.beliefs.pop(g, None)

### باور → وضعیت و رنگ
<sub>`security_core/fusion/policy.py`</sub>

In [ ]:
# ==========================================================================
#  باور → وضعیت و رنگ
#  (منبع: security_core/fusion/policy.py)
# ==========================================================================
from __future__ import annotations

"""
policy.py — ترجمهٔ «باور» به «تصمیم» (رنگ / هشدار).

تنها جایی از کل سیستم که رنگ و آلارم تعیین می‌شود. اگر مشتری
جدول رنگ متفاوتی خواست، فقط همین فایل و yaml عوض می‌شود.

دو مفهوم که در نسخهٔ قبلی نبود
------------------------------
۱) هیسترزیس: آستانهٔ ورود به هر وضعیت با آستانهٔ خروج از آن فرق دارد.
   بدون این، وقتی احتمال حول یک آستانه نوسان کند، کادر بین قرمز و
   خاکستری چشمک می‌زند و برای اپراتور آزاردهنده (و برای ارائه به
   کارفرما بد) است.

۲) وضعیت میانی «تحت نظر» (زرد): مشکوک است ولی هنوز به حد آلارم
   نرسیده. ارزش تجاری‌اش زیاد است — به مشتری نشان می‌دهد که سیستم
   محتاط است و بی‌گدار آژیر نمی‌کشد.

جدول رنگ نهایی
--------------
   ANALYZING   آبی روشن   در حال جمع‌آوری شواهد
   CLEAR       سبز        صورت واضح
   COVERED     خاکستری    ماسک پزشکی / پشت به دوربین / نامشخص  ← بی‌خطر
   WATCH       زرد        نشانهٔ پوشش کامل، زیر آستانهٔ آلارم
   SUSPECT     قرمز       ماسک دزدی / پوشش کامل صورت  ← آلارم
"""

from typing import Dict, Optional, Tuple


# رنگ‌ها به‌صورت BGR (استاندارد OpenCV)
STATE_COLORS: Dict[ThreatState, Tuple[int, int, int]] = {
    ThreatState.ANALYZING: (235, 200, 140),   # آبی روشن
    ThreatState.CLEAR:     (60, 200, 60),     # سبز
    ThreatState.COVERED:   (150, 150, 150),   # خاکستری
    ThreatState.WATCH:     (0, 215, 255),     # زرد/کهربایی
    ThreatState.SUSPECT:   (0, 0, 255),       # قرمز
    ThreatState.AWAY:      (120, 70, 40),     # آبی تیره — بی‌قضاوت
}

STATE_LABELS: Dict[ThreatState, str] = {
    ThreatState.ANALYZING: "Analyzing",
    ThreatState.CLEAR:     "Clear",
    ThreatState.COVERED:   "Covered",
    ThreatState.WATCH:     "Watch",
    ThreatState.SUSPECT:   "ALERT - Face Concealed",
    ThreatState.AWAY:      "Back to camera",
}

# اولویت برای زمان‌بندی محاسبات: هرچه بالاتر، مهم‌تر
STATE_PRIORITY: Dict[ThreatState, int] = {
    ThreatState.SUSPECT: 4,
    ThreatState.WATCH: 3,
    ThreatState.ANALYZING: 2,
    ThreatState.COVERED: 1,
    ThreatState.CLEAR: 0,
    ThreatState.AWAY: 0,      # هیچ محاسبه‌ای صرفش نمی‌شود
}


class ThreatPolicy:
    def __init__(self, cfg):
        self.cfg = cfg
        self._state: Dict[int, ThreatState] = {}
        self._since: Dict[int, float] = {}

    # ------------------------------------------------------------------ #
    def evaluate(self, gid: int, posterior: Dict[FaceClass, float],
                 n_obs: int, t: float,
                 threat_bonus: float = 0.0,
                 orientation=None) -> Tuple[ThreatState, float]:
        """
        `threat_bonus` امتیازی است که ماژول‌های دیگر (اسلحه/رفتار) اضافه
        می‌کنند. اثرش این است که آستانهٔ آلارمِ چهره را پایین می‌آورد:
        اگر کسی اسلحه دست دارد، برای اعلام خطر لازم نیست تا این حد
        مطمئن باشیم که صورتش هم پوشیده است. این همان «هم‌افزایی
        ماژول‌ها» است که هدف اصلی معماری بود.
        """
        f = self.cfg.fusion
        prev = self._state.get(gid, ThreatState.ANALYZING)

        # ── ★ گامِ صفر: آیا اصلاً می‌شود قضاوت کرد؟ ─────────────────────
        # اگر فرد پشتش به دوربین است، هیچ حکمی دربارهٔ پوششِ صورتش
        # صادر نمی‌کنیم. این همان کاری است که نسخهٔ اولِ پروژه (با گیتِ
        # کی‌پوینت) به‌درستی انجام می‌داد و من اشتباهاً حذفش کردم.
        # تفاوت مهم با نسخهٔ اول: اینجا «پشت بودن» را از هندسهٔ بدن
        # می‌فهمیم، نه از دیده‌نشدنِ چشم. پس فردِ نقاب‌دارِ رو به دوربین
        # همچنان تحلیل می‌شود — هر دو خاصیت را با هم داریم.
        if orientation is not None and orientation.is_back:
            return self._set(gid, ThreatState.AWAY, t), orientation.confidence

        if n_obs < f.min_observations:
            return self._set(gid, ThreatState.ANALYZING, t), 0.0

        # ── ★ بازنرمال‌سازی روی کلاس‌های «معنادار» ─────────────────────
        # سه کلاس زیر دربارهٔ پوششِ صورت حرف می‌زنند. دو کلاس دیگر
        # (BACK_HEAD و UNKNOWN) یعنی «این مشاهده اطلاعاتی ندارد».
        # تصمیم باید فقط بر پایهٔ بخشِ اطلاعاتی گرفته شود، وگرنه یک
        # کروپِ تارِ دوربین مداربسته که مدل آن را UNKNOWN می‌بیند،
        # به‌اشتباه مثل «شواهدی علیه پوشیده‌بودن» عمل می‌کند.
        p_c = posterior.get(FaceClass.CLEAR, 0.0)
        p_m = posterior.get(FaceClass.MEDICAL_MASK, 0.0)
        p_f = posterior.get(FaceClass.FULL_COVER, 0.0)
        informative = p_c + p_m + p_f

        if informative < f.min_informative_mass:
            # ★ اینجا قبلاً COVERED برمی‌گرداندیم و همین باعث می‌شد
            #   «همه‌چیز خاکستری» شود. خاکستری یعنی «ماسک پزشکی دیدم» —
            #   یک ادعای مثبت. ولی این حالت یعنی «چیزی ندیدم».
            #   ادعای نداشته نباید به کاربر نشان داده شود.
            return self._set(gid, ThreatState.ANALYZING, t), 1.0 - informative

        p_full = p_f / informative
        p_clear = p_c / informative

        # امتیاز تهدید = پوشش صورت + سهم ماژول‌های دیگر (اشباع‌شده)
        bonus = max(0.0, min(0.30, threat_bonus * 0.15))
        score = min(1.0, p_full + bonus)

        # --- آستانه‌های هیسترزیس ---------------------------------------
        if prev is ThreatState.SUSPECT:
            if score >= f.suspect_exit:
                return self._set(gid, ThreatState.SUSPECT, t), score
        elif score >= f.suspect_enter:
            return self._set(gid, ThreatState.SUSPECT, t), score

        if score >= f.watch_enter:
            return self._set(gid, ThreatState.WATCH, t), score

        if prev is ThreatState.CLEAR:
            if p_clear >= f.clear_exit:
                return self._set(gid, ThreatState.CLEAR, t), p_clear
        elif p_clear >= f.clear_enter:
            return self._set(gid, ThreatState.CLEAR, t), p_clear

        # ★ COVERED فقط وقتی اعلام می‌شود که واقعاً شاهدِ ماسکِ پزشکی
        #   داشته باشیم — نه به‌عنوانِ سطلِ ته‌ماندهٔ همهٔ حالت‌های مبهم.
        #   قبلاً هر ابهامی به اینجا می‌ریخت و خروجی پر از خاکستری می‌شد.
        p_med = p_m / informative
        if p_med >= f.covered_enter:
            return self._set(gid, ThreatState.COVERED, t), p_med

        # هنوز هیچ‌کدام از آستانه‌ها پر نشده — صادقانه بگو «در حال تحلیل»
        return self._set(gid, ThreatState.ANALYZING, t), max(p_clear, p_full)

    # ------------------------------------------------------------------ #
    def _set(self, gid: int, s: ThreatState, t: float) -> ThreatState:
        if self._state.get(gid) is not s:
            self._since[gid] = t
        self._state[gid] = s
        return s

    def state(self, gid: int) -> ThreatState:
        return self._state.get(gid, ThreatState.ANALYZING)

    def state_age(self, gid: int, t: float) -> float:
        return t - self._since.get(gid, t)

    def drop(self, gids):
        for g in gids:
            self._state.pop(g, None)
            self._since.pop(g, None)


def color_of(state: ThreatState) -> Tuple[int, int, int]:
    return STATE_COLORS[state]


def label_of(state: ThreatState) -> str:
    return STATE_LABELS[state]

### ★ موتور بهترین شات و رویدادها
<sub>`security_core/export/snapshot.py`</sub>

In [ ]:
# ==========================================================================
#  ★ موتور بهترین شات و رویدادها
#  (منبع: security_core/export/snapshot.py)
# ==========================================================================
from __future__ import annotations

"""
snapshot.py — ★ موتور «بهترین شات» و صادرکنندهٔ رویداد.

این همان بخشی است که در نسخهٔ قبلی خیلی دوستش داشتی (برش سوژهٔ
مشکوک و نمایشش در گوشهٔ تصویر). اینجا سه ارتقای اساسی گرفته است:

۱) کروپ از فریمِ **تمیز** گرفته می‌شود
   در نسخهٔ قبلی، کروپ از فریمی برداشته می‌شد که مستطیل رنگی، خطوط
   اسکلت و متن روی آن کشیده شده بود. آن تصاویر هم برای ارائه زشت
   بودند و هم به‌عنوان دادهٔ آموزشی بی‌ارزش.

۲) «بهترین» شات، نه «اولین» شات
   نسخهٔ قبلی همان لحظه‌ای که فرد قرمز می‌شد کروپ می‌گرفت — که
   معمولاً بدترین کیفیت ممکن است (فرد تازه وارد شده، دور و تار).
   اینجا برای هر هویت یک اسلات نگه می‌داریم و فقط وقتی تصویر
   بهتری آمد جایگزین می‌کنیم.

۳) خروجی ساختاریافته به‌جای یک تصویر
   هر رویداد یک پوشه با متادیتای JSON می‌شود. همین ساختار هم
   دموی کارفرما را می‌سازد و هم مستقیماً ورودی چرخهٔ برچسب‌زنی
   خودکار و آموزش مدل دانش‌آموز است (فاز ۳).

        runs/events/2026-08-25/cam0_gid0007/
            best_face.jpg      ← تمیز، بدون overlay
            best_body.jpg      ← تمیز
            context.jpg        ← فریم کامل با overlay (فقط برای ارائه)
            meta.json
"""

import json
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np



@dataclass
class _Slot:
    gid: int
    best_face: Optional[np.ndarray] = None
    best_face_q: float = -1.0
    best_body: Optional[np.ndarray] = None
    best_body_q: float = -1.0
    peak_state: ThreatState = ThreatState.ANALYZING
    peak_conf: float = 0.0
    first_t: float = 0.0
    last_t: float = 0.0
    first_frame: int = 0
    last_frame: int = 0
    bbox: Tuple[int, int, int, int] = (0, 0, 0, 0)
    context_written: bool = False
    dir: Optional[Path] = None
    harvest_count: int = 0
    harvest_last_t: float = 0.0
    dirty: bool = False


# شدت وضعیت‌ها — برای اینکه «بیشترین شدتِ دیده‌شده» را نگه داریم
_SEVERITY = {
    ThreatState.ANALYZING: 0, ThreatState.CLEAR: 1, ThreatState.COVERED: 2,
    ThreatState.WATCH: 3, ThreatState.SUSPECT: 4,
}


class SnapshotManager:
    # حداقل بهبود کیفیت برای بازنویسی فایل روی دیسک (جلوگیری از I/O بیهوده)
    IMPROVE_DELTA = 0.05

    def __init__(self, cfg, camera_id: str = "cam0"):
        self.cfg = cfg
        self.camera_id = camera_id
        self.slots: Dict[int, _Slot] = {}
        self.events: List[SecurityEvent] = []

        self._save_states = {ThreatState(s) for s in cfg.snapshot.save_states}
        self._jpg = [int(cv2.IMWRITE_JPEG_QUALITY), int(cfg.snapshot.jpeg_quality)]

        day = time.strftime("%Y-%m-%d")
        self.root = Path(cfg.snapshot.out_dir) / day
        self.harvest_root = Path(cfg.snapshot.harvest_dir) / day
        if cfg.snapshot.enabled:
            self.root.mkdir(parents=True, exist_ok=True)
        if cfg.snapshot.harvest_all_for_dataset:
            self.harvest_root.mkdir(parents=True, exist_ok=True)

    # ------------------------------------------------------------------ #
    def offer(self, gid: int, state: ThreatState, conf: float,
              face_clean: Optional[np.ndarray], body_clean: Optional[np.ndarray],
              quality: float, t: float, frame_idx: int,
              bbox: Tuple[int, int, int, int],
              full_clean: Optional[np.ndarray] = None,
              head_box: Optional[Tuple[int, int, int, int]] = None):
        """
        یک مشاهده را «پیشنهاد» می‌دهد. مدیر خودش تصمیم می‌گیرد که آیا
        از نمونهٔ فعلی بهتر است یا نه. صدا زدنش در هر فریم ارزان است.
        """
        if not self.cfg.snapshot.enabled:
            return

        s = self.slots.get(gid)
        if s is None:
            s = _Slot(gid=gid, first_t=t, first_frame=frame_idx)
            self.slots[gid] = s

        s.last_t, s.last_frame, s.bbox = t, frame_idx, bbox

        if _SEVERITY[state] > _SEVERITY[s.peak_state]:
            s.peak_state, s.peak_conf, s.dirty = state, conf, True
        elif state is s.peak_state and conf > s.peak_conf:
            s.peak_conf, s.dirty = conf, True

        if quality >= self.cfg.snapshot.min_quality:
            if face_clean is not None and face_clean.size and quality > s.best_face_q:
                s.best_face, s.best_face_q, s.dirty = face_clean.copy(), quality, True
            if (self.cfg.snapshot.save_body and body_clean is not None
                    and body_clean.size and quality > s.best_body_q):
                s.best_body, s.best_body_q, s.dirty = body_clean.copy(), quality, True

        # ---- حالت برداشت دیتاست (فاز ۳) --------------------------------
        if (self.cfg.snapshot.harvest_all_for_dataset and face_clean is not None
                and face_clean.size and quality >= self.cfg.snapshot.min_quality
                and s.harvest_count < 12 and (t - s.harvest_last_t) > 0.4):
            name = f"{self.camera_id}_g{gid:05d}_{s.harvest_count:02d}.jpg"
            cv2.imwrite(str(self.harvest_root / name), face_clean, self._jpg)

            # فریم کامل + جعبهٔ سر — سوخت آموزش سرِ RoI در فاز ۴
            if (self.cfg.snapshot.harvest_frames and full_clean is not None
                    and head_box is not None):
                fdir = self.harvest_root / "frames"
                fdir.mkdir(parents=True, exist_ok=True)
                cv2.imwrite(str(fdir / name), full_clean, self._jpg)
                rec = {"name": name, "frame": f"frames/{name}",
                       "head": list(head_box)}
                with (self.harvest_root / "boxes.jsonl").open("a", encoding="utf-8") as f:
                    f.write(json.dumps(rec) + "\n")

            s.harvest_count += 1
            s.harvest_last_t = t

    # ------------------------------------------------------------------ #
    def note_context(self, gid: int, annotated_frame: np.ndarray):
        """
        فریم کامل با overlay — فقط یک بار و فقط برای رویدادهای مهم.
        هدفش ارائه به اپراتور/کارفرماست، نه دیتاست.
        """
        s = self.slots.get(gid)
        if (not self.cfg.snapshot.enabled or not self.cfg.snapshot.save_context
                or s is None or s.context_written):
            return
        if s.peak_state not in self._save_states:
            return
        d = self._dir(s)
        cv2.imwrite(str(d / "context.jpg"), annotated_frame, self._jpg)
        s.context_written = True

    # ------------------------------------------------------------------ #
    def _dir(self, s: _Slot) -> Path:
        if s.dir is None:
            s.dir = self.root / f"{self.camera_id}_gid{s.gid:04d}"
            s.dir.mkdir(parents=True, exist_ok=True)
        return s.dir

    def flush(self, gid: int):
        """نوشتن/به‌روزرسانی فایل‌های یک هویت روی دیسک."""
        s = self.slots.get(gid)
        if s is None or not s.dirty or not self.cfg.snapshot.enabled:
            return
        if s.peak_state not in self._save_states:
            return

        d = self._dir(s)
        if s.best_face is not None:
            cv2.imwrite(str(d / "best_face.jpg"), s.best_face, self._jpg)
        if s.best_body is not None:
            cv2.imwrite(str(d / "best_body.jpg"), s.best_body, self._jpg)

        ev = SecurityEvent(
            global_id=s.gid, kind=f"face.{s.peak_state.value}",
            state=s.peak_state.value, confidence=round(s.peak_conf, 4),
            frame_idx=s.last_frame, timestamp=s.last_t, camera_id=self.camera_id,
            bbox=s.bbox,
            extra={
                "first_seen_frame": s.first_frame,
                "duration_s": round(s.last_t - s.first_t, 2),
                "best_face_quality": round(s.best_face_q, 3),
                "dir": str(d),
            },
        )
        (d / "meta.json").write_text(
            json.dumps(ev.to_dict(), ensure_ascii=False, indent=2), encoding="utf-8")
        s.dirty = False
        self.events.append(ev)

    # ------------------------------------------------------------------ #
    def finalize(self, gids) -> List[SecurityEvent]:
        """بستن پروندهٔ هویت‌هایی که از صحنه خارج شده‌اند و آزادسازی حافظه."""
        out = []
        for g in list(gids):
            self.flush(g)
            s = self.slots.pop(g, None)
            if s is not None and self.events and self.events[-1].global_id == g:
                out.append(self.events[-1])
        return out

    def finalize_all(self) -> List[SecurityEvent]:
        for g in list(self.slots.keys()):
            self.flush(g)
        self.slots.clear()
        return self.events

    # ------------------------------------------------------------------ #
    def gallery_items(self, limit: int = 4) -> List[Tuple[np.ndarray, str, ThreatState]]:
        """
        داده‌های لازم برای گالری گوشهٔ تصویر — مهم‌ترین سوژه‌ها اول.
        (بخش «نمایشیِ» محبوبِ نسخهٔ قبلی، حالا با بهترین کیفیت موجود.)
        """
        ranked = sorted(
            (s for s in self.slots.values() if s.best_face is not None),
            key=lambda s: (-_SEVERITY[s.peak_state], -s.last_t))
        out = []
        for s in ranked[:limit]:
            out.append((s.best_face, f"ID {s.gid}", s.peak_state))
        return out

### رسم کادر، اسکلت و نوار وضعیت
<sub>`security_core/viz/overlay.py`</sub>

In [ ]:
# ==========================================================================
#  رسم کادر، اسکلت و نوار وضعیت
#  (منبع: security_core/viz/overlay.py)
# ==========================================================================
from __future__ import annotations

"""
overlay.py — رسم روی تصویر.

⚠️ قانون سختِ معماری: هر چیزی که در این پوشه است، فقط و فقط روی
نسخهٔ *نمایشی* فریم می‌نویسد. فریم تمیز جداگانه نگه داشته می‌شود و
همهٔ کروپ‌ها از آن گرفته می‌شوند. این تفکیک، ریشهٔ یکی از باگ‌های
نسخهٔ قبلی را می‌خشکاند (کروپ‌های آلوده به خط و متن).

سبک ظاهری عمداً «کادر گوشه‌دار» انتخاب شده نه مستطیل ساده: هم
صورت سوژه را کمتر می‌پوشاند و هم در ارائه حرفه‌ای‌تر دیده می‌شود.
"""

from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np


K = Keypoints

# اسکلت بالاتنه — همان انتخاب درستِ نسخهٔ قبلی.
# پایین‌تنه رسم نمی‌شود چون هم شلوغی بصری می‌آورد و هم برای تشخیص
# سرقت بی‌فایده است؛ آنچه اهمیت دارد سر، شانه، آرنج و **مچ** است.
UPPER_SKELETON = [
    (K.LEYE, K.REYE), (K.NOSE, K.LEYE), (K.NOSE, K.REYE),
    (K.LEAR, K.LEYE), (K.REAR, K.REYE),
    (K.LSHO, K.RSHO),
    (K.LSHO, K.LELB), (K.LELB, K.LWRI),
    (K.RSHO, K.RELB), (K.RELB, K.RWRI),
    (K.NOSE, K.LSHO), (K.NOSE, K.RSHO),
]
WRISTS = (K.LWRI, K.RWRI)


def draw_corner_box(img, box, color, thickness=2, corner_ratio=0.22):
    """کادر گوشه‌دار — چهار گوشهٔ L شکل به‌جای مستطیل کامل."""
    x1, y1, x2, y2 = [int(v) for v in box]
    w, h = x2 - x1, y2 - y1
    c = max(6, int(min(w, h) * corner_ratio))
    for (px, py, dx, dy) in ((x1, y1, 1, 1), (x2, y1, -1, 1),
                             (x1, y2, 1, -1), (x2, y2, -1, -1)):
        cv2.line(img, (px, py), (px + dx * c, py), color, thickness, cv2.LINE_AA)
        cv2.line(img, (px, py), (px, py + dy * c), color, thickness, cv2.LINE_AA)


def draw_label(img, text, org, color, scale=0.55, thickness=1, pad=4):
    """برچسب با پس‌زمینهٔ توپر — روی هر تصویری خوانا می‌ماند."""
    (tw, th), base = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, thickness + 1)
    x, y = int(org[0]), int(org[1])
    y = max(y, th + pad * 2)
    cv2.rectangle(img, (x, y - th - pad * 2), (x + tw + pad * 2, y), color, -1)
    cv2.putText(img, text, (x + pad, y - pad), cv2.FONT_HERSHEY_SIMPLEX,
                scale, (20, 20, 20), thickness, cv2.LINE_AA)


def draw_skeleton(img, kp: Keypoints, color=(210, 210, 210), conf_th=0.35, thickness=2):
    """اسکلت بالاتنه. مچ‌ها با دایرهٔ بزرگ‌تر مشخص می‌شوند."""
    for a, b in UPPER_SKELETON:
        if kp.c(a) < conf_th or kp.c(b) < conf_th:
            continue
        cv2.line(img, tuple(map(int, kp.pt(a))), tuple(map(int, kp.pt(b))),
                 color, thickness, cv2.LINE_AA)
    for idx in (K.NOSE, K.LEYE, K.REYE, K.LEAR, K.REAR,
                K.LSHO, K.RSHO, K.LELB, K.RELB, K.LWRI, K.RWRI):
        if kp.c(idx) >= conf_th:
            r = 5 if idx in WRISTS else 3     # مچ‌ها برای ماژول اسلحه کلیدی‌اند
            cv2.circle(img, tuple(map(int, kp.pt(idx))), r, color, -1, cv2.LINE_AA)


def draw_person(img, bbox, gid: int, state: ThreatState, conf: float,
                thickness: int = 2, font_scale: float = 0.55,
                head_box: Optional[Tuple[int, int, int, int]] = None):
    color = color_of(state)
    x1, y1, x2, y2 = [int(v) for v in bbox]
    draw_corner_box(img, bbox, color, thickness)
    draw_label(img, f"#{gid}  {label_of(state)}  {conf:.0%}",
               (x1, y1 - 4), color, font_scale, 1)

    if head_box is not None:
        hx1, hy1, hx2, hy2 = head_box
        cv2.rectangle(img, (hx1, hy1), (hx2, hy2), color, 1, cv2.LINE_AA)

    if state is ThreatState.SUSPECT:
        # هالهٔ نیمه‌شفاف قرمز — در ارائه بسیار مؤثر است
        ov = img.copy()
        cv2.rectangle(ov, (x1, y1), (x2, y2), color, -1)
        cv2.addWeighted(ov, 0.14, img, 0.86, 0, img)
        draw_label(img, "ALERT", (x1, y2 + 26), color, font_scale + 0.1, 2)


def draw_hud(img, fps: float, n_people: int, n_alerts: int,
             backend: str = "", extra: str = ""):
    """نوار وضعیت بالای تصویر — برای دمو و برای دیباگ هر دو مفید است."""
    h, w = img.shape[:2]
    bar = 34
    ov = img.copy()
    cv2.rectangle(ov, (0, 0), (w, bar), (18, 18, 18), -1)
    cv2.addWeighted(ov, 0.62, img, 0.38, 0, img)

    txt = f"FPS {fps:5.1f}   People {n_people:2d}   Alerts {n_alerts:2d}"
    if backend:
        txt += f"   [{backend}]"
    if extra:
        txt += f"   {extra}"
    cv2.putText(img, txt, (12, 23), cv2.FONT_HERSHEY_SIMPLEX, 0.56,
                (240, 240, 240), 1, cv2.LINE_AA)

    if n_alerts > 0:
        cv2.rectangle(img, (0, 0), (w - 1, h - 1), (0, 0, 255), 3)
        draw_label(img, "SECURITY ALERT", (w - 230, bar + 30), (0, 0, 255), 0.6, 2)

### گالری برش‌ها در گوشهٔ تصویر
<sub>`security_core/viz/gallery.py`</sub>

In [ ]:
# ==========================================================================
#  گالری برش‌ها در گوشهٔ تصویر
#  (منبع: security_core/viz/gallery.py)
# ==========================================================================
from __future__ import annotations

"""
gallery.py — گالری تصاویر کوچک در گوشهٔ تصویر (Picture-in-Picture).

این همان قابلیتی است که در ارائه به کارفرما بیشترین اثر را دارد:
سیستم نه‌فقط هشدار می‌دهد، بلکه **چهرهٔ سوژه را بیرون می‌کشد و
نشان می‌دهد**.

تفاوت با نسخهٔ قبلی
------------------
  • تصویرها از فریم تمیز می‌آیند، نه از فریمِ نقاشی‌شده.
  • «بهترین» تصویر هر فرد نمایش داده می‌شود، نه اولین تصویر.
  • ترتیب بر اساس شدت تهدید است (قرمزها بالا).
  • نسبت ابعاد حفظ می‌شود (نسخهٔ قبلی به مربع کشیده می‌شد و صورت‌ها
    بدشکل می‌شدند).
  • بررسی مرزها اضافه شده — نسخهٔ قبلی اگر عرض فریم از اندازهٔ
    بندانگشتی کمتر بود کرش می‌کرد.
"""

from typing import List, Tuple

import cv2
import numpy as np



class Gallery:
    def __init__(self, max_items: int = 4, thumb: int = 150,
                 pad: int = 10, label_h: int = 22):
        self.max_items = max_items
        self.thumb = thumb
        self.pad = pad
        self.label_h = label_h

    def draw(self, frame: np.ndarray,
             items: List[Tuple[np.ndarray, str, ThreatState]]):
        if not items:
            return
        H, W = frame.shape[:2]
        t = min(self.thumb, max(48, W // 6), max(48, H // 5))   # ← محافظت از فریم کوچک
        if t < 40:
            return

        x2 = W - self.pad
        x1 = x2 - t
        if x1 < 0:
            return

        for i, (img, text, state) in enumerate(items[:self.max_items]):
            y1 = self.pad + 40 + i * (t + self.label_h + 8)      # ۴۰ = زیر نوار HUD
            y2 = y1 + t
            if y2 + self.label_h > H:
                break
            if img is None or img.size == 0:
                continue

            thumb = letterbox_square(img, t, pad_value=30)
            frame[y1:y2, x1:x2] = thumb

            col = color_of(state)
            cv2.rectangle(frame, (x1, y1), (x2, y2), col, 2, cv2.LINE_AA)

            # نوار برچسب زیر تصویر
            cv2.rectangle(frame, (x1, y2), (x2, y2 + self.label_h), col, -1)
            cv2.putText(frame, f"{text}", (x1 + 5, y2 + self.label_h - 6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.42, (20, 20, 20), 1, cv2.LINE_AA)

            if state is ThreatState.SUSPECT:
                cv2.putText(frame, "!", (x2 - 16, y1 + 20),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, col, 2, cv2.LINE_AA)

### اسلوموشن و فلاش هشدار (دمو)
<sub>`security_core/viz/effects.py`</sub>

In [ ]:
# ==========================================================================
#  اسلوموشن و فلاش هشدار (دمو)
#  (منبع: security_core/viz/effects.py)
# ==========================================================================
from __future__ import annotations

"""
effects.py — جلوه‌های نمایشی برای نسخهٔ دمو (اسلوموشن و فلاش هشدار).

⚠️ فقط در پروفایل `demo.yaml` فعال است. در پروفایل تولید کاملاً
خاموش می‌ماند، چون فریم تکراری در فایل خروجی برای سیستم واقعی
بی‌معنی است و فقط حجم می‌سازد.

بهبود نسبت به نسخهٔ قبلی
------------------------
  • در نسخهٔ قبلی، ورودِ **هر** فرد جدید (حتی یک تشخیص اشتباه در
    دوردست) اسلوموشن را فعال می‌کرد و ویدیو پرش‌دار می‌شد.
    اینجا فقط رویدادهای واقعاً مهم آن را فعال می‌کنند و یک
    «زمان خنک‌شدن» هم دارد.
  • به‌جای تکرار خشکِ N باره، شدت اسلوموشن نرم بالا و پایین می‌رود
    (ease-in/ease-out) که در ویدیو حرفه‌ای‌تر دیده می‌شود.
"""

from typing import Optional

import cv2
import numpy as np


class SlowMotionDirector:
    """
    تصمیم می‌گیرد هر فریم چند بار در فایل خروجی نوشته شود.

    یک «کلید» با `trigger()` زده می‌شود و اثرش روی چند فریم بعد
    پخش می‌شود: ابتدا شدید، بعد به‌تدریج به حالت عادی برمی‌گردد.
    """

    def __init__(self, enabled: bool, peak_repeat: int = 5,
                 ramp_frames: int = 6, cooldown_s: float = 2.0):
        self.enabled = enabled
        self.peak = max(1, peak_repeat)
        self.ramp = max(1, ramp_frames)
        self.cooldown = cooldown_s
        self._left = 0
        self._last_trigger_t = -1e9

    def trigger(self, t: float):
        if not self.enabled or (t - self._last_trigger_t) < self.cooldown:
            return
        self._last_trigger_t = t
        self._left = self.ramp

    def repeat_for_frame(self) -> int:
        if not self.enabled or self._left <= 0:
            return 1
        # منحنی نرم: از peak به ۱
        frac = self._left / self.ramp
        r = 1 + int(round((self.peak - 1) * (frac ** 1.6)))
        self._left -= 1
        return max(1, r)


class AlertFlash:
    """پالس قرمزِ محو‌شونده روی لبهٔ تصویر در لحظهٔ اعلام خطر."""

    def __init__(self, duration_frames: int = 10):
        self.duration = duration_frames
        self._left = 0

    def trigger(self):
        self._left = self.duration

    def apply(self, frame: np.ndarray):
        if self._left <= 0:
            return
        alpha = 0.35 * (self._left / self.duration)
        ov = frame.copy()
        cv2.rectangle(ov, (0, 0), (frame.shape[1] - 1, frame.shape[0] - 1),
                      (0, 0, 255), 26)
        cv2.addWeighted(ov, alpha, frame, 1 - alpha, 0, frame)
        self._left -= 1

### ★ ناحیهٔ حساس (ضد هشدار کاذب)
<sub>`security_core/runtime/zones.py`</sub>

In [ ]:
# ==========================================================================
#  ★ ناحیهٔ حساس (ضد هشدار کاذب)
#  (منبع: security_core/runtime/zones.py)
# ==========================================================================
from __future__ import annotations

"""
zones.py — ★ ناحیهٔ حساس (منطقهٔ پایش).

چرا در مرور نهایی اضافه شد
--------------------------
بزرگ‌ترین منبع هشدار کاذب در نصب واقعی، «مدل» نیست — **کادر** است.
دوربین مغازه معمولاً پیاده‌روِ بیرون، شیشهٔ ویترین یا کوچهٔ روبه‌رو را
هم می‌بیند. عابری که با کلاه و شال از جلوی مغازه رد می‌شود، برای
مدل کاملاً درست «پوشش کامل صورت» است — ولی هشدارش بی‌معنی است.

هیچ مقدار تیون‌کردن آستانه این را حل نمی‌کند، چون مسئله ادراکی نیست
بلکه جغرافیایی است. راه‌حل درست، تعریف چندضلعیِ ناحیهٔ حساس است.

نکتهٔ ظریف: معیارِ «داخل ناحیه بودن» **نقطهٔ پای** فرد است
(وسطِ لبهٔ پایینی جعبه)، نه مرکز جعبه. مرکز جعبه برای کسی که
نزدیک دوربین ایستاده ممکن است بالای خط ناحیه بیفتد در حالی که
پایش کاملاً داخل مغازه است.

مختصات به‌صورت نسبی (۰..۱) نوشته می‌شوند تا با تغییر رزولوشن
دوربین، پیکربندی نشکند.
"""

from typing import List, Optional, Sequence, Tuple

import cv2
import numpy as np


class ZoneFilter:
    def __init__(self, include: Optional[List[Sequence]] = None,
                 exclude: Optional[List[Sequence]] = None,
                 normalized: bool = True):
        self.include_raw = include or []
        self.exclude_raw = exclude or []
        self.normalized = normalized
        self._inc: List[np.ndarray] = []
        self._exc: List[np.ndarray] = []
        self._hw: Optional[Tuple[int, int]] = None

    @property
    def active(self) -> bool:
        return bool(self.include_raw or self.exclude_raw)

    # ------------------------------------------------------------------ #
    def _build(self, h: int, w: int):
        def conv(polys):
            out = []
            for p in polys:
                a = np.array(p, dtype=np.float32)
                if self.normalized:
                    a[:, 0] *= w
                    a[:, 1] *= h
                out.append(a.astype(np.int32))
            return out
        self._inc = conv(self.include_raw)
        self._exc = conv(self.exclude_raw)
        self._hw = (h, w)

    # ------------------------------------------------------------------ #
    def contains(self, bbox: Tuple[int, int, int, int],
                 frame_hw: Tuple[int, int]) -> bool:
        """آیا نقطهٔ پای این فرد داخل ناحیهٔ حساس است؟"""
        if not self.active:
            return True
        if self._hw != frame_hw:
            self._build(*frame_hw)

        x1, y1, x2, y2 = bbox
        foot = (float((x1 + x2) / 2.0), float(y2))     # وسطِ لبهٔ پایینی

        for poly in self._exc:
            if cv2.pointPolygonTest(poly, foot, False) >= 0:
                return False
        if not self._inc:
            return True
        return any(cv2.pointPolygonTest(p, foot, False) >= 0 for p in self._inc)

    # ------------------------------------------------------------------ #
    def draw(self, frame: np.ndarray):
        """رسم مرز ناحیه — هم برای دیباگ و هم برای ارائه مفید است."""
        if not self.active:
            return
        h, w = frame.shape[:2]
        if self._hw != (h, w):
            self._build(h, w)
        for poly in self._inc:
            cv2.polylines(frame, [poly], True, (120, 220, 120), 2, cv2.LINE_AA)
        for poly in self._exc:
            cv2.polylines(frame, [poly], True, (90, 90, 220), 2, cv2.LINE_AA)
            ov = frame.copy()
            cv2.fillPoly(ov, [poly], (60, 60, 160))
            cv2.addWeighted(ov, 0.12, frame, 0.88, 0, frame)


def build_zone_filter(cfg) -> ZoneFilter:
    z = getattr(cfg, "zones", None)
    if z is None:
        return ZoneFilter()
    return ZoneFilter(z.include, z.exclude, z.normalized)

### ★ زمان‌بند بودجه‌محور
<sub>`security_core/runtime/scheduler.py`</sub>

In [ ]:
# ==========================================================================
#  ★ زمان‌بند بودجه‌محور
#  (منبع: security_core/runtime/scheduler.py)
# ==========================================================================
from __future__ import annotations

"""
scheduler.py — ★ زمان‌بندِ محاسباتیِ تطبیقی و بودجه‌محور.

مسئله
-----
گران‌ترین بخش خط لوله، اجرای طبقه‌بند روی کروپ صورت است. اگر ۱۰ نفر
در کادر باشند و برای همه در هر فریم اجرا شود، سیستم از real-time
می‌افتد. نسخهٔ قبلی راه‌حل ابتدایی داشت: «هر فریم برای جدیدها،
یکی‌درمیان برای ماسک‌دارها، هرگز برای سبزها». دو ایراد:

  • «هرگز برای سبزها» یعنی قفل دائمی — حفرهٔ امنیتی.
  • سقفی برای تعداد کل نداشت؛ با ۱۵ نفر در کادر، latency منفجر می‌شد.

راه‌حل
------
دو مکانیزم روی هم:

  ۱) نرخ بازبینی وابسته به وضعیت:
        ANALYZING          → هر فریم (باید سریع تصمیم بگیریم)
        SUSPECT / WATCH    → هر ۰.۲۵ ثانیه (زیر نظر می‌مانَد)
        COVERED            → هر ۰.۲۵ ثانیه
        CLEAR              → هر ۲.۵ ثانیه (ارزان، ولی نه هرگز ← قفل نرم)

  ۲) سقف سختِ تعداد در هر فریم (`max_faces_per_frame`).
     اگر تعداد کاندیدها بیشتر بود، بر اساس امتیاز اولویت انتخاب
     می‌شوند: اهمیت وضعیت + مدت انتظار + کیفیت تصویر.

نتیجهٔ تجاری: latency هر فریم **کران بالا دارد** و مستقل از شلوغی
صحنه است. این دقیقاً همان تضمینی است که برای «۸ دوربین روی یک GPU»
لازم داری، و یک عدد قابل نوشتن در قرارداد است.
"""

from typing import Dict, List, Tuple



class AdaptiveScheduler:
    def __init__(self, cfg):
        self.cfg = cfg
        self._last_analyzed: Dict[int, float] = {}
        self.skipped_last_frame: int = 0     # برای گزارش شفاف در HUD/لاگ

    # ------------------------------------------------------------------ #
    def _period(self, state: ThreatState) -> float:
        f = self.cfg.fusion
        if state is ThreatState.ANALYZING:
            return 0.0                                  # هر فریم
        if state is ThreatState.CLEAR:
            return f.clear_revalidate_seconds           # قفلِ نرم
        return f.focus_revalidate_seconds               # مشکوک/تحت‌نظر/پوشیده

    # ------------------------------------------------------------------ #
    def select(self, candidates: List[Tuple[int, ThreatState, float]],
               now: float) -> List[int]:
        """
        ورودی: [(global_id, وضعیت فعلی, کیفیت مشاهدهٔ این فریم), ...]
        خروجی: لیست global_id هایی که این فریم باید تحلیل شوند.
        """
        due: List[Tuple[float, int]] = []

        for gid, state, quality in candidates:
            last = self._last_analyzed.get(gid, -1e9)
            waited = now - last
            if waited < self._period(state):
                continue

            # امتیاز اولویت — هر سه عامل عمداً در مقیاس‌های متفاوت‌اند
            # تا وضعیت مهم‌تر همیشه بر کیفیت بالاتر غلبه کند.
            score = (STATE_PRIORITY[state] * 100.0
                     + min(waited, 10.0) * 5.0
                     + quality * 10.0)
            due.append((score, gid))

        due.sort(key=lambda x: -x[0])
        cap = max(1, self.cfg.runtime.max_faces_per_frame)
        chosen = [gid for _, gid in due[:cap]]
        self.skipped_last_frame = max(0, len(due) - len(chosen))

        for gid in chosen:
            self._last_analyzed[gid] = now
        return chosen

    # ------------------------------------------------------------------ #
    def force_next(self, gid: int):
        """
        وادار کردن به تحلیل در فریم بعد — مثلاً وقتی ماژول دیگری
        (اسلحه/رفتار) روی این فرد شاهدی پیدا کرده و می‌خواهیم
        صورتش فوراً دوباره بررسی شود.
        """
        self._last_analyzed.pop(gid, None)

    def drop(self, gids):
        for g in gids:
            self._last_analyzed.pop(g, None)

### منابع ورودی (فایل / وبکم / RTSP)
<sub>`security_core/io/sources.py`</sub>

In [ ]:
# ==========================================================================
#  منابع ورودی (فایل / وبکم / RTSP)
#  (منبع: security_core/io/sources.py)
# ==========================================================================
from __future__ import annotations

"""
sources.py — منابع ورودی ویدیو.

چرا انتزاعی شده
---------------
نسخهٔ قبلی مسیر فایل را مستقیم به `model.track(source=path)` می‌داد،
یعنی خط لوله فقط با «فایل» کار می‌کرد. اینجا هر منبعی که فریم BGR
بدهد قابل استفاده است، بدون یک خط تغییر در بقیهٔ کد.

فعلاً پیاده‌سازی‌شده: فایل ویدیو و وبکم.
برای پخش زندهٔ RTSP، کلاس `RtspSource` پایین نوشته شده ولی به‌صورت
پیش‌فرض استفاده نمی‌شود (طبق تصمیم فعلی پروژه). وقتی سخت‌افزار
مناسب فراهم شد، فقط کافی است در yaml مقدار `io.source` را با
آدرس rtsp:// پر کنی — کارخانهٔ `open_source` خودش تشخیص می‌دهد.

★ نکتهٔ باگ نسخهٔ قبلی: مسیر ویندوزی
    "C:\\1\\1_پروژه\\..."  به‌صورت رشتهٔ معمولی نوشته شده بود.
    در پایتون، `\1` یک کاراکتر کنترلی (\\x01) است، نه بک‌اسلش+یک.
    آن مسیر هرگز باز نمی‌شد. اینجا همهٔ مسیرها از طریق `pathlib`
    نرمال‌سازی می‌شوند و اگر فایل نبود، **صریحاً** خطا می‌دهیم
    به‌جای اینکه بی‌صدا شکست بخوریم.
"""

import time
from dataclasses import dataclass
from pathlib import Path
from typing import Iterator, Optional, Tuple

import cv2
import numpy as np


@dataclass
class SourceInfo:
    width: int
    height: int
    fps: float
    total_frames: int          # ۰ یعنی نامعلوم (پخش زنده)
    is_live: bool


class BaseSource:
    info: SourceInfo

    def frames(self) -> Iterator[Tuple[int, float, np.ndarray]]:
        """(شمارهٔ فریم، زمان بر حسب ثانیه، تصویر BGR)"""
        raise NotImplementedError

    def release(self):
        pass


# --------------------------------------------------------------------------- #
class VideoFileSource(BaseSource):
    """خواندن از فایل. زمان از روی FPS ویدیو محاسبه می‌شود، نه ساعت دیوار."""

    def __init__(self, path: str, stride: int = 1, max_frames: int = 0):
        p = Path(path)
        if not p.exists():
            raise FileNotFoundError(f"فایل ویدیو پیدا نشد: {p}")
        self.cap = cv2.VideoCapture(str(p))
        if not self.cap.isOpened():
            raise RuntimeError(f"OpenCV نتوانست ویدیو را باز کند: {p}")

        fps = self.cap.get(cv2.CAP_PROP_FPS) or 25.0
        self.info = SourceInfo(
            width=int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
            height=int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
            fps=float(fps if fps > 1 else 25.0),
            total_frames=int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0),
            is_live=False,
        )
        self.stride = max(1, stride)
        self.max_frames = max_frames

    def frames(self):
        idx, produced = 0, 0
        while True:
            ok, frame = self.cap.read()
            if not ok:
                break
            if idx % self.stride == 0:
                # زمانِ منطقیِ ویدیو — باعث می‌شود آستانه‌های زمانیِ
                # لایهٔ fusion مستقل از سرعت پردازش باشند.
                yield idx, idx / self.info.fps, frame
                produced += 1
                if self.max_frames and produced >= self.max_frames:
                    break
            idx += 1

    def release(self):
        self.cap.release()


# --------------------------------------------------------------------------- #
class WebcamSource(BaseSource):
    """وبکم محلی. زمان از ساعت واقعی خوانده می‌شود."""

    def __init__(self, index: int = 0, stride: int = 1, max_frames: int = 0):
        self.cap = cv2.VideoCapture(index)
        if not self.cap.isOpened():
            raise RuntimeError(f"وبکم {index} باز نشد")
        self.info = SourceInfo(
            width=int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH)) or 640,
            height=int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) or 480,
            fps=float(self.cap.get(cv2.CAP_PROP_FPS) or 30.0),
            total_frames=0, is_live=True,
        )
        self.stride = max(1, stride)
        self.max_frames = max_frames

    def frames(self):
        idx, produced, t0 = 0, 0, time.perf_counter()
        while True:
            ok, frame = self.cap.read()
            if not ok:
                break
            if idx % self.stride == 0:
                yield idx, time.perf_counter() - t0, frame
                produced += 1
                if self.max_frames and produced >= self.max_frames:
                    break
            idx += 1

    def release(self):
        self.cap.release()


# --------------------------------------------------------------------------- #
class RtspSource(BaseSource):
    """
    پخش زندهٔ دوربین شبکه. ★ فعلاً استفاده نمی‌شود (طبق تصمیم پروژه)
    ولی کامل نوشته شده تا وقتی سخت‌افزار آماده شد، فقط آدرس در yaml
    عوض شود.

    نکتهٔ حیاتی برای زنده: اگر پردازش از دوربین عقب بیفتد، بافر
    OpenCV پر می‌شود و تصویر «تأخیردار» نشان داده می‌شود. با
    CAP_PROP_BUFFERSIZE=1 همیشه تازه‌ترین فریم خوانده می‌شود و
    فریم‌های عقب‌مانده دور ریخته می‌شوند — برای سیستم امنیتی،
    تازگی مهم‌تر از کامل‌بودن است.

    برای چند دوربین همزمان، هر منبع را در یک نخ جدا بگذار و
    فریم‌ها را در یک صف با maxsize=1 بریز (الگوی latest-frame).
    """

    def __init__(self, url: str, stride: int = 1, max_frames: int = 0,
                 reconnect_delay: float = 2.0):
        self.url = url
        self.stride = max(1, stride)
        self.max_frames = max_frames
        self.reconnect_delay = reconnect_delay
        self.cap = self._open()
        self.info = SourceInfo(
            width=int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH)) or 1920,
            height=int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) or 1080,
            fps=float(self.cap.get(cv2.CAP_PROP_FPS) or 25.0),
            total_frames=0, is_live=True,
        )

    def _open(self):
        cap = cv2.VideoCapture(self.url, cv2.CAP_FFMPEG)
        try:
            cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)   # همیشه تازه‌ترین فریم
        except Exception:
            pass
        if not cap.isOpened():
            raise RuntimeError(f"اتصال RTSP برقرار نشد: {self.url}")
        return cap

    def frames(self):
        idx, produced, t0 = 0, 0, time.perf_counter()
        while True:
            ok, frame = self.cap.read()
            if not ok:
                # قطعی شبکه در دوربین‌های واقعی عادی است — تلاش مجدد
                print(f"[rtsp] ارتباط قطع شد، تلاش مجدد در {self.reconnect_delay}s ...")
                self.cap.release()
                time.sleep(self.reconnect_delay)
                try:
                    self.cap = self._open()
                    continue
                except Exception as e:
                    print(f"[rtsp] اتصال مجدد ناموفق: {e}")
                    break
            if idx % self.stride == 0:
                yield idx, time.perf_counter() - t0, frame
                produced += 1
                if self.max_frames and produced >= self.max_frames:
                    break
            idx += 1

    def release(self):
        self.cap.release()


# --------------------------------------------------------------------------- #
def open_source(cfg) -> BaseSource:
    """کارخانه: از روی رشتهٔ `io.source` نوع منبع را تشخیص می‌دهد."""
    src = str(cfg.io.source).strip()
    stride, mx = cfg.io.frame_stride, cfg.io.max_frames

    if src.isdigit():
        return WebcamSource(int(src), stride, mx)
    if src.lower().startswith(("rtsp://", "rtmp://", "http://", "https://")):
        return RtspSource(src, stride, mx)
    return VideoFileSource(src, stride, mx)

### خروجی ویدیو و رویداد
<sub>`security_core/io/sinks.py`</sub>

In [ ]:
# ==========================================================================
#  خروجی ویدیو و رویداد
#  (منبع: security_core/io/sinks.py)
# ==========================================================================
from __future__ import annotations

"""
sinks.py — مقصدهای خروجی: ویدیو، فایل رویداد، و (آماده برای آینده) MQTT.

★ رفع باگ نسخهٔ قبلی: خروجیِ بی‌صدا شکست‌خورده
    out = cv2.VideoWriter(path, fourcc, fps, (W, H))
هیچ‌وقت بررسی نمی‌شد که آیا واقعاً باز شده است یا نه. اگر مسیر
اشتباه بود (مثلاً `/content/...` روی ویندوز) یا کدک نصب نبود،
برنامه بدون هیچ خطایی تا آخر اجرا می‌شد و در پایان یک فایل صفر
بایتی می‌ماند. اینجا صریحاً بررسی و در صورت نیاز به کدک دیگری
برگردانده می‌شود.
"""

import json
from pathlib import Path
from typing import List, Optional, Tuple

import cv2
import numpy as np



class VideoSink:
    """
    نویسندهٔ ویدیو با انتخاب خودکار کدک.
    ترتیب تلاش: mp4v (همه‌جا هست) → avc1 (کیفیت/حجم بهتر) → MJPG (آخرین راه‌حل)
    """
    _CANDIDATES = ["mp4v", "avc1", "MJPG"]

    def __init__(self, path: str, fps: float, size: Tuple[int, int],
                 protect: Optional[str] = None):
        """
        `protect`: مسیر ویدیوی ورودی. اگر خروجی به همان فایل اشاره کند،
        نوشتن متوقف می‌شود.

        ★ چرا این محافظ لازم است: فایل‌سیستم ویندوز به بزرگی/کوچکی
        حروف حساس نیست. پس `out.mp4` و `OUT.mp4` یک فایل‌اند، و
        `--source a.mp4 --output A.mp4` بدون هیچ هشداری **ویدیوی
        اصلی را نابود می‌کند** — درست وسط خواندنش. این را در آزمون
        خودمان به‌سختی پیدا کردیم؛ نگذاریم سرِ داده‌های مشتری تکرار شود.
        """
        self.path = Path(path)
        if protect:
            try:
                a = self.path.resolve()
                b = Path(protect).resolve()
                same = (a == b) or (str(a).lower() == str(b).lower())
            except Exception:
                same = False
            if same:
                raise ValueError(
                    f"مسیر خروجی با ورودی یکی است و ویدیوی اصلی را پاک می‌کند:\n"
                    f"  ورودی : {protect}\n  خروجی : {path}\n"
                    f"(توجه: ویندوز به بزرگی/کوچکی حروف حساس نیست)")
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.size = size
        self.writer = None
        self.frames_written = 0

        for cc in self._CANDIDATES:
            p = self.path if cc != "MJPG" else self.path.with_suffix(".avi")
            w = cv2.VideoWriter(str(p), cv2.VideoWriter_fourcc(*cc), float(fps), size)
            if w.isOpened():
                self.writer, self.path = w, p
                if cc != self._CANDIDATES[0]:
                    print(f"[sink] کدک {self._CANDIDATES[0]} کار نکرد؛ {cc} استفاده شد")
                break
            w.release()

        if self.writer is None:
            raise RuntimeError(
                f"هیچ کدکی برای نوشتن ویدیو کار نکرد: {self.path}\n"
                f"مسیر و پسوند فایل را بررسی کن (روی ویندوز مسیر /content/ معتبر نیست).")

    def write(self, frame: np.ndarray, repeat: int = 1):
        if frame.shape[1] != self.size[0] or frame.shape[0] != self.size[1]:
            frame = cv2.resize(frame, self.size)
        for _ in range(max(1, repeat)):
            self.writer.write(frame)
            self.frames_written += 1

    def release(self):
        if self.writer is not None:
            self.writer.release()
            self.writer = None


# --------------------------------------------------------------------------- #
class JsonlSink:
    """
    یک رویداد در هر خط. عمداً ساده است: هر ابزاری (pandas، jq، الاستیک)
    می‌تواند بخواندش و اگر برنامه وسط کار کشته شود، فایل سالم می‌ماند.
    """

    def __init__(self, path: str):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.f = open(self.path, "a", encoding="utf-8")
        self.count = 0

    def write(self, ev: SecurityEvent):
        self.f.write(json.dumps(ev.to_dict(), ensure_ascii=False) + "\n")
        self.f.flush()          # نوشتن فوری: اگر برق برود، رویداد از دست نرود
        self.count += 1

    def write_many(self, evs: List[SecurityEvent]):
        for e in evs:
            self.write(e)

    def release(self):
        try:
            self.f.close()
        except Exception:
            pass


# --------------------------------------------------------------------------- #
class MqttSink:
    """
    ★ برای آینده — ارسال هشدار زنده به سامانهٔ مشتری (آژیر، اپ موبایل، NVR).
    فعلاً غیرفعال است. برای فعال‌سازی:
        pip install paho-mqtt
    و در yaml بخش io یک کلید mqtt اضافه کن، سپس در pipeline صدایش بزن.

    علت اینکه از همین حالا نوشته شده: قرارداد `SecurityEvent` طوری
    طراحی شده که این ماژول بدون تغییر در بقیهٔ کد اضافه شود.
    """

    def __init__(self, host: str, port: int = 1883, topic: str = "security/events",
                 client_id: str = "security_core"):
        import paho.mqtt.client as mqtt          # وابستگی اختیاری
        self.topic = topic
        self.client = mqtt.Client(client_id=client_id)
        self.client.connect(host, port, keepalive=30)
        self.client.loop_start()

    def write(self, ev: SecurityEvent):
        self.client.publish(self.topic,
                            json.dumps(ev.to_dict(), ensure_ascii=False), qos=1)

    def release(self):
        try:
            self.client.loop_stop()
            self.client.disconnect()
        except Exception:
            pass

### ارکستراسیون — اتصال همهٔ لایه‌ها
<sub>`security_core/runtime/pipeline.py`</sub>

In [ ]:
# ==========================================================================
#  ارکستراسیون — اتصال همهٔ لایه‌ها
#  (منبع: security_core/runtime/pipeline.py)
# ==========================================================================
from __future__ import annotations

"""
pipeline.py — ارکستراسیون. تنها جایی که همهٔ لایه‌ها به هم وصل می‌شوند.

⚠️ قانون: در این فایل **هیچ منطق دامنه‌ای** نوشته نمی‌شود.
اگر خواستی آستانه‌ای را عوض کنی، جایش yaml است؛ اگر خواستی رفتار
تصمیم‌گیری را عوض کنی، جایش fusion/policy.py است. این فایل فقط
ترتیب اجرا و جریان داده را می‌داند. دلیلش این است که وقتی ماژول
اسلحه و آتش اضافه شوند، این فایل باید کوتاه و خوانا بماند.

ترتیب یک فریم
-------------
    فریم تمیز ──► تشخیص فرد + کی‌پوینت + ردیابی
              ──► جعبهٔ سر (از هندسهٔ بدن)
              ──► امضای ظاهری → هویت سراسری (global_id)
              ──► زمان‌بند: چه کسانی این فریم تحلیل شوند؟
              ──► طبقه‌بند + سرنخ‌های کمکی
              ──► انباشت شواهد (log-odds)
              ──► سیاست: باور → وضعیت/رنگ
              ──► ثبت بهترین شات + رویداد
              ──► رسم روی نسخهٔ نمایشی → نوشتن خروجی
"""

import time
from pathlib import Path
from typing import Dict, List, Optional, Set, Tuple

import cv2
import numpy as np



def resolve_device(pref: str) -> str:
    if pref != "auto":
        return pref
    try:
        import torch
        return "cuda:0" if torch.cuda.is_available() else "cpu"
    except Exception:
        return "cpu"


class SecurityPipeline:
    def __init__(self, cfg: AppConfig, project_root: Optional[Path] = None,
                 profile_cuda: bool = False,
                 shared: Optional[Dict] = None,
                 quiet: bool = False):
        """
        `shared`: دیکشنری مدل‌های مشترک بین چند دوربین، مثلاً
        {"classifier": ..., "reid": ...}.

        چرا فقط این دو تا مشترک می‌شوند و مدل ژست نه؟
        چون ultralytics وضعیت ردیاب را *داخل خود مدل* نگه می‌دارد
        (`persist=True`). اگر یک نمونهٔ YOLO بین چند دوربین مشترک شود،
        شناسه‌های دوربین‌ها با هم قاطی می‌شوند. در مقابل، طبقه‌بند و
        ReID کاملاً بی‌حالت‌اند و اشتراکشان امن است — و چون سنگین‌ترین
        بخش حافظهٔ GPU هستند، بیشترین صرفه‌جویی هم همان‌جاست.
        """
        self.cfg = cfg
        self.root = project_root or Path.cwd()
        self.device = resolve_device(cfg.runtime.device)
        self.quiet = quiet

        self._setup_torch()
        if not quiet:
            print(f"🔧 دستگاه: {self.device} | FP16: "
                  f"{cfg.runtime.half and self.device.startswith('cuda')}")

        # ---- لایه‌ها -----------------------------------------------------
        shared = shared or {}
        self.pose = PoseDetector(cfg, self.device, self.root)      # همیشه اختصاصی
        self.classifier = shared.get("classifier") or build_classifier(cfg, self.device)
        self.reid = shared.get("reid", "__none__")
        if self.reid == "__none__":
            self.reid = build_reid(cfg, self.device)
        self.bank = IdentityBank(cfg)
        self.fusion = EvidenceFusion(cfg)
        self.policy = ThreatPolicy(cfg)
        self.scheduler = AdaptiveScheduler(cfg)
        self.snapshots = SnapshotManager(cfg, cfg.io.camera_id)
        self.zones = build_zone_filter(cfg)      # ناحیهٔ حساس (ضد هشدار کاذب)
        if self.zones.active:
            print(f'[zones] ناحیهٔ حساس فعال: '
                  f'{len(cfg.zones.include)} شامل / {len(cfg.zones.exclude)} مستثنا')

        # backend فاز ۴ باید به مدل YOLO زنده وصل شود
        if getattr(self.classifier, "needs_rois", False):
            self.classifier.attach(self.pose)

        # ---- نمایش -------------------------------------------------------
        self.gallery = Gallery(cfg.viz.gallery_items, cfg.viz.gallery_thumb)
        self.slowmo = SlowMotionDirector(
            cfg.viz.slowmo_enabled and cfg.viz.slowmo_on_suspect,
            peak_repeat=cfg.viz.slowmo_repeat)
        self.flash = AlertFlash()

        self.prof = Profiler(enabled=True, sync_cuda=profile_cuda)
        self.fps_meter = FPSMeter()
        self._alerted: Set[int] = set()      # gid هایی که قبلاً آلارم داده‌اند
        self.events: List[SecurityEvent] = []

    # ------------------------------------------------------------------ #
    def _setup_torch(self):
        try:
            import torch
            if self.device.startswith("cuda"):
                torch.backends.cudnn.benchmark = self.cfg.runtime.cudnn_benchmark
                # TF32 روی کارت‌های Ampere به بعد: سرعت بیشتر، افت دقت ناچیز
                torch.backends.cuda.matmul.allow_tf32 = True
                torch.backends.cudnn.allow_tf32 = True
        except Exception:
            pass

    # ================================================================== #
    #                        پردازش یک فریم                                #
    # ================================================================== #
    def process_frame(self, frame: np.ndarray, frame_idx: int, t: float
                      ) -> Tuple[np.ndarray, int]:
        cfg = self.cfg
        H, W = frame.shape[:2]

        # ── فریم تمیز vs فریم نمایشی ─────────────────────────────────────
        # `frame` دست‌نخورده می‌ماند و همهٔ کروپ‌ها از آن گرفته می‌شوند.
        # `vis` فقط برای رسم است. این تفکیک عمدی و حیاتی است.
        clean = frame
        vis = frame.copy() if cfg.viz.enabled else frame

        # ── ۱) تشخیص + کی‌پوینت + ردیابی ────────────────────────────────
        with self.prof.section("1_pose_track"):
            dets = self.pose.track(clean)

        if getattr(self.classifier, "needs_rois", False):
            self.classifier.set_frame_shape((H, W))

        # ── ۲) جعبهٔ سر (ارزان) — برای همه ──────────────────────────────
        # ★ توجه: استخراج و ترازِ کروپ صورت اینجا انجام *نمی‌شود*.
        # آن کار (warpAffine + محاسبهٔ وضوح) حدود ۰.۶ms برای هر نفر
        # هزینه دارد و اگر برای هر ده نفرِ کادر در هر فریم انجام شود،
        # کلِ فایدهٔ زمان‌بندِ بودجه‌محور از بین می‌رود.
        # پس اینجا فقط یک «امتیاز ارزانِ اولویت» حساب می‌کنیم و کروپ
        # واقعی را فقط برای کسانی می‌سازیم که زمان‌بند انتخابشان کرد.
        with self.prof.section("2_head_roi"):
            records = []
            for d in dets:
                if d.track_id < 0:
                    continue      # ردیاب هنوز ID نداده — این فریم را رد کن
                if not self.zones.contains(d.bbox, (H, W)):
                    continue      # بیرون از ناحیهٔ حساس — اصلاً پردازش نکن
                head = estimate_head_roi(d.kpts, d.bbox, W, H,
                                         cfg.head_roi, cfg.pose.kpt_conf_th)
                hx1, hy1, hx2, hy2 = head.xyxy
                head_px = min(hx2 - hx1, hy2 - hy1)
                # ★ جهتِ سر از هندسهٔ بدن — چند عمل حسابی، عملاً رایگان،
                #   و مستقل از اینکه صورت دیده می‌شود یا نه.
                ori = (estimate_orientation(d.kpts, cfg.pose.kpt_conf_th,
                                            cfg.orientation)
                       if cfg.orientation.enabled else None)
                records.append({
                    "det": d, "head": head, "face": None,
                    "body": safe_crop(clean, d.bbox),
                    "ori": ori,
                    "facing": ori.facing if ori else 0.0,
                    "kvis": face_kpt_visibility(d.kpts),
                    # تخمین ارزانِ کیفیت: اندازهٔ سر × اعتبار تخمین جعبه
                    "prio": (min(1.0, head_px / max(1, cfg.face_quality.good_face_px))
                             * head.reliability),
                    "head_px": head_px,
                })

        # ── ۳) امضای ظاهری → هویت سراسری ────────────────────────────────
        with self.prof.section("3_reid_identity"):
            self._assign_identities(records, frame_idx, t)

        # ── ۴) زمان‌بندی: چه کسانی این فریم تحلیل شوند؟ ─────────────────
        with self.prof.section("4_schedule"):
            # ★ کسی که پشتش به دوربین است اصلاً وارد صفِ تحلیل نمی‌شود.
            #   دو سود همزمان: (۱) دیگر پشتِ سر را مثل صورت قضاوت
            #   نمی‌کنیم — ریشهٔ خطایی که در اجرای واقعی دیده شد؛
            #   (۲) بودجهٔ محاسباتی صرفِ کسانی می‌شود که واقعاً قابل
            #   ارزیابی‌اند، پس هم دقت بالا می‌رود هم سرعت.
            candidates = [(r["gid"], self.policy.state(r["gid"]), r["prio"])
                          for r in records
                          if r.get("gid")
                          and r["head_px"] >= cfg.face_quality.min_face_px
                          and not (r["ori"] is not None and r["ori"].is_back)]
            chosen = set(self.scheduler.select(candidates, t))

        # ── ۴.۵) استخراج و ترازِ صورت — فقط برای انتخاب‌شده‌ها ───────────
        with self.prof.section("4b_face_align"):
            todo = []
            for r in records:
                if r.get("gid") not in chosen:
                    continue
                r["face"] = extract_face(clean, r["head"], r["det"].kpts,
                                         cfg.face_quality)
                if r["face"] is not None:
                    todo.append(r)

        # ── ۵) طبقه‌بندی ────────────────────────────────────────────────
        with self.prof.section("5_classify"):
            probs_list = self._classify(todo)

        # ── ۶) انباشت شواهد ─────────────────────────────────────────────
        with self.prof.section("6_fusion"):
            for r, p in zip(todo, probs_list):
                p = self._apply_weak_cues(p, r)
                # ★ وزن مشاهده با اطمینانِ «رو به دوربین بودن» مقیاس می‌شود:
                #   روبه‌رو ۱.۰ | نیم‌رخ ۰.۶ | نامشخص ۰.۴۵ | پشت ۰.
                #   این نرم‌ترین شکلِ گیت است: مشاهده را دور نمی‌اندازیم،
                #   فقط به اندازهٔ قابل‌اتکا بودنش اثر می‌دهیم.
                w = r["face"].quality * (r["ori"].weight_scale if r["ori"] else 1.0)
                self.fusion.observe(r["gid"], p, w, t)
                r["probs"] = p

        # ── ۷) تصمیم + ثبت رویداد ───────────────────────────────────────
        new_alerts: List[int] = []
        with self.prof.section("7_policy_snapshot"):
            for r in records:
                gid = r.get("gid")
                if not gid:
                    continue
                post = self.fusion.posterior(gid, t)
                threat_bonus = self.fusion.module_axis(gid, "threat")
                state, conf = self.policy.evaluate(
                    gid, post, self.fusion.n_obs(gid), t, threat_bonus,
                    orientation=r["ori"])
                r["state"], r["conf"] = state, conf

                self.snapshots.offer(
                    gid, state, conf,
                    r["face"].crop if r["face"] else None,
                    r["body"],
                    r["face"].quality if r["face"] else 0.0,
                    t, frame_idx, r["det"].bbox,
                    full_clean=clean, head_box=r["head"].xyxy)

                # ★ آلارم بیرونی فقط وقتی صادر می‌شود که وضعیت قرمز
                # حداقل `alert_min_seconds` پایدار مانده باشد. یک جهش
                # لحظه‌ای (سایه، انسداد کوتاه) نباید آژیر بزند —
                # اعتماد مشتری به سیستم مستقیماً به همین بند وابسته است.
                if (state is ThreatState.SUSPECT and gid not in self._alerted
                        and self.policy.state_age(gid, t) >= cfg.fusion.alert_min_seconds):
                    self._alerted.add(gid)
                    new_alerts.append(gid)
                    self.snapshots.flush(gid)
                    ev = SecurityEvent(
                        global_id=gid, kind="face.suspect", state=state.value,
                        confidence=round(conf, 4), frame_idx=frame_idx, timestamp=t,
                        camera_id=cfg.io.camera_id, bbox=r["det"].bbox,
                        extra={"posterior": {k.value: round(v, 3)
                                             for k, v in post.items()},
                               "evidence": self.fusion.evidence_sources(gid)})
                    self.events.append(ev)

        # ── ۸) رسم ──────────────────────────────────────────────────────
        n_alerts = sum(1 for r in records
                       if r.get("state") is ThreatState.SUSPECT)
        if cfg.viz.enabled:
            with self.prof.section("8_draw"):
                self._draw(vis, records, n_alerts)

        # عکسِ زمینهٔ رویداد بعد از رسم گرفته می‌شود (این یکی عمداً
        # با overlay است — برای ارائه به اپراتور، نه برای دیتاست)
        for gid in new_alerts:
            self.snapshots.note_context(gid, vis)
            self.flash.trigger()
            self.slowmo.trigger(t)

        if cfg.viz.enabled:
            self.flash.apply(vis)

        # ── ۹) نظافت حافظه ──────────────────────────────────────────────
        if frame_idx % 60 == 0:
            dead = self.bank.collect_garbage(t)
            if dead:
                self.fusion.drop(dead)
                self.policy.drop(dead)
                self.scheduler.drop(dead)
                self.snapshots.finalize(dead)
                self._alerted -= set(dead)

        return vis, n_alerts

    # ------------------------------------------------------------------ #
    def _assign_identities(self, records, frame_idx: int, t: float):
        """محاسبهٔ بردار ظاهری (فقط برای کسانی که لازم است) و نگاشت به gid."""
        need, need_idx = [], []
        for i, r in enumerate(records):
            tid = r["det"].track_id
            fresh = not self.bank.knows(tid)
            periodic = (frame_idx % max(1, self.cfg.reid.embed_every_n)) == 0
            if self.reid is not None and (fresh or periodic) and r["body"].size:
                need.append(r["body"])
                need_idx.append(i)

        embs = self.reid.embed(need) if (self.reid is not None and need) else None

        emb_of: Dict[int, np.ndarray] = {}
        if embs is not None:
            for j, i in enumerate(need_idx):
                emb_of[i] = embs[j]

        active: Set[int] = set()
        for i, r in enumerate(records):
            gid = self.bank.update(r["det"].track_id, r["det"].bbox, t,
                                   emb_of.get(i), active)
            active.add(gid)
            r["gid"] = gid

    # ------------------------------------------------------------------ #
    def _classify(self, todo) -> List[dict]:
        if not todo:
            return []
        if getattr(self.classifier, "needs_rois", False):
            # فاز ۴: بدون کروپ — مستقیم از نقشهٔ ویژگی مشترک
            return self.classifier.classify_rois([r["head"].xyxy for r in todo])
        return self.classifier.classify([r["face"].crop for r in todo])

    # ------------------------------------------------------------------ #
    def _apply_weak_cues(self, p, r):
        """
        اضافه‌کردن سرنخ‌های ارزان به خروجی مدل، با وزن کم.

        دو سرنخ داریم و عمداً مکمل هم‌اند:
          • سرنخ رنگی: در تصویر رنگی خوب کار می‌کند، در IR سکوت می‌کند.
          • سرنخ کی‌پوینتی: مستقل از رنگ، پس شب هم کار می‌کند.
        وزن‌ها کوچک نگه داشته شده‌اند تا هیچ‌کدام نتوانند نظر مدل
        اصلی را وارونه کنند — فقط آن را کمی جابه‌جا می‌کنند.
        """
        if not self.cfg.classifier.use_weak_occlusion_cue:
            return p
        w_base = self.cfg.classifier.weak_cue_weight

        cue_p, cue_w = occlusion_cue(r["face"].crop)
        if cue_p is not None and cue_w > 0:
            p = blend(p, cue_p, w_base * cue_w * 0.6)

        # ★ سرنخِ «رو به دوربین است ولی صورتش دیده نمی‌شود» فقط وقتی
        #   معنا دارد که واقعاً بدانیم رو به دوربین است. تخمینِ قبلی
        #   (نسبت عرض شانه به طول تنه) پشت و رو را از هم تشخیص نمی‌داد.
        ori = r.get("ori")
        facing01 = (0.5 * (ori.facing + 1.0) * ori.confidence) if ori else 0.0
        kcue_p, kcue_w = keypoint_occlusion_cue(r["kvis"], facing01)
        if kcue_p is not None and kcue_w > 0:
            p = blend(p, kcue_p, w_base * kcue_w)
        return p

    # ------------------------------------------------------------------ #
    def _draw(self, vis, records, n_alerts: int):
        cfg = self.cfg
        if self.zones.active and cfg.zones.draw:
            self.zones.draw(vis)
        for r in records:
            gid = r.get("gid")
            if not gid:
                continue
            state = r.get("state", ThreatState.ANALYZING)
            if cfg.viz.draw_skeleton and r["det"].kpts is not None:
                draw_skeleton(vis, r["det"].kpts, color_of(state),
                              cfg.pose.kpt_conf_th, 2)
            draw_person(vis, r["det"].bbox, gid, state, r.get("conf", 0.0),
                        cfg.viz.thickness, cfg.viz.font_scale,
                        r["head"].xyxy if cfg.viz.draw_head_box else None)

        if cfg.viz.draw_gallery:
            self.gallery.draw(vis, self.snapshots.gallery_items(cfg.viz.gallery_items))

        if cfg.viz.hud:
            extra = ""
            if self.scheduler.skipped_last_frame:
                # شفافیت: اگر به‌خاطر بودجه کسی را رد کردیم، پنهانش نمی‌کنیم
                extra = f"queued {self.scheduler.skipped_last_frame}"
            draw_hud(vis, self.fps_meter.tick(), len(records), n_alerts,
                     self.classifier.name, extra)

    # ================================================================== #
    #                            اجرای کامل                                #
    # ================================================================== #
    def run(self) -> Dict:
        cfg = self.cfg
        src = open_source(cfg)
        info = src.info
        print(f"📹 منبع: {cfg.io.source}\n"
              f"   {info.width}×{info.height} @ {info.fps:.1f}fps"
              f" | فریم‌ها: {info.total_frames or 'زنده'}")

        self.pose.warmup(info.width, info.height)
        self.classifier.warmup(size=cfg.classifier.input_size)

        video: Optional[VideoSink] = None
        if cfg.io.write_video:
            video = VideoSink(cfg.io.output_video, info.fps / max(1, cfg.io.frame_stride),
                              (info.width, info.height),
                              protect=cfg.io.source)   # جلوگیری از بازنویسی ویدیوی ورودی
        jsonl = JsonlSink(cfg.io.events_jsonl)

        t_start = time.perf_counter()
        n = 0
        last_pct = -1
        try:
            for frame_idx, t, frame in src.frames():
                with self.prof.section("frame_total"):
                    vis, _ = self.process_frame(frame, frame_idx, t)
                    if video is not None:
                        video.write(vis, self.slowmo.repeat_for_frame())
                n += 1

                # رویدادهای جدید را فوری بنویس (اگر برنامه کشته شد، از دست نرود)
                while self.events:
                    jsonl.write(self.events.pop(0))

                if info.total_frames:
                    pct = int(frame_idx / info.total_frames * 100)
                    if pct >= last_pct + 5:
                        last_pct = pct
                        print(f"⏳ {pct:3d}%  |  فریم {n}  |  "
                              f"هویت‌های زنده: {self.bank.stats()['identities_alive']}")
                elif n % max(1, cfg.runtime.log_every_n_frames) == 0:
                    print(f"⏳ فریم {n} | هویت‌ها: {self.bank.stats()['identities_alive']}")
        except KeyboardInterrupt:
            print("\n⛔ توقف دستی — در حال بستن فایل‌ها ...")
        finally:
            src.release()
            final_events = self.snapshots.finalize_all()
            for ev in final_events:
                jsonl.write(ev)
            for ev in self.events:
                jsonl.write(ev)
            if video is not None:
                video.release()
            jsonl.release()

        elapsed = time.perf_counter() - t_start
        stats = {
            "frames": n,
            "elapsed_s": round(elapsed, 2),
            "fps": round(n / elapsed, 2) if elapsed > 0 else 0.0,
            "identities": self.bank.stats(),
            "alerts": len(self._alerted),
            "events_written": jsonl.count,
            "output_video": str(video.path) if video else None,
            "events_dir": str(self.snapshots.root),
        }

        print(self.prof.report())
        print(f"\n✅ تمام شد در {elapsed:.1f}s  ({stats['fps']} FPS پردازشی)")
        print(f"   هشدارها: {stats['alerts']} | هویت‌ها: {stats['identities']}")
        if video:
            print(f"   ویدیو: {video.path}")
        print(f"   رویدادها: {cfg.io.events_jsonl}  |  برش‌ها: {self.snapshots.root}")
        return stats

### ★ ابزار عیب‌یابی زنجیرهٔ تصمیم
<sub>`security_core/runtime/diagnose.py`</sub>

In [ ]:
# ==========================================================================
#  ★ ابزار عیب‌یابی زنجیرهٔ تصمیم
#  (منبع: security_core/runtime/diagnose.py)
# ==========================================================================
from __future__ import annotations

"""
diagnose.py — ★ ابزارِ عیب‌یابیِ زنجیرهٔ تصمیم.

چرا لازم است
------------
وقتی سیستم «هیچ هشداری نمی‌دهد»، پنج نقطه می‌توانند مقصر باشند و از
بیرون هیچ‌کدام دیده نمی‌شوند:

    ۱) کروپ صورت غلط ساخته می‌شود (کادر بزرگ/جابه‌جا)
    ۲) گیت کیفیت همه را رد می‌کند و رأیی ثبت نمی‌شود
    ۳) طبقه‌بند کلاس اشتباه می‌دهد
    ۴) لایهٔ fusion شواهد را رقیق می‌کند
    ۵) آستانه‌های policy خیلی سخت‌گیرند

حدس‌زدن بین این پنج‌تا اتلاف وقت است. این ماژول هر پنج نقطه را
هم‌زمان ثبت می‌کند و **خودِ کروپ‌ها را هم ذخیره می‌کند** تا با چشم
ببینی طبقه‌بند دقیقاً چه تصویری می‌بیند.

قاعده‌ای که این ابزار از آن دفاع می‌کند: هیچ‌وقت مدل را تیون نکن
تا وقتی ندیده‌ای ورودی‌اش چه شکلی است.

استفاده در نوت‌بوک:
    report = diagnose(cfg, max_frames=150)
"""

from collections import Counter
from pathlib import Path
from typing import Dict, List, Optional

import cv2
import numpy as np



def diagnose(cfg, max_frames: int = 150, save_crops: int = 40,
             out_dir: str = "runs/diagnose", project_root: Optional[Path] = None,
             verbose_rows: int = 25) -> Dict:
    """
    خط لوله را اجرا می‌کند و هر مشاهده را از ابتدا تا انتها ثبت می‌کند.
    خروجی: دیکشنری خلاصه + پوشه‌ای پر از کروپ‌های برچسب‌خورده.
    """
    import copy


    c = copy.deepcopy(cfg)
    c.io.max_frames = max_frames
    c.io.write_video = False
    c.snapshot.enabled = False
    c.viz.enabled = False

    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    for old in out.glob("*.jpg"):
        old.unlink()

    pipe = SecurityPipeline(c, project_root=project_root or Path("."))

    rows: List[dict] = []
    saved = {"n": 0}
    orig_observe = pipe.fusion.observe

    def spy_observe(gid, probs, weight, t):
        """قلاب روی لایهٔ fusion — هر مشاهده را قبل از انباشت ثبت می‌کند."""
        best = max(probs, key=probs.get)
        rows.append({
            "t": round(t, 2), "gid": gid, "weight": round(float(weight), 3),
            "pred": best.value, "conf": round(float(probs[best]), 3),
            **{f"p_{k.value}": round(float(v), 3) for k, v in probs.items()},
        })
        return orig_observe(gid, probs, weight, t)

    pipe.fusion.observe = spy_observe

    # قلاب دوم: ذخیرهٔ خودِ کروپ‌ها، با همان برچسبی که مدل داده
    orig_classify = pipe._classify

    def spy_classify(todo):
        res = orig_classify(todo)
        for r, p in zip(todo, res):
            if saved["n"] >= save_crops or r["face"] is None:
                continue
            best = max(p, key=p.get)
            crop = r["face"].crop
            h, w = crop.shape[:2]
            name = (f"{saved['n']:03d}_gid{r['gid']}_{best.value}"
                    f"_{p[best]:.2f}_q{r['face'].quality:.2f}_{w}x{h}.jpg")
            cv2.imwrite(str(out / name), crop)
            saved["n"] += 1
        return res

    pipe._classify = spy_classify

    # شمارش مواردی که اصلاً به طبقه‌بند نرسیدند
    # فضای‌نامی که خودِ خط لوله `extract_face` را از آن می‌خواند.
    # از __globals__ تابع استفاده می‌کنیم تا هم در حالت پکیج کار کند و
    # هم در نوت‌بوکِ تک‌فایلی که همه‌چیز در یک فضای‌نام است.
    pl_ns = SecurityPipeline.process_frame.__globals__
    orig_extract = pl_ns["extract_face"]
    gate = Counter()

    # ★ قلاب روی تخمین جهت — می‌خواهیم بدانیم چند درصد افراد «پشت به
    #   دوربین» تشخیص داده می‌شوند. اگر این عدد غیرمنتظره بالا یا پایین
    #   باشد، ریشهٔ مشکل همان‌جاست و نه در طبقه‌بند.
    orig_ori = pl_ns["estimate_orientation"]
    ori_stat = Counter()

    def spy_ori(kp, conf_th, ocfg):
        o = orig_ori(kp, conf_th, ocfg)
        ori_stat[o.mode] += 1
        return o

    pl_ns["estimate_orientation"] = spy_ori

    def spy_extract(frame, head, kp, qcfg):
        r = orig_extract(frame, head, kp, qcfg)
        hw = head.xyxy[2] - head.xyxy[0]
        hh = head.xyxy[3] - head.xyxy[1]
        if r is None:
            gate["رد‌شده"] += 1
            gate["رد: کادر کوچک‌تر از min_face_px"] += int(min(hw, hh) < qcfg.min_face_px)
        else:
            gate["پذیرفته"] += 1
        gate[f"منبع کادر: {head.source}"] += 1
        return r

    pl_ns["extract_face"] = spy_extract
    try:
        stats = pipe.run()
    finally:
        pl_ns["extract_face"] = orig_extract
        pl_ns["estimate_orientation"] = orig_ori

    # ---------------- گزارش ----------------------------------------------
    print("\n" + "=" * 78)
    print("  گزارش عیب‌یابی")
    print("=" * 78)

    print()
    print("[۰] جهتِ سر — پیش از هر قضاوتی")
    tot_o = sum(ori_stat.values())
    if tot_o:
        _note = {"frontal": "تحلیل کامل", "profile": "وزن ۰.۶",
                 "unknown": "وزن ۰.۴۵", "back": "اصلاً تحلیل نمی‌شود"}
        for k in ("frontal", "profile", "unknown", "back"):
            v = ori_stat.get(k, 0)
            print(f"      {k:<9} {v:5d}  ({v/tot_o:5.1%})   ← {_note[k]}")
        if ori_stat.get("back", 0) > 0.7 * tot_o:
            print("    ⚠️ تقریباً همه «پشت به دوربین» تشخیص داده شده‌اند. "
                  "cfg.orientation.back_threshold را به صفر نزدیک‌تر کن.")
        if ori_stat.get("back", 0) == 0 and tot_o > 30:
            print("    ⚠️ هیچ‌کس «پشت» تشخیص داده نشد. اگر در ویدیو فردِ "
                  "پشت‌به‌دوربین داری، back_threshold را منفی‌تر کن.")

    print("\n[۱] گیت کیفیت — چند مشاهده اصلاً به طبقه‌بند رسید؟")
    total_gate = gate["پذیرفته"] + gate["رد‌شده"]
    if total_gate:
        print(f"    پذیرفته: {gate['پذیرفته']}  |  رد‌شده: {gate['رد‌شده']}"
              f"  ({gate['پذیرفته']/total_gate:.0%} عبور)")
        if gate["رد‌شده"] > gate["پذیرفته"]:
            print("    ⚠️ بیشترِ مشاهدات رد می‌شوند → cfg.face_quality.min_face_px "
                  "و min_quality_to_vote را کم کن")
    for k, v in sorted(gate.items()):
        if k.startswith("منبع کادر") or k.startswith("رد:"):
            print(f"      {k}: {v}")

    print(f"\n[۲] طبقه‌بند — {len(rows)} مشاهده ثبت شد")
    if rows:
        cnt = Counter(r["pred"] for r in rows)
        for k, v in cnt.most_common():
            print(f"      {k:<14} {v:5d}  ({v/len(rows):5.1%})")
        mean_w = float(np.mean([r["weight"] for r in rows]))
        print(f"    میانگین وزنِ کیفیت: {mean_w:.3f}")
        if mean_w < 0.25:
            print("    ⚠️ وزن‌ها خیلی پایین‌اند → شواهد کند انباشته می‌شوند "
                  "(کروپ‌ها کوچک یا تارند)")
        if cnt.get("unknown", 0) + cnt.get("back_head", 0) > 0.5 * len(rows):
            print("    ⚠️ بیش از نیمی از مشاهدات unknown/back_head اند → "
                  "پرامپت‌ها یا کادرِ سر مشکل دارند")

        print(f"\n[۳] نمونهٔ مشاهدات (اولین {verbose_rows} تا)")
        hdr = f"      {'t':>6} {'gid':>4} {'وزن':>6} {'پیش‌بینی':<14} {'اطمینان':>7}"
        print(hdr); print("      " + "-" * (len(hdr) - 6))
        for r in rows[:verbose_rows]:
            print(f"      {r['t']:>6.2f} {r['gid']:>4} {r['weight']:>6.2f} "
                  f"{r['pred']:<14} {r['conf']:>7.2f}")

    print("\n[۴] باورِ نهاییِ هر هویت (بعد از انباشت)")
    print(f"      {'gid':>4} {'مشاهده':>7} {'وضعیت':<10} "
          f"{'p_clear':>8}{'p_medical':>10}{'p_full':>8}{'p_back':>8}{'p_unk':>7}")
    print("      " + "-" * 66)
    for gid in sorted(pipe.fusion.beliefs):
        post = pipe.fusion.posterior(gid)
        st = pipe.policy.state(gid)
        print(f"      {gid:>4} {pipe.fusion.n_obs(gid):>7} {st.value:<10} "
              f"{post[FaceClass.CLEAR]:>8.2f}{post[FaceClass.MEDICAL_MASK]:>10.2f}"
              f"{post[FaceClass.FULL_COVER]:>8.2f}{post[FaceClass.BACK_HEAD]:>8.2f}"
              f"{post[FaceClass.UNKNOWN]:>7.2f}")

    print(f"\n[۵] آستانه‌ها")
    f = c.fusion
    print(f"      ورود به آلارم p_full ≥ {f.suspect_enter}  |  "
          f"تحت‌نظر ≥ {f.watch_enter}  |  جرم اطلاعاتی ≥ {f.min_informative_mass}")
    best_full = max((pipe.fusion.posterior(g)[FaceClass.FULL_COVER]
                     for g in pipe.fusion.beliefs), default=0.0)
    print(f"      بیشترین p_full در کل ویدیو: {best_full:.2f}")
    if best_full < f.suspect_enter:
        print("    ⚠️ هیچ هویتی به آستانه نرسید. قبل از پایین‌آوردن آستانه، "
              "اول کروپ‌های ذخیره‌شده را با چشم ببین.")

    print(f"\n[۶] {saved['n']} کروپ ذخیره شد در: {out.resolve()}")
    print("      نام هر فایل = برچسب مدل + اطمینان + کیفیت + ابعاد")
    print("      ← این مهم‌ترین بخش گزارش است. حتماً نگاهشان کن.")
    print("=" * 78)

    return {"stats": stats, "rows": rows, "gate": dict(gate),
            "orientation": dict(ori_stat),
            "crops_dir": str(out.resolve()), "max_p_full": best_full}

---
# بخش دوم: تنظیمات و اجرا

## ۱) تنظیمات

همهٔ آستانه‌ها اینجاست — هیچ عدد جادویی داخل کد هسته نیست. برای
تیون‌کردن فقط همین سلول را عوض کن.

In [ ]:
import os, torch

cfg = AppConfig()

# ---------- ورودی و خروجی ----------------------------------------------
cfg.io.source        = "/content/drive/MyDrive/9.mp4"   # ← ویدیوی خودت
cfg.io.camera_id     = "cam0"
cfg.io.output_video  = "runs/output.mp4"
cfg.io.events_jsonl  = "runs/events.jsonl"
cfg.io.write_video   = True
cfg.io.frame_stride  = 1      # ۲ یعنی یکی‌درمیان (سریع‌تر)
cfg.io.max_frames    = 300    # ۰ = کل ویدیو. برای اولین اجرا کم بگذار!

# ---------- مدل‌ها -------------------------------------------------------
cfg.pose.weights     = "yolo11n-pose.pt"   # دقیق‌تر: yolo11s-pose.pt
cfg.pose.imgsz       = 640                 # سریع‌تر: 512 | دقیق‌تر: 960
cfg.pose.conf        = 0.35

# ★ حالت ترکیبی: سؤال به دو نیم شکسته می‌شود و هر نیم به مدلی
# می‌رود که در همان قوی است.
#   «پوشیده هست یا نه؟»  → مدل فاین‌تیون‌شدهٔ نسخهٔ اول (دقیق و پایدار)
#   «چه نوع پوششی؟»      → صفر-شات، فقط به‌صورت مقایسهٔ دوتایی
# اجرای فقط صفر-شات روی کروپ‌های دوربین مداربسته از مدل نسخهٔ اول
# ضعیف‌تر بود؛ این حالت هر دو قوت را با هم دارد.
# گزینه‌ها: "hybrid" | "siglip_binary" | "siglip_zeroshot"
cfg.classifier.backend    = "hybrid"
cfg.classifier.batch_max  = 16

# ---------- بودجهٔ محاسباتی ---------------------------------------------
# سقف تعداد صورتِ تحلیل‌شده در هر فریم. باعث می‌شود تأخیر هر فریم
# کران بالا داشته باشد و مستقل از شلوغی صحنه بماند.
cfg.runtime.max_faces_per_frame = 8
cfg.runtime.half = torch.cuda.is_available()

# ---------- آستانه‌های تصمیم (هیسترزیس) ---------------------------------
cfg.fusion.suspect_enter = 0.72   # ورود به آلارم — سخت‌گیرانه
cfg.fusion.suspect_exit  = 0.42   # خروج از آلارم — آسان‌تر (ضد چشمک‌زدن)
cfg.fusion.watch_enter   = 0.45
cfg.fusion.alert_min_seconds = 0.35   # قرمز باید این‌قدر پایدار بماند

# سبز هیچ‌وقت دائمی نیست: هر ۲.۵ ثانیه دوباره بررسی می‌شود، چون
# دزد می‌تواند با صورت باز وارد شود و بعد ماسک بکشد.
cfg.fusion.clear_revalidate_seconds = 2.5

# ---------- ★ جهتِ سر (تازه اضافه شده) ----------------------------------
# مهم‌ترین اصلاح این نسخه: قبل از هر قضاوتی دربارهٔ پوششِ صورت،
# اول از هندسهٔ بدن می‌پرسیم فرد رو به دوربین است یا پشت.
# کسی که پشتش به ماست اصلاً تحلیل نمی‌شود و وضعیت «Back to camera»
# می‌گیرد — نه سبز، نه خاکستری، نه قرمز.
cfg.orientation.enabled           = True
cfg.orientation.back_threshold    = -0.28   # منفی‌تر = سخت‌گیرتر در اعلام «پشت»
cfg.orientation.frontal_threshold = 0.22
cfg.orientation.min_confidence    = 0.22

# ---------- نمایش --------------------------------------------------------
cfg.viz.enabled       = True
cfg.viz.draw_skeleton = True
cfg.viz.draw_gallery  = True      # گالری برش‌ها در گوشهٔ تصویر
cfg.viz.draw_head_box = False
cfg.viz.slowmo_enabled = False    # برای دمو True کن

# ---------- ذخیرهٔ برش‌ها -------------------------------------------------
cfg.snapshot.enabled     = True
cfg.snapshot.out_dir     = "runs/events"
cfg.snapshot.save_states = ["suspect", "watch", "covered"]
# برای جمع‌آوری دیتاست فازهای بعدی True کن:
cfg.snapshot.harvest_all_for_dataset = False

# ---------- ناحیهٔ حساس (اختیاری ولی خیلی مؤثر) --------------------------
# بزرگ‌ترین منبع هشدار کاذب «مدل» نیست، «کادر» است: دوربین مغازه
# پیاده‌روِ بیرون را هم می‌بیند و عابرِ کلاه‌دار برای مدل کاملاً درست
# «پوشش کامل صورت» است. مختصات نسبی (۰..۱) است.
# cfg.zones.include = [[[0.02, 0.32], [0.98, 0.32], [0.98, 0.98], [0.02, 0.98]]]
# cfg.zones.exclude = [[[0.00, 0.00], [1.00, 0.00], [1.00, 0.28], [0.00, 0.28]]]

os.makedirs("runs", exist_ok=True)
print("✅ تنظیمات آماده |", cfg.io.source)

## ۲) اتصال Google Drive (اگر ویدیو آنجاست)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
print("ویدیو پیدا شد ✅" if os.path.exists(cfg.io.source)
      else f"❌ پیدا نشد: {cfg.io.source}")

## ۳) اجرا

اولین اجرا چند دقیقه طول می‌کشد چون وزن‌های YOLO و SigLIP دانلود
می‌شوند. اجراهای بعدی سریع‌اند.

> **توصیه:** برای اولین بار `cfg.io.max_frames = 300` بگذار. اگر
> تنظیمی غلط باشد در یک دقیقه می‌فهمی، نه بعد از یک ساعت.

In [ ]:
pipe = SecurityPipeline(cfg, project_root=Path("."))
stats = pipe.run()

## ۳.۵) عیب‌یابی — اگر نتیجه راضی‌کننده نبود، **اول این را اجرا کن**

این سلول روی ۱۵۰ فریم اول اجرا می‌شود و کل زنجیرهٔ تصمیم را باز
می‌کند: چند مشاهده از گیت کیفیت رد شد، طبقه‌بند چه گفت، شواهد چطور
انباشته شد، و چرا به آستانه رسید یا نرسید.

مهم‌تر از همهٔ اعداد: **کروپ‌ها را ذخیره می‌کند**. با چشم ببین
طبقه‌بند دقیقاً چه تصویری می‌بیند. تقریباً همیشه مشکل همان‌جا
دیده می‌شود.

In [ ]:
report = diagnose(cfg, max_frames=150, save_crops=40)

### دیدنِ کروپ‌هایی که طبقه‌بند واقعاً می‌بیند

اگر در این تصاویر صورت ریز است و نصف کادر را پس‌زمینه و شانه گرفته،
مشکل از مدل نیست — از کادر است. آن‌وقت `cfg.head_roi.expand` و
`eye_box_scale` را کم کن.

In [ ]:
import glob, cv2
import matplotlib.pyplot as plt

files = sorted(glob.glob(f"{report['crops_dir']}/*.jpg"))[:12]
if files:
    rows = (len(files) + 5) // 6
    fig, axes = plt.subplots(rows, 6, figsize=(18, 3.4 * rows))
    for ax, f in zip(axes.ravel(), files):
        ax.imshow(cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB)); ax.axis("off")
        parts = f.split("_")
        ax.set_title("_".join(parts[2:4]), fontsize=8)
    for ax in axes.ravel()[len(files):]:
        ax.axis("off")
    plt.tight_layout(); plt.show()
else:
    print("هیچ کروپی ذخیره نشد — یعنی گیت کیفیت همه را رد کرده است.")

## ۴) نمایش ویدیوی خروجی

In [ ]:
import subprocess, os
from base64 import b64encode
from IPython.display import HTML

src = cfg.io.output_video
web = "runs/preview.mp4"
# بازکدگذاری با H.264 — مرورگر کدک mp4v را پخش نمی‌کند
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", src,
                "-vcodec", "libx264", "-crf", "26", web], check=False)

path = web if os.path.exists(web) else src
data = b64encode(open(path, "rb").read()).decode()
HTML(f'<video width=800 controls>'
     f'<source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')

## ۵) برش‌های ذخیره‌شده

این همان بخشی است که در ارائه بیشترین اثر را دارد: سیستم فقط هشدار
نمی‌دهد، بلکه **چهرهٔ سوژه را بیرون می‌کشد**.

دو نکته که با نسخهٔ قبلی فرق دارد:
- برش‌ها از فریمِ **تمیز** گرفته می‌شوند (بدون کادر رنگی و خط اسکلت
  روی صورت)، پس هم برای ارائه تمیزند و هم بعداً به‌عنوان دادهٔ آموزشی
  قابل استفاده.
- **بهترین** تصویر هر نفر ذخیره می‌شود، نه اولین تصویر (که معمولاً
  لحظهٔ ورود و تارترین حالت است).

In [ ]:
import glob, json, os, cv2
import matplotlib.pyplot as plt

faces = sorted(glob.glob(f"{cfg.snapshot.out_dir}/*/*/best_face.jpg"))
print(f"{len(faces)} سوژه ثبت شده")

if faces:
    n = min(len(faces), 8)
    fig, axes = plt.subplots(1, n, figsize=(3.2 * n, 3.8))
    axes = [axes] if n == 1 else axes
    for ax, f in zip(axes, faces[:n]):
        ax.imshow(cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB))
        ax.axis("off")
        meta = json.load(open(os.path.join(os.path.dirname(f), "meta.json"),
                              encoding="utf-8"))
        ax.set_title(f"#{meta['global_id']}\n{meta['state']} "
                     f"{meta['confidence']:.0%}", fontsize=10)
    plt.tight_layout()
    plt.show()
else:
    print("هیچ سوژه‌ای در وضعیت‌های ذخیره‌شدنی پیدا نشد.")
    print("cfg.snapshot.save_states =", cfg.snapshot.save_states)

## ۶) رویدادها و پروفایل زمان

پروفایل می‌گوید گلوگاه دقیقاً کجاست. بدون این عدد، بهینه‌سازی
حدس‌وگمان می‌شود.

In [ ]:
import json
from pathlib import Path

p = Path(cfg.io.events_jsonl)
if p.exists():
    for line in p.read_text(encoding="utf-8").splitlines():
        e = json.loads(line)
        print(f"[{e['timestamp']:7.2f}s] #{e['global_id']:<3} {e['kind']:<16} "
              f"اطمینان {e['confidence']:.0%}")
else:
    print("هیچ رویدادی ثبت نشد.")

print(pipe.prof.report())

ms = pipe.prof.summary().get("frame_total", 0)
if ms > 0:
    fps = 1000 / ms
    print(f"\n📊 برآورد ظرفیت روی همین سخت‌افزار:")
    print(f"   دوربین ۱۵fps : ~{fps/15:.1f} عدد")
    print(f"   دوربین ۲۵fps : ~{fps/25:.1f} عدد")
    print("   (محافظه‌کارانه ۷۰٪ این عدد را در قرارداد بنویس.)")

## ۷) نسخهٔ دمو برای کارفرما (اختیاری)

همان منطق، با جلوه‌های نمایشی: اسلوموشنِ نرم در لحظهٔ اعلام خطر،
جعبهٔ سر، و گالری بزرگ‌تر.

In [ ]:
import copy

demo = copy.deepcopy(cfg)
demo.io.output_video = "runs/demo.mp4"
demo.io.events_jsonl = "runs/demo_events.jsonl"
demo.io.max_frames   = 0        # کل ویدیو

demo.pose.weights    = "yolo11s-pose.pt"   # در دمو کیفیت حرف اول را می‌زند
demo.runtime.max_faces_per_frame = 12

demo.viz.slowmo_enabled = True
demo.viz.slowmo_repeat  = 6
demo.viz.draw_head_box  = True
demo.viz.gallery_thumb  = 170

demo_pipe = SecurityPipeline(demo, project_root=Path("."))
demo_stats = demo_pipe.run()

---
## فازهای بعدی (فعلاً لازم نیست)

این نوت‌بوک عمداً فقط مسیر **بدون آموزش** را دارد. دو گام بعدی در
پکیج کامل `security_core/` آماده‌اند و هرکدام فقط با عوض‌کردن یک
کلید فعال می‌شوند:

**فاز ۳ — مدل سبک اختصاصی.** همین SigLIP از «مدل زمان اجرا» به
«مدل معلمِ آفلاین» تبدیل می‌شود: کروپ‌های ذخیره‌شده را خودکار برچسب
می‌زند و دانشش در یک MobileNetV4 (حدود ۴ میلیون پارامتر، در برابر
۲۰۰ میلیون) تقطیر می‌شود. چند برابر سریع‌تر، و چون روی دادهٔ *خودت*
آموزش می‌بیند معمولاً دقیق‌تر هم می‌شود.

برای شروع جمع‌آوری داده، همین حالا کافی است در سلول تنظیمات بگذاری:
`cfg.snapshot.harvest_all_for_dataset = True`

**فاز ۴ — سرِ RoI روی ویژگی مشترک.** به‌جای encode مجدد کروپ صورت،
ویژگی‌ها مستقیم از نقشهٔ ویژگی خودِ YOLO برداشته می‌شوند. هزینهٔ
طبقه‌بندی از خطی نسبت به تعداد افراد به **ثابت** تبدیل می‌شود.

**چند دوربین همزمان** هم در پکیج کامل پیاده شده
(`scripts/run_multicam.py`) — خواندن موازی، پردازش نوبتی، و مدل
مشترک بین دوربین‌ها.